# NYC Yellow Taxi demand forecasting with graph neural networks

Forecast hourly pickups for each taxi zone at **1, 3, 6, 12 and 24 hours ahead**. The benchmark uses all 12 months of the original 2025 NYC TLC Yellow Taxi parquet files. A seeded shuffle assigns whole months to training, validation and test. These splits measure generalization to held-out months of 2025. For a forward deployment evaluation, reserve a later period and train only on earlier observations.

Run the notebook from the top in Colab with a GPU. The final section exports the validation-selected graph model as ONNX with a Vercel inference API and dashboard.


In [ ]:
# Colab runtime settings: select a GPU before running this notebook.
import torch

REQUIRE_GPU = True
USE_MIXED_PRECISION = True

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("In Colab, choose Runtime > Change runtime type > GPU, then run again.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cuda_training = device.type == "cuda"
amp_enabled = cuda_training and USE_MIXED_PRECISION
amp_dtype = torch.bfloat16 if cuda_training and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

if cuda_training:
    print("GPU:", torch.cuda.get_device_name())
    print("Neural training precision:", amp_dtype if amp_enabled else torch.float32)
else:
    print("CPU mode · full precision")

import psutil

ram = psutil.virtual_memory()

print("Total RAM:", round(ram.total / (1024**3), 2), "GB")
print("Available RAM:", round(ram.available / (1024**3), 2), "GB")

GPU: NVIDIA L4
Neural training precision: torch.bfloat16
Total RAM: 52.96 GB
Available RAM: 50.73 GB


In [ ]:
# Start with one month to understand the raw data before scaling our analysis to the entire year.
import pandas as pd
import numpy as np
import psutil

url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2025-01.parquet"
)

df = pd.read_parquet(url)

print("January 2025 loaded successfully!")
print("=" * 45)

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMemory usage:",
      round(df.memory_usage(deep=True).sum() / (1024**2), 2),
      "MB")

print("\nAvailable RAM:",
      round(psutil.virtual_memory().available / (1024**3), 2),
      "GB")

January 2025 loaded successfully!
Dataset shape: (3475226, 20)

Column names:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']

First 5 rows:


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0



Memory usage: 616.31 MB

Available RAM: 49.56 GB


In [ ]:
# Before cleaning anything, understand what each row represents and which missing values actually affect our target.
print("INITIAL DATA EXPLORATION")
print("=" * 50)

print("\n1. DATASET SHAPE")
print(df.shape)

print("\n2. DATA TYPES")
print(df.dtypes)

print("\n3. MISSING VALUES")
print(df.isnull().sum())

print("\n4. EXACT DUPLICATE RECORDS")
print(df.duplicated().sum())

print("\n5. PICKUP DATE RANGE")
print("Earliest:", df["tpep_pickup_datetime"].min())
print("Latest:", df["tpep_pickup_datetime"].max())

print("\n6. PICKUP LOCATION IDS")
print("Unique zones:", df["PULocationID"].nunique())
print("Minimum ID:", df["PULocationID"].min())
print("Maximum ID:", df["PULocationID"].max())

print("\nINITIAL DATA EXPLORATION COMPLETED!")

INITIAL DATA EXPLORATION

1. DATASET SHAPE
(3475226, 20)

2. DATA TYPES
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

3. MISSING VALUES
VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0

The January 2025 NYC Yellow Taxi dataset contains **3,475,226 records and 20 columns**.
Missing values were identified in five columns, while essential pickup timestamps and location IDs have no missing values.
Records with missing passenger counts will not be automatically removed, as they may still represent valid taxi pickups.
No exact duplicate records were found, but some pickup timestamps fall outside January 2025 and require filtering.
The dataset contains 261 unique pickup location IDs, which will be validated using the official NYC TLC taxi zone lookup.


In [ ]:

zone_url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zone_lookup.csv"
)

zones = pd.read_csv(zone_url)

valid_boroughs = [
    "Manhattan",
    "Brooklyn",
    "Queens",
    "Bronx",
    "Staten Island",
    "EWR"
]

valid_zones = zones[
    zones["Borough"].isin(valid_boroughs)
].copy()

zone_ids = sorted(valid_zones["LocationID"].unique())

print("TAXI ZONE VALIDATION")
print("=" * 45)

print("Total lookup records:", len(zones))
print("Valid geographic zones:", len(zone_ids))

print("\nZones by borough:")
print(valid_zones["Borough"].value_counts())

assert len(zone_ids) == 263

print("\nZONE VALIDATION PASSED!")


TAXI ZONE VALIDATION
Total lookup records: 265
Valid geographic zones: 263

Zones by borough:
Borough
Queens           69
Manhattan        69
Brooklyn         61
Bronx            43
Staten Island    20
EWR               1
Name: count, dtype: int64

ZONE VALIDATION PASSED!


The NYC TLC taxi zone lookup contains 265 location records, of which 263 represent the selected geographic zones across five NYC boroughs and EWR (Newark Airport). Queens and Manhattan each contain 69 zones, followed by Brooklyn (61), Bronx (43), Staten Island (20), and EWR (1). The remaining two special location IDs will be excluded from geographic demand analysis. These 263 zones will be used consistently throughout the 12-month forecasting dataset.

In [ ]:
import pandas as pd
import numpy as np
import gc

# Why aggregate by hour and zone? Our model will predict pickup counts, not individual taxi trips.

monthly_matrices = []
quality_reports = []

# Why process month by month? To handle a full year of taxi data without loading everything into memory at once.

for month in range(1, 13):

    month_start = pd.Timestamp(2025, month, 1)
    month_end = month_start + pd.DateOffset(months=1)

    print(f"\nProcessing {month_start.strftime('%B %Y')}...")

    # Reuse January already loaded; download other months
    if month == 1:
        month_df = df
    else:
        url = (
            "https://d37ci6vzurychx.cloudfront.net/"
            f"trip-data/yellow_tripdata_2025-{month:02d}.parquet"
        )
        month_df = pd.read_parquet(url)

    original_count = len(month_df)

    # Data quality checks
    missing_pickup = month_df["tpep_pickup_datetime"].isna().sum()
    missing_zone = month_df["PULocationID"].isna().sum()
    exact_duplicates = month_df.duplicated().sum()

    # Keep valid pickup dates for this month
    date_mask = (
        (month_df["tpep_pickup_datetime"] >= month_start) &
        (month_df["tpep_pickup_datetime"] < month_end)
    )

    # Same positive-duration check as January notebook
    duration_mask = (
        month_df["tpep_dropoff_datetime"] >
        month_df["tpep_pickup_datetime"]
    )

    # Valid geographic pickup zones
    zone_mask = month_df["PULocationID"].isin(zone_ids)

    # Track impact of cleaning rules
    valid_date_count = int(date_mask.sum())
    bad_duration_count = int((date_mask & ~duration_mask).sum())
    valid_date_duration_count = int(
        (date_mask & duration_mask).sum()
    )

    keep_mask = date_mask & duration_mask & zone_mask

    clean_month = month_df.loc[
        keep_mask,
        ["tpep_pickup_datetime", "PULocationID"]
    ].copy()

    clean_month["hour"] = (
        clean_month["tpep_pickup_datetime"].dt.floor("h")
    )

    hourly = (
        clean_month
        .groupby(["hour", "PULocationID"])
        .size()
        .unstack("PULocationID", fill_value=0)
    )

    all_hours = pd.date_range(
        month_start,
        month_end,
        freq="h",
        inclusive="left"
    )

    hourly = hourly.reindex(
        index=all_hours,
        columns=zone_ids,
        fill_value=0
    )

    assert hourly.to_numpy().sum() == len(clean_month)
    assert hourly.shape == (len(all_hours), 263)

    monthly_matrices.append(hourly)

    quality_reports.append({
        "month": month_start.strftime("%Y-%m"),
        "original_records": original_count,
        "missing_pickup_time": int(missing_pickup),
        "missing_pickup_zone": int(missing_zone),
        "exact_duplicates": int(exact_duplicates),
        "outside_month": original_count - valid_date_count,
        "bad_duration_in_month": bad_duration_count,
        "valid_date_and_duration": valid_date_duration_count,
        "retained_pickups": len(clean_month)
    })

    print(f"Original records: {original_count:,}")
    print(f"Retained pickups: {len(clean_month):,}")
    print(f"Hourly matrix shape: {hourly.shape}")

    del clean_month, hourly
    if month != 1:
        del month_df
    gc.collect()

# Combine all 12 monthly demand matrices
demand_matrix = pd.concat(monthly_matrices).sort_index()
quality_report = pd.DataFrame(quality_reports)

print("\nFULL-YEAR PROCESSING COMPLETED!")
print("=" * 50)
print("Final demand matrix shape:", demand_matrix.shape)
print("Total retained pickups:",
      f"{demand_matrix.to_numpy().sum():,}")

assert demand_matrix.shape == (8760, 263)
assert demand_matrix.index.is_unique
assert not demand_matrix.isna().any().any()
assert (demand_matrix.to_numpy() >= 0).all()

print("\nFINAL MATRIX VALIDATION PASSED!")

print("\nMonthly quality report:")
display(quality_report)



Processing January 2025...
Original records: 3,475,226
Retained pickups: 3,463,680
Hourly matrix shape: (744, 263)

Processing February 2025...
Original records: 3,577,543
Retained pickups: 3,563,756
Hourly matrix shape: (672, 263)

Processing March 2025...
Original records: 4,145,257
Retained pickups: 4,113,820
Hourly matrix shape: (744, 263)

Processing April 2025...
Original records: 3,970,553
Retained pickups: 3,926,849
Hourly matrix shape: (720, 263)

Processing May 2025...
Original records: 4,591,845
Retained pickups: 4,517,934
Hourly matrix shape: (744, 263)

Processing June 2025...
Original records: 4,322,960
Retained pickups: 4,245,804
Hourly matrix shape: (720, 263)

Processing July 2025...
Original records: 3,898,963
Retained pickups: 3,834,587
Hourly matrix shape: (744, 263)

Processing August 2025...
Original records: 3,574,091
Retained pickups: 3,518,693
Hourly matrix shape: (744, 263)

Processing September 2025...
Original records: 4,251,015
Retained pickups: 4,185,680


,month,original_records,missing_pickup_time,missing_pickup_zone,exact_duplicates,outside_month,bad_duration_in_month,valid_date_and_duration,retained_pickups
0,2025-01,3475226,0,0,0,22,2051,3473153,3463680
1,2025-02,3577543,0,0,0,31,5164,3572348,3563756
2,2025-03,4145257,0,0,0,33,22280,4122944,4113820
3,2025-04,3970553,0,0,0,7,34769,3935777,3926849
4,2025-05,4591845,0,0,0,24,64272,4527549,4517934
5,2025-06,4322960,0,0,0,20,68583,4254357,4245804
6,2025-07,3898963,0,0,1,7,56064,3842892,3834587
7,2025-08,3574091,0,0,0,17,47833,3526241,3518693
8,2025-09,4251015,0,0,0,7,57179,4193829,4185680
9,2025-10,4428699,0,0,0,13,67968,4360718,4352236


In [ ]:

import pandas as pd

url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2025-06.parquet"
)

june_df = pd.read_parquet(
    url,
    columns=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID"
    ]
)

# Keep pickups recorded during June
june_df = june_df[
    (june_df["tpep_pickup_datetime"] >= "2025-06-01") &
    (june_df["tpep_pickup_datetime"] < "2025-07-01")
].copy()

# Calculate trip duration in seconds
duration = (
    june_df["tpep_dropoff_datetime"] -
    june_df["tpep_pickup_datetime"]
).dt.total_seconds()

zero_duration = (duration == 0).sum()
negative_duration = (duration < 0).sum()
missing_duration = duration.isna().sum()

print("JUNE 2025 - DURATION INVESTIGATION")
print("=" * 45)

print("Zero-duration records:", f"{zero_duration:,}")
print("Negative-duration records:", f"{negative_duration:,}")
print("Missing-duration records:", f"{missing_duration:,}")

total_failed = (
    zero_duration +
    negative_duration +
    missing_duration
)

print("\nTotal failed-duration records:",
      f"{total_failed:,}")

assert total_failed == 68583

# Check whether failed records have valid pickup zones
failed_mask = duration.isna() | (duration <= 0)

valid_pickup_zones = (
    june_df.loc[failed_mask, "PULocationID"]
    .isin(zone_ids)
    .sum()
)

print("\nFailed-duration records with valid geographic pickup zones:",
      f"{valid_pickup_zones:,}")

print("\nExamples:")
display(
    june_df.loc[failed_mask].head(10)
)

JUNE 2025 - DURATION INVESTIGATION
Zero-duration records: 68,352
Negative-duration records: 231
Missing-duration records: 0

Total failed-duration records: 68,583

Failed-duration records with valid geographic pickup zones: 68,541

Examples:


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID
4,2025-06-01 00:16:32,2025-06-01 00:16:32,48
25,2025-06-01 00:07:41,2025-06-01 00:07:41,186
26,2025-06-01 00:16:41,2025-06-01 00:16:41,79
27,2025-06-01 00:32:06,2025-06-01 00:32:06,114
48,2025-06-01 00:00:09,2025-06-01 00:00:09,158
68,2025-06-01 00:16:17,2025-06-01 00:16:17,164
69,2025-06-01 00:41:18,2025-06-01 00:41:18,158
123,2025-06-01 00:54:09,2025-06-01 00:54:09,48
163,2025-06-01 00:03:00,2025-06-01 00:03:00,162
164,2025-06-01 00:38:06,2025-06-01 00:38:06,48


In [ ]:

import pandas as pd
import gc

duration_reports = []

for month in range(1, 13):

    url = (
        "https://d37ci6vzurychx.cloudfront.net/"
        f"trip-data/yellow_tripdata_2025-{month:02d}.parquet"
    )

    temp = pd.read_parquet(
        url,
        columns=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID"
        ]
    )

    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)

    temp = temp.loc[
        (temp["tpep_pickup_datetime"] >= start) &
        (temp["tpep_pickup_datetime"] < end)
    ]

    duration = (
        temp["tpep_dropoff_datetime"] -
        temp["tpep_pickup_datetime"]
    ).dt.total_seconds()

    failed = duration.isna() | (duration <= 0)

    duration_reports.append({
        "month": start.strftime("%Y-%m"),
        "zero_duration": int((duration == 0).sum()),
        "negative_duration": int((duration < 0).sum()),
        "missing_duration": int(duration.isna().sum()),
        "failed_with_valid_pickup_zone": int(
            (failed & temp["PULocationID"].isin(zone_ids)).sum()
        )
    })

    del temp, duration
    gc.collect()

duration_report = pd.DataFrame(duration_reports)

print("FULL-YEAR DURATION INVESTIGATION")
print("=" * 45)

display(duration_report)

print("\nTotal zero-duration records:",
      f"{duration_report['zero_duration'].sum():,}")

print("Total negative-duration records:",
      f"{duration_report['negative_duration'].sum():,}")

print("Total failed records with valid pickup zones:",
      f"{duration_report['failed_with_valid_pickup_zone'].sum():,}")


FULL-YEAR DURATION INVESTIGATION


,month,zero_duration,negative_duration,missing_duration,failed_with_valid_pickup_zone
0,2025-01,1927,124,0,2003
1,2025-02,5071,93,0,5121
2,2025-03,22199,81,0,22169
3,2025-04,34606,163,0,34706
4,2025-05,64171,101,0,64200
5,2025-06,68352,231,0,68541
6,2025-07,56063,1,0,56003
7,2025-08,47831,2,0,47774
8,2025-09,57179,0,0,57113
9,2025-10,67966,2,0,67908



Total zero-duration records: 544,069
Total negative-duration records: 2,235
Total failed records with valid pickup zones: 545,589


The full-year duration investigation identified 544,069 zero-duration records and 2,235 negative-duration records, with no missing durations. Among these, 545,589 records had valid geographic pickup zones. Since the project focuses on hourly taxi pickup demand rather than trip duration, records will not be excluded solely due to invalid trip durations. Instead, valid pickup timestamps and geographic pickup zones will be used for demand aggregation, while duration anomalies will be documented separately.

In [ ]:

import pandas as pd
import gc

# A trip with an invalid duration may still be a real pickup. Should we discard it?

print("RESTORING EXCLUDED PICKUP RECORDS")
print("=" * 55)

# Preserve the original strict-cleaning matrix
strict_demand_matrix = demand_matrix.copy()

# Create a separate updated matrix
updated_demand_matrix = strict_demand_matrix.copy()

restored_counts = []

for month in range(1, 13):

    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)

    url = (
        "https://d37ci6vzurychx.cloudfront.net/"
        f"trip-data/yellow_tripdata_2025-{month:02d}.parquet"
    )

    # Load only the columns needed to restore excluded records
    temp = pd.read_parquet(
        url,
        columns=[
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
            "PULocationID"
        ]
    )

    # Same date and zone rules as our original cleaning
    valid_date = (
        (temp["tpep_pickup_datetime"] >= start) &
        (temp["tpep_pickup_datetime"] < end)
    )

    valid_zone = temp["PULocationID"].isin(zone_ids)

    # Identify records previously excluded by duration
    failed_duration = ~(
        temp["tpep_dropoff_datetime"] >
        temp["tpep_pickup_datetime"]
    )

    # Select only eligible records to restore
    additional = temp.loc[
        valid_date & valid_zone & failed_duration,
        ["tpep_pickup_datetime", "PULocationID"]
    ].copy()

    # Convert pickup timestamps to hourly intervals
    additional["hour"] = (
        additional["tpep_pickup_datetime"].dt.floor("h")
    )

    # Count restored pickups by hour and zone
    additional_counts = (
        additional
        .groupby(["hour", "PULocationID"])
        .size()
        .unstack("PULocationID", fill_value=0)
    )

    # Align counts with the existing full-year matrix
    additional_counts = additional_counts.reindex(
        index=updated_demand_matrix.index,
        columns=updated_demand_matrix.columns,
        fill_value=0
    )

    # Add restored pickups
    updated_demand_matrix = (
        updated_demand_matrix + additional_counts
    )

    restored_counts.append({
        "month": start.strftime("%Y-%m"),
        "restored_pickups": len(additional)
    })

    print(
        f"{start.strftime('%B %Y')}: "
        f"{len(additional):,} pickups restored"
    )

    del temp, additional, additional_counts
    gc.collect()

# Create restoration report
restoration_report = pd.DataFrame(restored_counts)

# Final validation
original_total = int(strict_demand_matrix.to_numpy().sum())
restored_total = int(restoration_report["restored_pickups"].sum())
updated_total = int(updated_demand_matrix.to_numpy().sum())

assert restored_total == 545589
assert updated_total == original_total + restored_total
assert updated_demand_matrix.shape == (8760, 263)
assert updated_demand_matrix.index.is_unique
assert not updated_demand_matrix.isna().any().any()
assert (updated_demand_matrix.to_numpy() >= 0).all()

print("\nRESTORATION COMPLETED SUCCESSFULLY!")
print("=" * 55)

print("Original pickups:", f"{original_total:,}")
print("Restored pickups:", f"{restored_total:,}")
print("Updated pickups:", f"{updated_total:,}")
print("Updated matrix shape:", updated_demand_matrix.shape)

print("\nFINAL VALIDATION PASSED!")

display(restoration_report)


RESTORING EXCLUDED PICKUP RECORDS
January 2025: 2,003 pickups restored
February 2025: 5,121 pickups restored
March 2025: 22,169 pickups restored
April 2025: 34,706 pickups restored
May 2025: 64,200 pickups restored
June 2025: 68,541 pickups restored
July 2025: 56,003 pickups restored
August 2025: 47,774 pickups restored
September 2025: 57,113 pickups restored
October 2025: 67,908 pickups restored
November 2025: 62,076 pickups restored
December 2025: 57,975 pickups restored

RESTORATION COMPLETED SUCCESSFULLY!
Original pickups: 48,073,756
Restored pickups: 545,589
Updated pickups: 48,619,345
Updated matrix shape: (8760, 263)

FINAL VALIDATION PASSED!


,month,restored_pickups
0,2025-01,2003
1,2025-02,5121
2,2025-03,22169
3,2025-04,34706
4,2025-05,64200
5,2025-06,68541
6,2025-07,56003
7,2025-08,47774
8,2025-09,57113
9,2025-10,67908


The initial cleaning process excluded records with zero or negative trip durations. Since the project focuses on hourly taxi pickup demand, 545,589 records with valid pickup timestamps and geographic pickup zones were restored. The updated demand matrix preserves the original 263-zone structure and includes these additional recorded pickups. The revised dataset will be validated before use in forecasting models.

The initial cleaning process excluded records with zero or negative trip durations. After investigating these anomalies, 545,589 records with valid pickup timestamps and geographic pickup zones were restored because the project focuses on pickup demand rather than trip duration. The updated dataset contains 48,619,345 recorded pickups across 263 geographic zones and 8,760 hourly intervals. Validation confirmed that the updated matrix contains no missing or negative values and that all restored records were accounted for.

In [ ]:

print("FINAL CLEANED NYC TAXI DATASET - 2025")
print("=" * 55)

# 1. Display first 5 rows
print("\n1. FIRST 5 ROWS OF CLEANED DATA")
display(updated_demand_matrix.head())

# 2. Display last 5 rows
print("\n2. LAST 5 ROWS OF CLEANED DATA")
display(updated_demand_matrix.tail())

# 3. Dataset dimensions
print("\n3. DATASET SHAPE")
print("Rows (hours):", updated_demand_matrix.shape[0])
print("Columns (taxi zones):", updated_demand_matrix.shape[1])

# 4. Missing values
print("\n4. TOTAL MISSING VALUES")
print(updated_demand_matrix.isnull().sum().sum())

# 5. Negative pickup counts
print("\n5. NEGATIVE PICKUP COUNTS")
print((updated_demand_matrix < 0).sum().sum())

# 6. Duplicate hourly timestamps
print("\n6. DUPLICATE HOURLY TIMESTAMPS")
print(updated_demand_matrix.index.duplicated().sum())

# 7. Date range
print("\n7. DATASET DATE RANGE")
print("Start:", updated_demand_matrix.index.min())
print("End:", updated_demand_matrix.index.max())

# 8. Total pickups
print("\n8. TOTAL TAXI PICKUPS")
print(f"{updated_demand_matrix.to_numpy().sum():,}")

print("\nFINAL DATASET INSPECTION COMPLETED!")


FINAL CLEANED NYC TAXI DATASET - 2025

1. FIRST 5 ROWS OF CLEANED DATA


PULocationID,1,2,3,4,5,6,7,8,9,10,...,254,255,256,257,258,259,260,261,262,263
2025-01-01 00:00:00,0,0,0,28,0,0,12,0,1,1,...,0,20,13,2,3,0,4,36,48,223
2025-01-01 01:00:00,0,0,1,54,0,0,37,0,1,8,...,2,55,36,3,4,2,19,31,65,245
2025-01-01 02:00:00,0,0,0,52,0,0,62,0,1,6,...,3,52,34,4,8,3,29,20,36,168
2025-01-01 03:00:00,0,0,1,42,0,0,29,0,0,0,...,0,51,48,1,4,1,9,10,11,103
2025-01-01 04:00:00,0,0,1,32,1,0,32,0,0,0,...,1,41,30,0,2,0,13,8,12,48



2. LAST 5 ROWS OF CLEANED DATA


PULocationID,1,2,3,4,5,6,7,8,9,10,...,254,255,256,257,258,259,260,261,262,263
2025-12-31 19:00:00,0,0,2,41,0,0,37,0,0,4,...,1,21,19,2,1,0,11,46,170,248
2025-12-31 20:00:00,0,0,0,60,0,1,43,0,0,7,...,3,20,20,0,2,2,11,41,194,268
2025-12-31 21:00:00,0,0,2,65,0,0,56,0,3,4,...,6,18,25,3,9,5,23,43,149,217
2025-12-31 22:00:00,0,0,1,46,0,0,51,0,3,8,...,4,30,29,2,6,1,21,25,86,197
2025-12-31 23:00:00,0,0,1,28,0,0,21,0,1,7,...,3,40,18,1,2,1,6,24,69,154



3. DATASET SHAPE
Rows (hours): 8760
Columns (taxi zones): 263

4. TOTAL MISSING VALUES
0

5. NEGATIVE PICKUP COUNTS
0

6. DUPLICATE HOURLY TIMESTAMPS
0

7. DATASET DATE RANGE
Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00

8. TOTAL TAXI PICKUPS
48,619,345

FINAL DATASET INSPECTION COMPLETED!


### Final Cleaned Dataset Validation

The cleaned NYC Yellow Taxi dataset for January–December 2025 contains **48,619,345 recorded pickups**, aggregated into **8,760 hourly observations across 263 geographic taxi zones**. Each row represents one hourly interval, and each column represents a taxi pickup zone. The dataset was inspected using `head()`, `tail()`, and validation checks to confirm its structure and integrity. The results show **zero missing values, zero negative pickup counts, and zero duplicate hourly timestamps**, confirming that the aggregated demand matrix is structurally consistent and suitable for further analysis.

In [ ]:
# Verify that all expected hourly timestamps are present

# Are all 8,760 hourly timestamps present? Missing hours could distort the patterns we learn.

expected_hours = pd.date_range(
    start="2025-01-01 00:00:00",
    end="2025-12-31 23:00:00",
    freq="h"
)

missing_hours = expected_hours.difference(
    updated_demand_matrix.index
)

extra_hours = updated_demand_matrix.index.difference(
    expected_hours
)

print("Expected hours:", len(expected_hours))
print("Actual hours:", len(updated_demand_matrix))
print("Missing hours:", len(missing_hours))
print("Unexpected hours:", len(extra_hours))
print("Chronologically sorted:",
      updated_demand_matrix.index.is_monotonic_increasing)

assert len(missing_hours) == 0
assert len(extra_hours) == 0
assert updated_demand_matrix.index.is_monotonic_increasing

print("\nHOURLY TIMESTAMP VALIDATION PASSED!")


Expected hours: 8760
Actual hours: 8760
Missing hours: 0
Unexpected hours: 0
Chronologically sorted: True

HOURLY TIMESTAMP VALIDATION PASSED!


### Hourly Timestamp Continuity Validation

The hourly timestamp index was validated to ensure complete and chronological coverage of the year 2025. A reference index containing **8,760 expected hourly timestamps** was generated and compared with the cleaned demand matrix. The results confirmed **8,760 actual timestamps, zero missing hours, zero unexpected hours, and correctly sorted chronological order**. This verifies continuous coverage of the expected hourly labels, which is important for time-series forecasting and Graph Neural Network modeling. Daylight-saving-time effects will be reviewed separately because the timestamps do not contain timezone information.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Assign whole months before fitting any benchmark preprocessing.
SPLIT_SEED = 42
shuffled_months = np.random.default_rng(SPLIT_SEED).permutation(np.arange(1, 13))
month_groups = {
    "train": sorted(shuffled_months[:8].tolist()),
    "val": sorted(shuffled_months[8:10].tolist()),
    "test": sorted(shuffled_months[10:].tolist()),
}
month_to_split = {month: name for name, months in month_groups.items() for month in months}
split_names = {
    name: ", ".join(pd.Timestamp(2025, month, 1).strftime("%b") for month in months)
    for name, months in month_groups.items()
}
split_description = " · ".join(f"{name}: {months}" for name, months in split_names.items())
print(f"2025 month split (seed {SPLIT_SEED}) | {split_description}")

def split_for_index(times):
    return times.month.map(month_to_split).to_numpy()

def shade_month_splits(ax, alpha=0.12):
    colors = {"train": "tab:blue", "val": "tab:orange", "test": "tab:green"}
    for name, months in month_groups.items():
        for j, month in enumerate(months):
            start = pd.Timestamp(2025, month, 1)
            ax.axvspan(start, start + pd.offsets.MonthBegin(), color=colors[name],
                       alpha=alpha, label=f"{name}: {split_names[name]}" if j == 0 else None)

SAVE = f"/content/drive/MyDrive/cs668/random_months_five_horizons_seed{SPLIT_SEED}/"
import os; os.makedirs(SAVE, exist_ok=True)
updated_demand_matrix.rename(columns=str).to_parquet(SAVE + 'demand_2025.parquet')

# DST: Mar 9 02:00 doesn't exist locally; Nov 2 01:00 happens twice
m = updated_demand_matrix.sum(axis=1)
print(m.loc['2025-03-09 00:00':'2025-03-09 04:00'])
print(m.loc['2025-11-02 00:00':'2025-11-02 03:00'])

Mounted at /content/drive
2025 month split (seed 42) | train: Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec · val: May, Nov · test: Feb, Sep
2025-03-09 00:00:00    9550
2025-03-09 01:00:00    7883
2025-03-09 02:00:00       0
2025-03-09 03:00:00    6184
2025-03-09 04:00:00    3915
Freq: h, dtype: int64
2025-11-02 00:00:00     9791
2025-11-02 01:00:00    11663
2025-11-02 02:00:00     3866
2025-11-02 03:00:00     3121
Freq: h, dtype: int64


In [ ]:
print("DAYLIGHT-SAVING-TIME ADJUSTMENT")
print("=" * 55)

# Keep an untouched copy for reproducibility
raw_demand_matrix = updated_demand_matrix.copy()
final_demand_matrix = updated_demand_matrix.copy()

# Clock changes can create artificial demand gaps or spikes. How do we avoid treating them as real patterns?

spring = pd.Timestamp("2025-03-09 02:00")   # hour skipped (clocks jump 02:00 -> 03:00)
fall = pd.Timestamp("2025-11-02 01:00")     # hour repeated (clocks fall back 02:00 -> 01:00)
one_hr = pd.Timedelta(hours=1)

# Spring forward: impute each zone as the average of 01:00 and 03:00
final_demand_matrix.loc[spring] = (
    (final_demand_matrix.loc[spring - one_hr] +
     final_demand_matrix.loc[spring + one_hr]) / 2
).round().astype(int)

# Fall back: two clock-hours were merged into one label, so keep half
final_demand_matrix.loc[fall] = (
    final_demand_matrix.loc[fall] / 2
).round().astype(int)

# Flag the adjusted hours (used as a model feature / excluded from evaluation)
dst_flag = pd.Series(0, index=final_demand_matrix.index, name="dst_flag")
dst_flag.loc[[spring, fall]] = 1

city_before = raw_demand_matrix.sum(axis=1)
city_after = final_demand_matrix.sum(axis=1)

print("\nSpring forward (Mar 9):")
display(pd.DataFrame({
    "before": city_before.loc["2025-03-09 00:00":"2025-03-09 04:00"],
    "after": city_after.loc["2025-03-09 00:00":"2025-03-09 04:00"]}))

print("\nFall back (Nov 2):")
display(pd.DataFrame({
    "before": city_before.loc["2025-11-02 00:00":"2025-11-02 03:00"],
    "after": city_after.loc["2025-11-02 00:00":"2025-11-02 03:00"]}))

print("\nTotal pickups before:", f"{int(city_before.sum()):,}")
print("Total pickups after: ", f"{int(city_after.sum()):,}")
print("Net change:          ", f"{int(city_after.sum() - city_before.sum()):+,}")

assert final_demand_matrix.shape == (8760, 263)
assert not final_demand_matrix.isna().any().any()
assert (final_demand_matrix.to_numpy() >= 0).all()
assert dst_flag.sum() == 2

# Save final dataset to Drive
final_demand_matrix.rename(columns=str).to_parquet(SAVE + "demand_2025_final.parquet")
raw_demand_matrix.rename(columns=str).to_parquet(SAVE + "demand_2025_raw.parquet")
dst_flag.to_frame().to_parquet(SAVE + "dst_flag.parquet")

print("\nDST ADJUSTMENT COMPLETED AND FILES SAVED!")

DAYLIGHT-SAVING-TIME ADJUSTMENT

Spring forward (Mar 9):


,before,after
2025-03-09 00:00:00,9550,9550
2025-03-09 01:00:00,7883,7883
2025-03-09 02:00:00,0,7017
2025-03-09 03:00:00,6184,6184
2025-03-09 04:00:00,3915,3915



Fall back (Nov 2):


,before,after
2025-11-02 00:00:00,9791,9791
2025-11-02 01:00:00,11663,5818
2025-11-02 02:00:00,3866,3866
2025-11-02 03:00:00,3121,3121



Total pickups before: 48,619,345
Total pickups after:  48,620,517
Net change:           +1,172

DST ADJUSTMENT COMPLETED AND FILES SAVED!


NYC TLC pickup timestamps use local clock time, which creates obvious artifacts in the hourly data during the two 2025 daylight-saving transitions:

Spring forward (March 9): Clocks jumped directly from 02:00 to 03:00. Because that 02:00 hour never actually existed, it registered 0 pickups across all 263 zones—a clear technical artifact rather than a sudden drop in demand, especially compared to the 7,883 pickups at 01:00 and 6,184 at 03:00. To fix this gap, each zone's count was imputed by averaging its 01:00 and 03:00 figures, yielding [X] pickups.

Fall back (November 2): Clocks turned back from 02:00 to 01:00, effectively combining two actual hours under a single 01:00 timestamp. This caused a spike to 11,663 pickups—well above the 9,791 recorded at midnight, despite late-night demand usually falling off. Halving the count for each zone brought the adjusted total to [Y] pickups, fitting neatly into the surrounding trend (between 9,791 at midnight and 3,866 at 02:00) to reflect the typical overnight decay.

The net change to total pickups is [Z], a negligible share of the 48.6 million records. Both adjusted hours are marked with a dst_flag feature so they can be excluded from evaluation. The unadjusted matrix is saved separately for reproducibility.

***Exploratory Data Analytics***

In [ ]:
import matplotlib.pyplot as plt

print("TEMPORAL DEMAND PATTERNS")

# Do taxi pickups follow predictable hourly, daily, or weekly patterns?

city = final_demand_matrix.sum(axis=1)
daily = city.resample("D").sum()

fig, ax = plt.subplots(2, 1, figsize=(14, 8))
daily.plot(ax=ax[0], title="Daily Citywide Taxi Pickups, 2025")
ax[0].set_ylabel("Pickups per day")
city.loc["2025-03-03":"2025-03-16"].plot(ax=ax[1], title="Hourly Pickups, Two Sample Weeks (Mar 3–16)")
ax[1].set_ylabel("Pickups per hour")
plt.tight_layout(); plt.show()

heat = city.groupby([city.index.dayofweek, city.index.hour]).mean().unstack()
heat.index = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
plt.figure(figsize=(14, 4))
plt.imshow(heat, aspect="auto", cmap="viridis")
plt.colorbar(label="Mean pickups per hour")
plt.yticks(range(7), heat.index); plt.xticks(range(24))
plt.xlabel("Hour of day"); plt.title("Average Hourly Demand by Day of Week")
plt.show()

hourly_profile = city.groupby(city.index.hour).mean()
dow_profile = daily.groupby(daily.index.dayofweek).mean()
monthly = city.resample("ME").sum()

print("Mean pickups per hour:", f"{city.mean():,.0f}")
print("Mean pickups per day:", f"{daily.mean():,.0f}")
print("Peak hour of day:", hourly_profile.idxmax(), f"({hourly_profile.max():,.0f})")
print("Lowest hour of day:", hourly_profile.idxmin(), f"({hourly_profile.min():,.0f})")
print("Busiest weekday:", heat.index[dow_profile.idxmax()], f"({dow_profile.max():,.0f}/day)")
print("Quietest weekday:", heat.index[dow_profile.idxmin()], f"({dow_profile.min():,.0f}/day)")
print("\nBusiest day:", daily.idxmax().date(), f"({daily.max():,})")
print("Quietest day:", daily.idxmin().date(), f"({daily.min():,})")
print("\nQuietest 5 days:"); print(daily.nsmallest(5))
print("\nMonthly totals:"); print(monthly.map("{:,}".format))

TEMPORAL DEMAND PATTERNS


<Figure size 1400x800 with 2 Axes>

<Figure size 1400x400 with 2 Axes>

Mean pickups per hour: 5,550
Mean pickups per day: 133,207
Peak hour of day: 18 (9,498)
Lowest hour of day: 4 (986)
Busiest weekday: Sat (144,625/day)
Quietest weekday: Mon (114,170/day)

Busiest day: 2025-11-01 (182,674)
Quietest day: 2025-12-25 (67,953)

Quietest 5 days:
2025-12-25    67953
2025-01-05    79321
2025-01-06    79878
2025-12-26    83747
2025-01-02    84559
dtype: int64

Monthly totals:
2025-01-31    3,465,683
2025-02-28    3,568,877
2025-03-31    4,143,006
2025-04-30    3,961,555
2025-05-31    4,582,134
2025-06-30    4,314,345
2025-07-31    3,890,590
2025-08-31    3,566,467
2025-09-30    4,242,793
2025-10-31    4,420,144
2025-11-30    4,167,540
2025-12-31    4,297,383
Freq: ME, dtype: object


In [ ]:
final_demand_matrix = pd.read_parquet(SAVE + "demand_2025_final.parquet")
final_demand_matrix.columns = final_demand_matrix.columns.astype(int)
zones = pd.read_csv("https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv")
zone_ids = final_demand_matrix.columns.tolist()
city = final_demand_matrix.sum(axis=1)

In [ ]:
print("ZONE-LEVEL DEMAND DISTRIBUTION AND SPARSITY")
print("=" * 55)

zone_info = zones.set_index("LocationID").loc[zone_ids, ["Borough", "Zone"]]

# Not every zone has equal demand. Can one forecasting approach handle both busy and nearly inactive zones?

zone_stats = pd.DataFrame({
    "mean_per_hour": final_demand_matrix.mean(),
    "max_per_hour": final_demand_matrix.max(),
    "zero_share": (final_demand_matrix == 0).mean(),
    "total_pickups": final_demand_matrix.sum(),
}).join(zone_info)

zone_stats["activity"] = pd.cut(
    zone_stats["mean_per_hour"],
    bins=[-0.01, 0.1, 1, 10, 100, np.inf],
    labels=["<0.1/hr", "0.1–1/hr", "1–10/hr", "10–100/hr", "100+/hr"])

print("Zones by activity level:")
print(zone_stats["activity"].value_counts().sort_index())

ranked = zone_stats["total_pickups"].sort_values(ascending=False)
cum_share = ranked.cumsum() / ranked.sum()
print("\nShare of pickups from top 10 zones:", f"{cum_share.iloc[9]:.1%}")
print("Share from top 26 zones (top 10%):", f"{cum_share.iloc[25]:.1%}")
print("Zones needed for 80% of pickups:", int((cum_share < 0.8).sum() + 1))
print("Zones with >90% zero hours:", int((zone_stats["zero_share"] > 0.9).sum()))
print("Overall share of zero zone-hours:",
      f"{(final_demand_matrix == 0).to_numpy().mean():.1%}")

print("\nTop 15 zones by total pickups:")
display(zone_stats.sort_values("total_pickups", ascending=False).head(15)
        [["Zone", "Borough", "mean_per_hour", "max_per_hour", "zero_share"]].round(2))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].hist(np.log1p(final_demand_matrix.to_numpy().ravel()), bins=60)
ax[0].set_title("Distribution of Hourly Zone Demand (log scale)")
ax[0].set_xlabel("log(1 + pickups)"); ax[0].set_ylabel("Zone-hours")
ax[1].plot(range(1, len(cum_share) + 1), cum_share.values)
ax[1].axhline(0.8, ls="--", c="gray")
ax[1].set_title("Cumulative Share of Pickups by Ranked Zone")
ax[1].set_xlabel("Number of zones (ranked)"); ax[1].set_ylabel("Cumulative share")
plt.tight_layout(); plt.show()

ZONE-LEVEL DEMAND DISTRIBUTION AND SPARSITY
Zones by activity level:
activity
<0.1/hr      34
0.1–1/hr     70
1–10/hr      95
10–100/hr    42
100+/hr      22
Name: count, dtype: int64

Share of pickups from top 10 zones: 34.3%
Share from top 26 zones (top 10%): 68.2%
Zones needed for 80% of pickups: 36
Zones with >90% zero hours: 35
Overall share of zero zone-hours: 38.6%

Top 15 zones by total pickups:


,Zone,Borough,mean_per_hour,max_per_hour,zero_share
PULocationID,,,,,
237,Upper East Side South,Manhattan,242.64,945,0.00
161,Midtown Center,Manhattan,238.56,891,0.00
132,JFK Airport,Queens,235.44,741,0.00
236,Upper East Side North,Manhattan,214.02,779,0.01
186,Penn Station/Madison Sq West,Manhattan,176.17,591,0.00
230,Times Sq/Theatre District,Manhattan,173.10,761,0.00
162,Midtown East,Manhattan,172.16,651,0.00
142,Lincoln Square East,Manhattan,158.40,775,0.00
170,Murray Hill,Manhattan,147.76,448,0.00


<Figure size 1400x400 with 2 Axes>

In [ ]:
print("BOROUGH AND ZONE-TYPE DEMAND PATTERNS")
print("=" * 55)

borough_hourly = final_demand_matrix.T.groupby(zone_info["Borough"]).sum().T
b_tot = borough_hourly.sum().sort_values(ascending=False)
display(pd.DataFrame({
    "zones": zone_info["Borough"].value_counts(),
    "pickups": b_tot,
    "share_%": (b_tot / b_tot.sum() * 100).round(2),
}).sort_values("pickups", ascending=False))

main = ["Manhattan", "Queens", "Brooklyn", "Bronx"]
is_wkend = borough_hourly.index.dayofweek >= 5
fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
for a, mask, title in [(ax[0], ~is_wkend, "Weekdays"), (ax[1], is_wkend, "Weekends")]:
    prof = borough_hourly.loc[mask, main].groupby(borough_hourly.index[mask].hour).mean()
    (prof / prof.mean()).plot(ax=a, title=f"Normalized Hourly Profile by Borough ({title})")
    a.set_xlabel("Hour of day"); a.set_ylabel("Demand / borough mean")
plt.tight_layout(); plt.show()

# Representative zones: business district, residential, airports, nightlife
rep_ids = [161, 237, 132, 138, 79]
names = zone_info.loc[rep_ids, "Zone"].to_dict()
how = final_demand_matrix.index.dayofweek * 24 + final_demand_matrix.index.hour
weekly = final_demand_matrix[rep_ids].groupby(how).mean()
weekly = (weekly / weekly.mean()).rename(columns=names)
weekly.plot(figsize=(14, 4), title="Normalized Weekly Profile of Representative Zones")
plt.xticks(range(0, 168, 24), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
plt.ylabel("Demand / zone mean"); plt.show()

print("Peak hour of week per zone:")
for z, name in names.items():
    peak = weekly[name].idxmax()
    print(f"  {name}: {['Mon','Tue','Wed','Thu','Fri','Sat','Sun'][peak // 24]} {peak % 24:02d}:00")

BOROUGH AND ZONE-TYPE DEMAND PATTERNS


,zones,pickups,share_%
Borough,,,
Manhattan,69,41821268,86.02
Queens,69,4643326,9.55
Brooklyn,61,1767239,3.63
Bronx,43,377898,0.78
EWR,1,6056,0.01
Staten Island,20,4730,0.01


<Figure size 1400x400 with 2 Axes>

<Figure size 1400x400 with 1 Axes>

Peak hour of week per zone:
  Midtown Center: Tue 18:00
  Upper East Side South: Thu 18:00
  JFK Airport: Sun 22:00
  LaGuardia Airport: Fri 14:00
  East Village: Sun 01:00


In [ ]:
!pip -q install geopandas mapclassify
!wget -q -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip && unzip -oq taxi_zones.zip -d tz
import geopandas as gpd, glob

print("SPATIAL DISTRIBUTION OF DEMAND")
print("=" * 55)

gdf = gpd.read_file(glob.glob("tz/**/*.shp", recursive=True)[0]).to_crs(2263)
gdf = gdf.dissolve(by="LocationID").reindex(zone_ids)
assert gdf.geometry.notna().all() and len(gdf) == 263
gdf = gdf[["geometry"]].join(zone_stats[["mean_per_hour", "zero_share", "Borough"]])
gdf["log_mean"] = np.log1p(gdf["mean_per_hour"])

fig, ax = plt.subplots(1, 2, figsize=(16, 8))
gdf.plot(column="log_mean", cmap="viridis", legend=True, ax=ax[0],
         legend_kwds={"label": "log(1 + mean pickups/hour)", "shrink": 0.6})
ax[0].set_title("Mean Hourly Pickups by Taxi Zone (log scale)"); ax[0].axis("off")
gdf.plot(column="zero_share", cmap="magma_r", legend=True, ax=ax[1],
         legend_kwds={"label": "Share of hours with zero pickups", "shrink": 0.6})
ax[1].set_title("Demand Sparsity by Taxi Zone"); ax[1].axis("off")
plt.tight_layout(); plt.show()

SPATIAL DISTRIBUTION OF DEMAND


<Figure size 1600x800 with 4 Axes>

In [ ]:
print("SPATIAL DEPENDENCE BETWEEN ZONES")
print("=" * 55)

# Contiguity graph (50 ft buffer handles small gaps between polygons)
# Our core research idea: Could demand in one taxi zone help predict demand in connected zones?

geoms = gdf.geometry.buffer(50)
A_adj = np.array([geoms.intersects(g).values for g in geoms], dtype=float)
np.fill_diagonal(A_adj, 0)

cent = np.c_[gdf.centroid.x, gdf.centroid.y]
D = np.sqrt(((cent[:, None] - cent[None]) ** 2).sum(-1)) / 5280  # miles

isolated = np.where(A_adj.sum(1) == 0)[0]
print("Zones with no touching neighbour:",
      [f"{zone_ids[i]} ({zone_info.iloc[i]['Zone']})" for i in isolated])
for i in isolated:                       # connect islands to 3 nearest zones
    for j in np.argsort(D[i])[1:4]:
        A_adj[i, j] = A_adj[j, i] = 1
print("Edges:", int(A_adj.sum() / 2),
      "| Mean neighbours per zone:", round(A_adj.sum(1).mean(), 1))
np.save(SAVE + "A_adj.npy", A_adj); np.save(SAVE + "D_miles.npy", D)

# Global Moran's I on log mean demand
def morans_i(x, A):
    W = A / A.sum(1, keepdims=True)
    z = x - x.mean()
    return (z @ W @ z) / (z @ z)

x = np.log1p(gdf["mean_per_hour"].values)
I = morans_i(x, A_adj)
rng = np.random.default_rng(42)
perm = np.array([morans_i(rng.permutation(x), A_adj) for _ in range(999)])
p = (np.sum(perm >= I) + 1) / 1000
print(f"\nGlobal Moran's I (log mean demand): {I:.3f}  (permutation p = {p:.3f})")

# Do neighbouring zones move together beyond the shared daily/weekly cycle?
active = zone_stats.index[zone_stats["mean_per_hour"] >= 1].tolist()
pos = [zone_ids.index(z) for z in active]
X = np.log1p(final_demand_matrix[active])
how = X.index.dayofweek * 24 + X.index.hour
resid = X - X.groupby(how).transform("mean")     # remove each zone's weekly pattern
C = np.corrcoef(resid.to_numpy().T)
iu = np.triu_indices(len(active), 1)
corr = C[iu]
adj = A_adj[np.ix_(pos, pos)][iu]
dist = D[np.ix_(pos, pos)][iu]

print(f"\nActive zones analysed (mean ≥ 1 pickup/hr): {len(active)}")
print(f"Mean residual correlation, adjacent pairs:     {corr[adj == 1].mean():.3f}")
print(f"Mean residual correlation, non-adjacent pairs: {corr[adj == 0].mean():.3f}")

bins = pd.cut(dist, [0, 1, 2, 5, 10, np.inf], labels=["0–1", "1–2", "2–5", "5–10", "10+"])
by_dist = pd.Series(corr).groupby(bins, observed=False).agg(["mean", "count"])
print("\nResidual correlation by distance (miles):"); print(by_dist.round(3))

by_dist["mean"].plot(kind="bar", figsize=(8, 4),
                     title="Residual Demand Correlation vs Distance Between Zones")
plt.xlabel("Distance between zone centroids (miles)"); plt.ylabel("Mean correlation")
plt.xticks(rotation=0); plt.show()

SPATIAL DEPENDENCE BETWEEN ZONES
Zones with no touching neighbour: ['1 (Newark Airport)', '46 (City Island)', "103 (Governor's Island/Ellis Island/Liberty Island)", "104 (Governor's Island/Ellis Island/Liberty Island)", "105 (Governor's Island/Ellis Island/Liberty Island)", '202 (Roosevelt Island)']
Edges: 672 | Mean neighbours per zone: 5.1

Global Moran's I (log mean demand): 0.818  (permutation p = 0.001)

Active zones analysed (mean ≥ 1 pickup/hr): 159
Mean residual correlation, adjacent pairs:     0.388
Mean residual correlation, non-adjacent pairs: 0.200

Residual correlation by distance (miles):
       mean  count
0–1   0.450    304
1–2   0.363    829
2–5   0.268   3326
5–10  0.164   5419
10+   0.134   2683


<Figure size 800x400 with 1 Axes>

In [ ]:
from statsmodels.tsa.stattools import acf

print("TEMPORAL AUTOCORRELATION")
print("=" * 55)

series = {"Citywide": np.log1p(city)}
for z in [161, 132, 79]:
    series[zone_info.loc[z, "Zone"]] = np.log1p(final_demand_matrix[z])

# Does demand repeat after 24 hours or 168 hours? These patterns may help us forecast future pickups.

lag_list = [1, 2, 3, 6, 12, 24, 48, 168]
rows = {}
fig, ax = plt.subplots(figsize=(14, 4))
for name, s in series.items():
    r = acf(s, nlags=24 * 8)
    ax.plot(r, label=name)
    rows[name] = r[lag_list]
for lag in [24, 168]:
    ax.axvline(lag, ls="--", c="gray")
ax.set_title("Autocorrelation of Hourly Demand (log scale), Lags up to 8 Days")
ax.set_xlabel("Lag (hours)"); ax.set_ylabel("ACF"); ax.legend(); plt.show()

display(pd.DataFrame(rows, index=[f"lag {l}h" for l in lag_list]).T.round(3))

TEMPORAL AUTOCORRELATION


<Figure size 1400x400 with 1 Axes>

,lag 1h,lag 2h,lag 3h,lag 6h,lag 12h,lag 24h,lag 48h,lag 168h
Citywide,0.924,0.733,0.482,-0.150,-0.432,0.875,0.723,0.942
Midtown Center,0.928,0.776,0.562,-0.111,-0.582,0.927,0.882,0.942
JFK Airport,0.816,0.535,0.264,-0.039,-0.499,0.897,0.866,0.875
East Village,0.895,0.710,0.484,-0.013,0.008,0.671,0.311,0.885


In [ ]:
!pip -q install holidays
import holidays

print("HOLIDAYS AND ANOMALOUS DAYS")
print("=" * 55)

daily = city.resample("D").sum()
ny_hol = holidays.US(state="NY", years=2025)
hol_dates = pd.to_datetime(sorted(ny_hol.keys()))
normal = daily[~daily.index.isin(hol_dates)]

# Expected demand = median of same weekday in the same month (non-holiday days)
med = normal.groupby([normal.index.month, normal.index.dayofweek]).median()
expected = pd.Series(
    med.reindex(list(zip(daily.index.month, daily.index.dayofweek))).values,
    index=daily.index)
ratio = daily / expected

hol_table = pd.DataFrame({
    "holiday": [ny_hol.get(d.date()) for d in hol_dates],
    "weekday": hol_dates.day_name(),
    "pickups": daily.loc[hol_dates].values,
    "typical": expected.loc[hol_dates].values.round(),
}, index=hol_dates.date)
hol_table["pct_vs_typical"] = ((hol_table["pickups"] / hol_table["typical"] - 1) * 100).round(1)
print("Holiday demand vs a typical same-weekday in the same month:")
display(hol_table)

label = lambda d: ny_hol.get(d.date(), "")
low = ratio.nsmallest(10); high = ratio.nlargest(5)
print("\n10 most unusually LOW days:")
display(pd.DataFrame({"weekday": low.index.day_name(), "ratio": low.round(3),
                      "holiday": [label(d) for d in low.index]}))
print("\n5 most unusually HIGH days:")
display(pd.DataFrame({"weekday": high.index.day_name(), "ratio": high.round(3),
                      "holiday": [label(d) for d in high.index]}))

ax = ratio.plot(figsize=(14, 4), title="Daily Demand Relative to Typical Same-Weekday Demand")
ax.scatter(hol_dates, ratio.loc[hol_dates], c="red", zorder=3, label="NY holiday")
ax.axhline(1, ls="--", c="gray"); ax.set_ylabel("Actual / expected"); ax.legend(); plt.show()

HOLIDAYS AND ANOMALOUS DAYS
Holiday demand vs a typical same-weekday in the same month:


,holiday,weekday,pickups,typical,pct_vs_typical
2025-01-01,New Year's Day,Wednesday,89905,122656.0,-26.7
2025-01-20,Martin Luther King Jr. Day,Monday,89857,92826.0,-3.2
2025-02-12,Lincoln's Birthday,Wednesday,128606,128003.0,0.5
2025-02-15,Susan B. Anthony Day,Saturday,148125,145686.0,1.7
2025-02-17,Washington's Birthday,Monday,94308,98246.0,-4.0
2025-05-26,Memorial Day,Monday,98478,135273.0,-27.2
2025-06-19,Juneteenth National Independence Day,Thursday,143551,166905.0,-14.0
2025-07-04,Independence Day,Friday,88000,128147.0,-31.3
2025-09-01,Labor Day,Monday,97739,124085.0,-21.2
2025-10-13,Columbus Day,Monday,111776,120305.0,-7.1



10 most unusually LOW days:


,weekday,ratio,holiday
2025-12-25,Thursday,0.398,Christmas Day
2025-12-26,Friday,0.490,
2025-12-27,Saturday,0.527,
2025-11-27,Thursday,0.612,Thanksgiving Day
2025-01-02,Thursday,0.636,
2025-12-24,Wednesday,0.668,
2025-11-28,Friday,0.669,
2025-12-28,Sunday,0.686,
2025-07-04,Friday,0.687,Independence Day
2025-07-05,Saturday,0.694,



5 most unusually HIGH days:


,weekday,ratio,holiday
2025-12-14,Sunday,1.205,
2025-12-15,Monday,1.203,
2025-11-01,Saturday,1.187,
2025-04-27,Sunday,1.187,
2025-06-23,Monday,1.182,


<Figure size 1400x400 with 1 Axes>

In [ ]:
import requests

print("WEATHER EFFECTS ON DEMAND")
print("=" * 55)

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params=dict(
    latitude=40.7128, longitude=-74.006, start_date="2025-01-01", end_date="2025-12-31",
    hourly="temperature_2m,precipitation,snowfall,wind_speed_10m",
    timezone="America/New_York"), timeout=60)
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"])
weather["time"] = pd.to_datetime(weather["time"])
weather = (weather.drop_duplicates("time").set_index("time")
           .reindex(final_demand_matrix.index).ffill())
assert weather.notna().all().all()
weather.to_parquet(SAVE + "weather_2025.parquet")
print("Weather data aligned:", weather.shape)

# Demand relative to the expected level for that month and hour-of-week
how = city.index.dayofweek * 24 + city.index.hour
rel = city / city.groupby([city.index.month, how]).transform("mean")

precip_cat = pd.cut(weather["precipitation"], [-0.01, 0, 1, 5, np.inf],
                    labels=["Dry", "Light (≤1 mm/h)", "Moderate (1–5 mm/h)", "Heavy (>5 mm/h)"])
temp_cat = pd.cut(weather["temperature_2m"], [-np.inf, 0, 10, 20, 30, np.inf],
                  labels=["<0°C", "0–10°C", "10–20°C", "20–30°C", ">30°C"])

print("\nDemand relative to expected, by precipitation:")
display(rel.groupby(precip_cat, observed=False).agg(["mean", "count"]).round(3))
print("\nDemand relative to expected, by temperature:")
display(rel.groupby(temp_cat, observed=False).agg(["mean", "count"]).round(3))
print("\nSnow hours:", int((weather["snowfall"] > 0).sum()),
      "| Mean relative demand during snow:", round(rel[weather["snowfall"] > 0].mean(), 3))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
rel.groupby(precip_cat, observed=False).mean().plot(kind="bar", ax=ax[0],
    title="Relative Demand by Precipitation Intensity")
rel.groupby(temp_cat, observed=False).mean().plot(kind="bar", ax=ax[1],
    title="Relative Demand by Temperature")
for a in ax:
    a.axhline(1, ls="--", c="gray"); a.set_ylabel("Actual / expected"); a.tick_params(axis="x", rotation=20)
plt.tight_layout(); plt.show()

WEATHER EFFECTS ON DEMAND
Weather data aligned: (8760, 4)

Demand relative to expected, by precipitation:


,mean,count
precipitation,,
Dry,0.994,7552
Light (≤1 mm/h),1.024,922
Moderate (1–5 mm/h),1.077,251
Heavy (>5 mm/h),1.064,35



Demand relative to expected, by temperature:


,mean,count
temperature_2m,,
<0°C,1.005,1340
0–10°C,0.995,2307
10–20°C,1.000,2621
20–30°C,0.999,2170
>30°C,1.024,322



Snow hours: 135 | Mean relative demand during snow: 1.017


<Figure size 1400x400 with 2 Axes>

In [ ]:
print("OUTLIER DETECTION")
print("=" * 55)

Z_THRESH = 5        # robust z-score threshold
MIN_EXCESS = 20     # minimum absolute deviation (pickups) to count as an outlier

zone_info = zones.set_index("LocationID").loc[zone_ids, ["Borough", "Zone"]]
C = final_demand_matrix.astype(float)
L = np.log1p(C)
how = C.index.dayofweek * 24 + C.index.hour

# ---------- Zone-level outliers (robust z-score vs same hour-of-week) ----------
med = L.groupby(how).transform("median")
mad = (L - med).abs().groupby(how).transform("median")
scale = (1.4826 * mad).clip(lower=0.25)     # floor avoids divide-by-zero in sparse zones
rz = (L - med) / scale
expected = np.expm1(med)
excess = C - expected
flag = (rz.abs() > Z_THRESH) & (excess.abs() >= MIN_EXCESS)

n_flag = int(flag.to_numpy().sum())
n_high = int((flag & (excess > 0)).to_numpy().sum())
print("\nA. ZONE-LEVEL OUTLIERS")
print(f"Zone-hours checked: {flag.size:,}")
print(f"Flagged: {n_flag:,} ({n_flag / flag.size:.3%}) | spikes: {n_high:,} | drops: {n_flag - n_high:,}")

out = pd.DataFrame({"count": C.stack(), "expected": expected.stack(),
                    "robust_z": rz.stack(), "excess": excess.stack()})[flag.stack()].reset_index()
out.columns = ["time", "zone", "count", "expected", "robust_z", "excess"]
out["Zone"] = out["zone"].map(zone_info["Zone"])
out["Borough"] = out["zone"].map(zone_info["Borough"])

print("\nTop 15 spikes:")
display(out.nlargest(15, "excess").round(1))
print("\nTop 10 drops:")
display(out.nsmallest(10, "excess").round(1))
print("\nZones with the most flagged hours:")
print(out.groupby("Zone").size().nlargest(10))
print("\nDays with the most flagged zone-hours (clusters suggest citywide events):")
display(out.groupby(out["time"].dt.date)
          .agg(flagged=("zone", "size"), zones=("zone", "nunique"),
               spikes=("excess", lambda e: (e > 0).sum()))
          .nlargest(10, "flagged"))
print("\nFlagged zone-hours by month:")
print(out.groupby(out["time"].dt.month).size())

# ---------- Citywide hourly outliers ----------
city = C.sum(axis=1)
lc = np.log1p(city)
cmed = lc.groupby(how).transform("median")
cmad = (lc - cmed).abs().groupby(how).transform("median")
crz = (lc - cmed) / (1.4826 * cmad)
city_out = pd.DataFrame({"pickups": city, "expected": np.expm1(cmed).round(),
                         "robust_z": crz.round(2)})[crz.abs() > 4]
print("\nB. CITYWIDE HOURLY OUTLIERS (|robust z| > 4):", len(city_out))
display(city_out.sort_values("robust_z"))

ax = city.plot(figsize=(14, 4), lw=0.5, title="Citywide Hourly Pickups with Flagged Outlier Hours")
ax.scatter(city_out.index, city_out["pickups"], c="red", s=15, zorder=3, label="Outlier hour")
ax.set_ylabel("Pickups per hour"); ax.legend(); plt.show()

# --------- Zero runs in busy zones (possible data outages) ----------
busy = C.columns[C.mean() >= 10]
runs = []
for z in busy:
    s = C[z] == 0
    if not s.any():
        continue
    blocks = (pd.DataFrame({"zero": s, "t": C.index})
              .groupby((s != s.shift()).cumsum())
              .agg(zero=("zero", "first"), hours=("zero", "size"), start=("t", "first")))
    for _, b in blocks[blocks["zero"] & (blocks["hours"] >= 3)].iterrows():
        runs.append({"zone": z, "Zone": zone_info.loc[z, "Zone"],
                     "start": b["start"], "hours": b["hours"]})
print(f"\nC. ZERO RUNS (≥3 consecutive hours) IN {len(busy)} BUSY ZONES (mean ≥ 10/hr):", len(runs))
if runs:
    display(pd.DataFrame(runs).sort_values("hours", ascending=False).head(15))
zero_share = (C[busy] == 0).mean(axis=1)
print("Hours with the highest share of busy zones at zero:")
print(zero_share.nlargest(5).round(3))

# ---------- Concentration of restored zero-duration records ----------
print("\nD. RESTORED ZERO-DURATION RECORDS")
if "strict_demand_matrix" in globals():
    restored = updated_demand_matrix - strict_demand_matrix
    rs = pd.DataFrame({"restored": restored.sum(),
                       "total": updated_demand_matrix.sum()}).join(zone_info)
    rs["restored_share"] = rs["restored"] / rs["total"]
    print(f"Overall restored share: {rs['restored'].sum() / rs['total'].sum():.2%}")
    print("\nMonthly restored share:")
    print((restored.resample("ME").sum().sum(axis=1) /
           updated_demand_matrix.resample("ME").sum().sum(axis=1)).round(4))
    print("\nZones with the highest restored share (≥10,000 total pickups):")
    display(rs[rs["total"] >= 10000].nlargest(10, "restored_share").round(3))
else:
    print("strict_demand_matrix not in memory (runtime restarted) — skipping Part D.")

# --------- What are the zero-duration records? (June sample) ----------
print("\nE. PROFILE OF ZERO/NEGATIVE-DURATION RECORDS (JUNE 2025)")
june = pd.read_parquet(
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-06.parquet",
    columns=["VendorID", "tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance",
             "fare_amount", "total_amount", "payment_type", "PULocationID", "DOLocationID"])
june = june[(june["tpep_pickup_datetime"] >= "2025-06-01") &
            (june["tpep_pickup_datetime"] < "2025-07-01") &
            june["PULocationID"].isin(zone_ids)]
zd = june["tpep_dropoff_datetime"] <= june["tpep_pickup_datetime"]
grp = np.where(zd, "zero/neg duration", "valid duration")

def profile(d):
    return pd.Series({
        "records": len(d),
        "median_distance_mi": d["trip_distance"].median(),
        "share_distance_0": (d["trip_distance"] == 0).mean(),
        "median_fare": d["fare_amount"].median(),
        "share_fare_le_0": (d["fare_amount"] <= 0).mean(),
        "share_same_PU_DO": (d["PULocationID"] == d["DOLocationID"]).mean()})

display(pd.DataFrame({"zero/neg duration": profile(june[zd]),
                      "valid duration": profile(june[~zd])}).round(3))
pay_names = {0: "Flex fare", 1: "Credit card", 2: "Cash", 3: "No charge",
             4: "Dispute", 5: "Unknown", 6: "Voided"}
print("Payment type distribution:")
display(pd.crosstab(june["payment_type"].map(pay_names), grp, normalize="columns").round(3))
print("Vendor distribution:")
display(pd.crosstab(june["VendorID"], grp, normalize="columns").round(3))

# Save flags for later analysis (e.g., evaluating model errors on outlier hours)
flag.rename(columns=str).to_parquet(SAVE + "outlier_flags.parquet")
del june; gc.collect()
print("\nOUTLIER DETECTION COMPLETED!")

OUTLIER DETECTION

A. ZONE-LEVEL OUTLIERS
Zone-hours checked: 2,303,880
Flagged: 1,682 (0.073%) | spikes: 320 | drops: 1,362

Top 15 spikes:


,time,zone,count,expected,robust_z,excess,Zone,Borough
987,2025-09-02 00:00:00,186,521.0,57.0,6.0,464.0,Penn Station/Madison Sq West,Manhattan
51,2025-01-01 02:00:00,79,492.0,37.0,8.1,455.0,East Village,Manhattan
83,2025-01-01 03:00:00,79,447.0,26.0,6.3,421.0,East Village,Manhattan
17,2025-01-01 01:00:00,79,474.0,58.0,7.1,416.0,East Village,Manhattan
5,2025-01-01 00:00:00,142,354.0,34.0,7.0,320.0,Lincoln Square East,Manhattan
20,2025-01-01 01:00:00,107,306.0,17.0,7.8,289.0,Gramercy,Manhattan
916,2025-08-26 23:00:00,93,289.0,0.0,22.7,289.0,Flushing Meadows-Corona Park,Queens
91,2025-01-01 03:00:00,148,272.0,7.0,7.5,265.0,Lower East Side,Manhattan
943,2025-08-31 22:00:00,93,267.0,2.0,7.5,265.0,Flushing Meadows-Corona Park,Queens
47,2025-01-01 02:00:00,48,289.0,26.0,9.4,263.0,Clinton East,Manhattan



Top 10 drops:


,time,zone,count,expected,robust_z,excess,Zone,Borough
1548,2025-12-27 01:00:00,79,64.0,740.0,-9.7,-676.0,East Village,Manhattan
1610,2025-12-28 01:00:00,79,157.0,823.0,-6.6,-666.0,East Village,Manhattan
1529,2025-12-27 00:00:00,79,106.0,745.5,-7.8,-639.5,East Village,Manhattan
1600,2025-12-28 00:00:00,79,154.0,767.0,-6.4,-613.0,East Village,Manhattan
883,2025-07-06 01:00:00,79,220.0,823.0,-5.3,-603.0,East Village,Manhattan
1183,2025-11-29 00:00:00,79,163.0,745.5,-6.1,-582.5,East Village,Manhattan
1186,2025-11-29 01:00:00,79,159.0,740.0,-6.1,-581.0,East Village,Manhattan
1020,2025-09-23 17:00:00,161,64.0,629.5,-9.1,-565.5,Midtown Center,Manhattan
878,2025-07-06 00:00:00,79,213.0,767.0,-5.1,-554.0,East Village,Manhattan
1021,2025-09-23 18:00:00,161,132.0,684.0,-6.6,-552.0,Midtown Center,Manhattan



Zones with the most flagged hours:
Zone
LaGuardia Airport               72
Brooklyn Navy Yard              69
Union Sq                        61
Battery Park City               54
Flushing Meadows-Corona Park    53
West Village                    52
Greenwich Village North         46
Upper East Side North           45
East Village                    42
TriBeCa/Civic Center            40
dtype: int64

Days with the most flagged zone-hours (clusters suggest citywide events):


,flagged,zones,spikes
time,,,
2025-01-01,365,65,113
2025-12-25,211,50,2
2025-11-27,84,33,0
2025-01-04,82,30,0
2025-12-26,72,30,0
2025-12-27,72,25,0
2025-01-05,66,25,0
2025-01-02,52,28,0
2025-07-04,50,26,2


... (10 outputs left out)

col_0,valid duration,zero/neg duration
payment_type,,
Cash,0.089,0.134
Credit card,0.596,0.849
Dispute,0.023,0.005
Flex fare,0.286,0.004
No charge,0.006,0.008
Unknown,0.000,0.000


Vendor distribution:


col_0,valid duration,zero/neg duration
VendorID,,
1,0.195,0.010
2,0.805,0.001
6,0.000,0.003
7,0.000,0.986



OUTLIER DETECTION COMPLETED!


In [ ]:
import gc

print("OUTLIER HANDLING")
print("=" * 55)

# Not every unusual pickup count is an error. How do we remove suspicious records without losing genuine demand spikes?

raw = pd.read_parquet(SAVE + "demand_2025_raw.parquet")   # pre-DST, includes restored records
raw.columns = raw.columns.astype(int)
assert raw.shape == (8760, 263)

# ---------- Remove phantom zero-duration records ----------
removed = pd.DataFrame(0, index=raw.index, columns=raw.columns)
phantom_report = []

for month in range(1, 13):
    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)
    url = ("https://d37ci6vzurychx.cloudfront.net/"
           f"trip-data/yellow_tripdata_2025-{month:02d}.parquet")
    t = pd.read_parquet(url, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                                      "PULocationID", "trip_distance",
                                      "fare_amount", "payment_type"])

    base = ((t["tpep_pickup_datetime"] >= start) & (t["tpep_pickup_datetime"] < end) &
            t["PULocationID"].isin(zone_ids))
    zero_dur = base & (t["tpep_dropoff_datetime"] <= t["tpep_pickup_datetime"])
    phantom = zero_dur & (
        (t["payment_type"] == 6) |                                   # voided trip
        (t["fare_amount"] < 0) |                                     # reversal/refund record
        ((t["trip_distance"] <= 0) & (t["fare_amount"] <= 0))        # no distance and no fare
    )

    p = t.loc[phantom, ["tpep_pickup_datetime", "PULocationID"]]
    cnt = (p.groupby([p["tpep_pickup_datetime"].dt.floor("h"), p["PULocationID"]])
            .size().unstack(fill_value=0)
            .reindex(index=raw.index, columns=raw.columns, fill_value=0))
    removed = removed + cnt

    phantom_report.append({
        "month": start.strftime("%Y-%m"),
        "zero_duration": int(zero_dur.sum()),
        "phantom_removed": int(phantom.sum()),
        "real_trips_kept": int(zero_dur.sum() - phantom.sum()),
    })
    print(f"{start.strftime('%B')}: {int(zero_dur.sum()):,} zero-duration | "
          f"{int(phantom.sum()):,} phantom removed")
    del t, p, cnt; gc.collect()

phantom_report = pd.DataFrame(phantom_report)
clean = (raw - removed).astype(int)
assert (clean.to_numpy() >= 0).all()
assert int(removed.to_numpy().sum()) == phantom_report["phantom_removed"].sum()

# ---------- Re-apply the DST adjustment ----------
spring, fall = pd.Timestamp("2025-03-09 02:00"), pd.Timestamp("2025-11-02 01:00")
one_hr = pd.Timedelta(hours=1)
clean.loc[spring] = clean.loc[spring - one_hr]
clean.loc[fall] = (clean.loc[fall] / 2).round().astype(int)

# ---------- Impute data gaps: zero pickups where the zone normally has ≥10 ----------
how = clean.index.dayofweek * 24 + clean.index.hour
fit_rows = (split_for_index(clean.index) == "train") & ~clean.index.isin([spring, fall])

def training_hour_of_week(values, group_stat):
    table = values.loc[fit_rows].groupby(how[fit_rows]).agg(group_stat)
    aligned = table.reindex(how)
    aligned.index = values.index
    return aligned

expected = np.expm1(training_hour_of_week(np.log1p(clean.astype(float)), "median"))
gap = (clean == 0) & (expected >= 10)

print(f"\nData-gap cells (0 pickups where ≥10 expected): {int(gap.to_numpy().sum()):,}")
if gap.to_numpy().any():
    g = gap.stack()
    g = g[g].reset_index()
    g.columns = ["time", "zone", "_"]
    g["Zone"] = g["zone"].map(zone_info["Zone"])
    g["expected"] = [round(expected.at[r.time, r.zone]) for r in g.itertuples()]
    print("Gap cells by zone:")
    print(g.groupby("Zone").size().sort_values(ascending=False).head(10))
    print("\nGap cells by date:")
    print(g.groupby(g["time"].dt.date).size().sort_values(ascending=False).head(10))

final = clean.mask(gap, expected.round()).astype(int)

# --------- Flag real spikes/drops (kept, not modified) ----------
Lf = np.log1p(final.astype(float))
med = training_hour_of_week(Lf, "median")
scale = (1.4826 * training_hour_of_week((Lf - med).abs(), "median")).clip(lower=0.25)
rz = (Lf - med) / scale
event_flag = (rz.abs() > 5) & ((final - np.expm1(med)).abs() >= 20) & ~gap

# Cells excluded from evaluation: imputed gaps + DST-adjusted hours
eval_exclude = gap.copy()
eval_exclude.loc[[spring, fall]] = True

# ---------- Validation and summary ----------
assert final.shape == (8760, 263)
assert not final.isna().any().any()
assert (final.to_numpy() >= 0).all()
assert final.index.is_unique and final.index.is_monotonic_increasing

raw_total = int(raw.to_numpy().sum())
phantom_total = int(phantom_report["phantom_removed"].sum())
final_total = int(final.to_numpy().sum())

print("\nSUMMARY")
print("=" * 55)
print(f"Pickups before handling:        {raw_total:,}")
print(f"Phantom records removed:        {phantom_total:,} ({phantom_total / raw_total:.2%})")
print(f"Zero-duration real trips kept:  {int(phantom_report['real_trips_kept'].sum()):,}")
print(f"Gap cells imputed:              {int(gap.to_numpy().sum()):,}")
print(f"Event outliers flagged (kept):  {int(event_flag.to_numpy().sum()):,}")
print(f"Cells excluded from evaluation: {int(eval_exclude.to_numpy().sum()):,}")
print(f"Final total pickups:            {final_total:,}")

display(phantom_report)

phantom_report.set_index("month")[["real_trips_kept", "phantom_removed"]].plot(
    kind="bar", stacked=True, figsize=(12, 4),
    title="Zero-Duration Records by Month: Real Trips Kept vs Phantom Records Removed")
plt.ylabel("Records"); plt.xticks(rotation=45); plt.tight_layout(); plt.show()

# ---------- Save ----------
final_demand_matrix = final
final.rename(columns=str).to_parquet(SAVE + "demand_2025_clean.parquet")
event_flag.rename(columns=str).to_parquet(SAVE + "event_flags.parquet")
eval_exclude.rename(columns=str).to_parquet(SAVE + "eval_exclude.parquet")
print("\nOUTLIER HANDLING COMPLETED — use demand_2025_clean.parquet from here on.")

OUTLIER HANDLING
January: 2,003 zero-duration | 46 phantom removed
February: 5,121 zero-duration | 46 phantom removed
March: 22,169 zero-duration | 48 phantom removed
April: 34,706 zero-duration | 70 phantom removed
May: 64,200 zero-duration | 48 phantom removed
June: 68,541 zero-duration | 44 phantom removed
July: 56,003 zero-duration | 44 phantom removed
August: 47,774 zero-duration | 37 phantom removed
September: 57,113 zero-duration | 43 phantom removed
October: 67,908 zero-duration | 62 phantom removed
November: 62,076 zero-duration | 154 phantom removed
December: 57,975 zero-duration | 34 phantom removed

Data-gap cells (0 pickups where ≥10 expected): 483
Gap cells by zone:
Zone
LaGuardia Airport                 50
Bushwick South                    28
Washington Heights South          27
Williamsburg (North Side)         27
Williamsburg (South Side)         27
East Williamsburg                 23
Crown Heights North               22
Long Island City/Hunters Point    20
DUMBO/Vine

,month,zero_duration,phantom_removed,real_trips_kept
0,2025-01,2003,46,1957
1,2025-02,5121,46,5075
2,2025-03,22169,48,22121
3,2025-04,34706,70,34636
4,2025-05,64200,48,64152
5,2025-06,68541,44,68497
6,2025-07,56003,44,55959
7,2025-08,47774,37,47737
8,2025-09,57113,43,57070
9,2025-10,67908,62,67846


<Figure size 1200x400 with 1 Axes>


OUTLIER HANDLING COMPLETED — use demand_2025_clean.parquet from here on.


In [ ]:
import holidays

print("STEP 18: FEATURE ENGINEERING")
print("=" * 55)

# Load the cleaned demand matrix from Step 17
final_demand_matrix = pd.read_parquet(SAVE + "demand_2025_clean.parquet")
final_demand_matrix.columns = final_demand_matrix.columns.astype(int)
idx = final_demand_matrix.index

hour_split = split_for_index(idx)
train_mask = hour_split == "train"

weather = pd.read_parquet(SAVE + "weather_2025.parquet").reindex(idx)
dst_flag = pd.read_parquet(SAVE + "dst_flag.parquet")["dst_flag"].reindex(idx).fillna(0)

# Federal holidays (2026 included so Dec 31 gets its pre-holiday flag)
us_hol = holidays.US(years=[2025, 2026])
hol_days = pd.to_datetime(list(us_hol.keys()))
day = idx.normalize()

context = pd.DataFrame(index=idx)

# Cyclical time encodings (hour 23 is next to hour 0; Sunday next to Monday)
context["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
context["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
context["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
context["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
context["is_weekend"] = (idx.dayofweek >= 5).astype(int)

# Holiday indicators
context["is_holiday"] = day.isin(hol_days).astype(int)
context["pre_holiday"] = (day + pd.Timedelta(days=1)).isin(hol_days).astype(int)
context["post_holiday"] = (day - pd.Timedelta(days=1)).isin(hol_days).astype(int)
context["dst_flag"] = dst_flag.astype(int).to_numpy()

# Weather: standardized with TRAINING statistics only (no leakage)
wx_cols = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]
wx_stats = {}
for c in wx_cols:
    mu, sd = weather.loc[train_mask, c].mean(), weather.loc[train_mask, c].std()
    sd = sd if np.isfinite(sd) and sd > 1e-6 else 1.0
    wx_stats[c] = (mu, sd)
    context[c] = (weather[c] - mu) / sd
context["is_raining"] = (weather["precipitation"] > 0.1).astype(int)

# ---------- Validation ----------
assert len(context) == 8760
assert not context.isna().any().any()
assert context.index.equals(final_demand_matrix.index)

print("Context feature matrix:", context.shape)
print("Features:", context.columns.tolist())

print("\nSplit sizes (hours):")
for name in month_groups:
    print(f"  {name} ({split_names[name]}): {(hour_split == name).sum():,}")

print("\nHolidays in the data:")
for d in sorted(hol_days[(hol_days >= idx.min().normalize()) & (hol_days <= idx.max())]):
    split = month_to_split[d.month]
    print(f"  {d.date()} ({d.day_name()}): {us_hol.get(d.date())} [{split}]")

print("\nWeather statistics used for scaling (training period):")
display(pd.DataFrame(wx_stats, index=["mean", "std"]).T.round(3))

print("\nFeature summary:")
display(context.describe().T[["mean", "std", "min", "max"]].round(3))

print("\nCorrelation with citywide log demand:")
city_log = np.log1p(final_demand_matrix.sum(axis=1))
print(context.corrwith(city_log).round(3).sort_values())

# Visual check: one sample week
fig, ax = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
wk = slice("2025-07-07", "2025-07-13")
ax[0].plot(city_log.loc[wk]); ax[0].set_title("Citywide log demand (sample week)")
ax[1].plot(context.loc[wk, ["hour_sin", "hour_cos"]]); ax[1].set_title("Cyclical hour encoding")
ax[2].plot(context.loc[wk, ["temperature_2m", "precipitation"]]); ax[2].set_title("Standardized weather")
ax[2].legend(["temperature", "precipitation"])
plt.tight_layout(); plt.show()

context.to_parquet(SAVE + "context_features.parquet")
print("\nFEATURE ENGINEERING COMPLETED!")
pd.DataFrame(wx_stats, index=["mean", "std"]).to_csv(SAVE + "weather_scaling.csv")


STEP 18: FEATURE ENGINEERING
Context feature matrix: (8760, 14)
Features: ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday', 'pre_holiday', 'post_holiday', 'dst_flag', 'temperature_2m', 'precipitation', 'snowfall', 'wind_speed_10m', 'is_raining']

Split sizes (hours):
  train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec): 5,904
  val (May, Nov): 1,464
  test (Feb, Sep): 1,392

Holidays in the data:
  2025-01-01 (Wednesday): New Year's Day [train]
  2025-01-20 (Monday): Martin Luther King Jr. Day [train]
  2025-02-17 (Monday): Washington's Birthday [test]
  2025-05-26 (Monday): Memorial Day [val]
  2025-06-19 (Thursday): Juneteenth National Independence Day [train]
  2025-07-04 (Friday): Independence Day [train]
  2025-09-01 (Monday): Labor Day [test]
  2025-10-13 (Monday): Columbus Day [train]
  2025-11-11 (Tuesday): Veterans Day [val]
  2025-11-27 (Thursday): Thanksgiving Day [val]
  2025-12-25 (Thursday): Christmas Day [train]

Weather statistics used for scaling (

,mean,std
temperature_2m,13.045,11.477
precipitation,0.112,0.601
snowfall,0.005,0.063
wind_speed_10m,10.151,5.398



Feature summary:


,mean,std,min,max
hour_sin,-0.000,0.707,-1.000,1.000
hour_cos,-0.000,0.707,-1.000,1.000
dow_sin,0.003,0.708,-0.975,0.975
dow_cos,-0.001,0.706,-0.901,1.000
is_weekend,0.285,0.451,0.000,1.000
is_holiday,0.030,0.171,0.000,1.000
pre_holiday,0.030,0.171,0.000,1.000
post_holiday,0.030,0.171,0.000,1.000
dst_flag,0.000,0.015,0.000,1.000
temperature_2m,-0.044,0.935,-2.592,2.383



Correlation with citywide log demand:
hour_sin         -0.654
hour_cos         -0.391
dow_sin          -0.087
dow_cos          -0.074
is_holiday       -0.040
post_holiday     -0.038
pre_holiday      -0.009
snowfall          0.003
dst_flag          0.008
precipitation     0.031
is_raining        0.067
is_weekend        0.083
wind_speed_10m    0.127
temperature_2m    0.193
dtype: float64


<Figure size 1400x700 with 3 Axes>


FEATURE ENGINEERING COMPLETED!


In [ ]:
import gc, glob
import geopandas as gpd
from matplotlib.collections import LineCollection

print("GRAPH CONSTRUCTION")
print("=" * 55)

N = len(zone_ids)
K = 8
pos = {int(z): i for i, z in enumerate(zone_ids)}
zone_info = zones.set_index("LocationID").loc[zone_ids, ["Borough", "Zone"]]

# Zone geometry
if "gdf" not in globals():
    !wget -q -nc https://d37ci6vzurychx.cloudfront.net/misc/taxi_zones.zip && unzip -oq taxi_zones.zip -d tz
    gdf = gpd.read_file(glob.glob("tz/**/*.shp", recursive=True)[0]).to_crs(2263)
    gdf = gdf.dissolve(by="LocationID").reindex(zone_ids)
cent = np.c_[gdf.centroid.x, gdf.centroid.y]

# ---------- Contiguity graph  ----------
A_adj = np.load(SAVE + "A_adj.npy")
D = np.load(SAVE + "D_miles.npy")

# --------- Distance graph: Gaussian kernel over K nearest zones ----------
nn = np.argsort(D, axis=1)[:, 1:K + 1]
sigma = np.take_along_axis(D, nn, 1).mean()
rows = np.repeat(np.arange(N), K)
A_dist = np.zeros((N, N))
A_dist[rows, nn.ravel()] = np.exp(-(D[rows, nn.ravel()] / sigma) ** 2)
A_dist = np.maximum(A_dist, A_dist.T)
print(f"Distance kernel bandwidth (sigma): {sigma:.2f} miles")

# ---------- OD-flow graph: training months only ----------
print("\nBuilding OD-flow matrix from training months:")
od = np.zeros((N, N))
for month in month_groups["train"]:
    url = ("https://d37ci6vzurychx.cloudfront.net/"
           f"trip-data/yellow_tripdata_2025-{month:02d}.parquet")
    t = pd.read_parquet(url, columns=["tpep_pickup_datetime", "tpep_dropoff_datetime",
                                      "PULocationID", "DOLocationID"])
    start = pd.Timestamp(2025, month, 1)
    end = start + pd.DateOffset(months=1)
    m = ((t["tpep_pickup_datetime"] >= start) & (t["tpep_pickup_datetime"] < end) &
         (t["tpep_dropoff_datetime"] > t["tpep_pickup_datetime"]) &      # real completed trips
         t["PULocationID"].isin(zone_ids) & t["DOLocationID"].isin(zone_ids))
    g = t.loc[m].groupby(["PULocationID", "DOLocationID"]).size()
    i = g.index.get_level_values(0).map(pos).to_numpy()
    j = g.index.get_level_values(1).map(pos).to_numpy()
    np.add.at(od, (i, j), g.to_numpy())
    print(f"  {start.strftime('%B')}: {int(g.sum()):,} trips")
    del t, g; gc.collect()

print(f"\nTotal OD trips: {int(od.sum()):,}")
print(f"Intra-zone trip share (pickup = dropoff zone): {np.trace(od) / od.sum():.1%}")

flow = od + od.T
np.fill_diagonal(flow, 0)

def top_k_graph(M, k=10):
    """Keep each zone's k strongest links, scale rows to [0, 1], symmetrize."""
    out = np.zeros_like(M, dtype=float)
    top = np.argsort(-M, axis=1)[:, :k]
    r = np.repeat(np.arange(len(M)), k)
    out[r, top.ravel()] = M[r, top.ravel()]
    out[M <= 0] = 0
    out = out / np.maximum(out.max(1, keepdims=True), 1e-9)
    return np.maximum(out, out.T)

A_od = top_k_graph(np.log1p(flow), k=10)

print("\nTop 10 zone pairs by two-way trip flow:")
flat = np.argsort(-np.triu(flow), axis=None)[:10]
for a, b in zip(*np.unravel_index(flat, flow.shape)):
    print(f"  {zone_info.iloc[a]['Zone']} <-> {zone_info.iloc[b]['Zone']}: "
          f"{int(flow[a, b]):,} trips ({D[a, b]:.1f} mi apart)")

# ---------- Correlation graph: residual demand co-movement, training only ----------
Ltr = np.log1p(final_demand_matrix.to_numpy()[train_mask].astype(float))
how_tr = (idx.dayofweek * 24 + idx.hour).to_numpy()[train_mask]
resid = Ltr - pd.DataFrame(Ltr).groupby(how_tr).transform("mean").to_numpy()
with np.errstate(invalid="ignore", divide="ignore"):
    Cr = np.nan_to_num(np.corrcoef(resid.T))
np.fill_diagonal(Cr, 0)
A_corr = top_k_graph(np.clip(Cr, 0, None), k=10)

# ---------- Comparison ----------
graphs = {"Contiguity": A_adj, "Distance (kNN)": A_dist, "OD flow": A_od, "Correlation": A_corr}

def jaccard(a, b):
    a, b = a > 0, b > 0
    return (a & b).sum() / (a | b).sum()

summary = pd.DataFrame({
    name: {"edges": int((A > 0).sum() / 2),
           "mean_degree": round((A > 0).sum(1).mean(), 1),
           "isolated_nodes": int(((A > 0).sum(1) == 0).sum()),
           "mean_edge_length_mi": round(D[A > 0].mean(), 2),
           "overlap_with_contiguity": round(jaccard(A, A_adj), 3)}
    for name, A in graphs.items()}).T
print("\nGraph comparison:")
display(summary)

for name, A in graphs.items():
    assert A.shape == (N, N) and np.allclose(A, A.T) and (np.diag(A) == 0).all()

# ---------- Visualize each graph on the map ----------
fig, axes = plt.subplots(2, 2, figsize=(16, 16))
for ax, (name, A) in zip(axes.ravel(), graphs.items()):
    gdf.boundary.plot(ax=ax, lw=0.2, color="lightgray")
    i, j = np.where(np.triu(A) > 0)
    segs = np.stack([cent[i], cent[j]], axis=1)
    ax.add_collection(LineCollection(segs, lw=0.5, alpha=0.6,
                                     colors=plt.cm.viridis(A[i, j] / A.max())))
    ax.scatter(cent[:, 0], cent[:, 1], s=3, c="black")
    ax.set_title(f"{name} graph ({len(i):,} edges)"); ax.axis("off")
plt.tight_layout(); plt.show()

np.savez(SAVE + "graphs.npz", A_adj=A_adj, A_dist=A_dist, A_od=A_od, A_corr=A_corr, od_raw=od)
print("\nGRAPH CONSTRUCTION COMPLETED!")

GRAPH CONSTRUCTION
Distance kernel bandwidth (sigma): 1.40 miles

Building OD-flow matrix from training months:
  January: 3,445,399 trips
  March: 4,095,210 trips
  April: 3,907,391 trips
  June: 4,222,213 trips
  July: 3,811,698 trips
  August: 3,494,844 trips
  October: 4,327,909 trips
  December: 4,215,345 trips

Total OD trips: 31,520,009
Intra-zone trip share (pickup = dropoff zone): 4.6%

Top 10 zone pairs by two-way trip flow:
  Upper East Side North <-> Upper East Side South: 351,729 trips (0.9 mi apart)
  Midtown Center <-> Upper East Side South: 178,440 trips (1.0 mi apart)
  Midtown Center <-> Upper East Side North: 140,218 trips (1.9 mi apart)
  Lincoln Square East <-> Upper West Side South: 134,364 trips (0.7 mi apart)
  Midtown East <-> Upper East Side South: 126,950 trips (0.9 mi apart)
  Upper West Side North <-> Upper West Side South: 122,627 trips (0.6 mi apart)
  Lenox Hill West <-> Upper East Side North: 119,525 trips (0.9 mi apart)
  Penn Station/Madison Sq West <

,edges,mean_degree,isolated_nodes,mean_edge_length_mi,overlap_with_contiguity
Contiguity,672.0,5.1,0.0,1.22,1.000
Distance (kNN),1254.0,9.5,0.0,1.52,0.494
OD flow,2320.0,17.6,3.0,4.59,0.170
Correlation,2421.0,18.4,3.0,6.46,0.105


<Figure size 1600x1600 with 4 Axes>


GRAPH CONSTRUCTION COMPLETED!


In [ ]:
import folium
import branca.colormap as bcm

print("\nINTERACTIVE GRAPH MAP")

# Zone polygons and centroids in lat/lon
zl = gdf[["geometry"]].reset_index(drop=True).copy()
zl["LocationID"] = [int(z) for z in zone_ids]
zl["Zone"] = zone_info["Zone"].values
zl["Borough"] = zone_info["Borough"].values
zl["mean_per_hour"] = final_demand_matrix.mean().round(1).values
short = {"Contiguity": "deg_adj", "Distance (kNN)": "deg_dist",
         "OD flow": "deg_od", "Correlation": "deg_corr"}
for name, A in graphs.items():
    zl[short[name]] = (A > 0).sum(1).astype(int)
zl = zl.to_crs(4326)

cent_ll = gdf.centroid.to_crs(4326)
lat, lon = cent_ll.y.values, cent_ll.x.values

m = folium.Map(location=[40.73, -73.94], zoom_start=11, tiles="cartodbpositron")

# Zone layer coloured by mean demand (log scale)
cmap = bcm.linear.YlOrRd_09.scale(0, np.log1p(zl["mean_per_hour"].max()))
cmap.caption = "log(1 + mean pickups per hour)"
folium.GeoJson(
    zl,
    name="Taxi zones (mean demand)",
    style_function=lambda f: {
        "fillColor": cmap(np.log1p(f["properties"]["mean_per_hour"])),
        "color": "#555", "weight": 0.4, "fillOpacity": 0.45},
    highlight_function=lambda f: {"weight": 2, "color": "black", "fillOpacity": 0.7},
    tooltip=folium.GeoJsonTooltip(
        fields=["LocationID", "Zone", "Borough", "mean_per_hour",
                "deg_adj", "deg_dist", "deg_od", "deg_corr"],
        aliases=["Zone ID", "Zone", "Borough", "Mean pickups/hr",
                 "Contiguity links", "Distance links", "OD-flow links", "Correlation links"]),
).add_to(m)
cmap.add_to(m)

# One toggleable layer per graph
colors = {"Contiguity": "#1f77b4", "Distance (kNN)": "#2ca02c",
          "OD flow": "#d62728", "Correlation": "#9467bd"}
for k, (name, A) in enumerate(graphs.items()):
    fg = folium.FeatureGroup(name=f"{name} edges", show=(k == 0))
    i, j = np.where(np.triu(A) > 0)
    for a, b in zip(i, j):
        tip = (f"<b>{zone_info.iloc[a]['Zone']} ↔ {zone_info.iloc[b]['Zone']}</b><br>"
               f"Edge weight: {A[a, b]:.2f}<br>Distance: {D[a, b]:.1f} mi")
        if name == "OD flow":
            tip += f"<br>Trips (training months): {int(flow[a, b]):,}"
        folium.PolyLine(
            [(lat[a], lon[a]), (lat[b], lon[b])],
            color=colors[name], weight=1 + 3 * A[a, b] / A.max(), opacity=0.65,
            tooltip=tip).add_to(fg)
    fg.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save(SAVE + "zone_graphs_interactive.html")
display(m)


INTERACTIVE GRAPH MAP


/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [ ]:
import os

print("STEP 20: DATA SPLIT, FORECASTING WINDOWS, AND EVALUATION FRAMEWORK")
print("=" * 65)

# ---------- Load cleaned inputs ----------
final_demand_matrix = pd.read_parquet(SAVE + "demand_2025_clean.parquet")
final_demand_matrix.columns = final_demand_matrix.columns.astype(int)
context = pd.read_parquet(SAVE + "context_features.parquet")
eval_exclude = pd.read_parquet(SAVE + "eval_exclude.parquet").to_numpy()
event_flags = pd.read_parquet(SAVE + "event_flags.parquet").to_numpy()

idx = final_demand_matrix.index
counts = final_demand_matrix.to_numpy().astype(np.float32)   # [T, N] pickups
L = np.log1p(counts)                                         # log scale for modelling
T, N = counts.shape

hour_split = split_for_index(idx)
train_mask = hour_split == "train"
test_mask = hour_split == "test"

T_IN = 12
H = 24
EVAL_H = [1, 3, 6, 12, 24]
MIN_HISTORY = 168

# Keep every observable input and all 24 target hours inside one calendar month.
origins = np.arange(MIN_HISTORY - 1, T - H)
months = idx.to_period("M").asi8
assert set(idx.month) == set(range(1, 13))
assert sorted(month for group in month_groups.values() for month in group) == list(range(1, 13))
origins = origins[months[origins - MIN_HISTORY + 1] == months[origins + H]]
train_o, val_o, test_o = (origins[hour_split[origins] == name] for name in month_groups)
grouped_o = np.concatenate([train_o, val_o, test_o])
full_order = np.argsort(grouped_o)
full_o = grouped_o[full_order]
test_positions = np.searchsorted(full_o, test_o)
assert np.array_equal(full_o, origins)
assert np.array_equal(full_o[test_positions], test_o)
for name, o in zip(month_groups, (train_o, val_o, test_o)):
    assert len(o) and np.all(hour_split[o - MIN_HISTORY + 1] == name)
    assert np.all(hour_split[o + H] == name)

def combine_predictions(preds):
    return {h: np.concatenate([preds[name][h] for name in month_groups])[full_order]
            for h in EVAL_H}

split_info = []
for month in range(1, 13):
    o = full_o[idx[full_o].month == month]
    split_info.append({"month": f"2025-{month:02d}", "split": month_to_split[month],
                       "origins": len(o), "first_target": idx[o[0] + 1],
                       "last_target": idx[o[-1] + H]})
split_info = pd.DataFrame(split_info)
display(split_info)
split_info.to_csv(SAVE + "month_split.csv", index=False)

# ---------- Scaling: per-zone statistics from TRAINING data only ----------
mu = L[train_mask].mean(0)
sd = L[train_mask].std(0)
sd = np.where(sd < 1e-3, 1.0, sd)        # always-zero zones: avoid divide-by-zero
L_scaled = (L - mu) / sd

active = counts[train_mask].mean(0) >= 1
print(f"\nActive zones (training mean ≥ 1 pickup/hr): {active.sum()} of {N}")
print(f"Share of training demand from active zones: "
      f"{counts[train_mask][:, active].sum() / counts[train_mask].sum():.2%}")

test_cells = eval_exclude[test_mask]
print(f"Test cells excluded from evaluation: {test_cells.sum():,} ({test_cells.mean():.3%})")
print(f"Test cells flagged as events:        {event_flags[test_mask].sum():,}")
print(f"Share of zero-demand test cells:     {(counts[test_mask] == 0).mean():.1%}")

# ---------- Shared evaluation function ----------
RESULTS_PATH = SAVE + "model_results.csv"
results = pd.DataFrame()

def evaluate(preds, orig, model, split="test", save=True):
    """
    preds: dict {horizon: array [samples, N]} of predicted PICKUP COUNTS
    orig:  forecast origins the predictions belong to
    """
    global results
    rows = []
    for h in EVAL_H:
        P = np.clip(np.asarray(preds[h], dtype=np.float64), 0, None)
        tgt = orig + h
        Y = counts[tgt].astype(np.float64)
        valid = ~eval_exclude[tgt]
        subsets = {"all": valid,
                   "active": valid & active[None, :],
                   "events": valid & event_flags[tgt]}
        for name, msk in subsets.items():
            if msk.sum() == 0:
                continue
            y, p = Y[msk], P[msk]
            e = p - y
            rows.append(dict(model=model, split=split, horizon=h, subset=name,
                             cells=int(msk.sum()),
                             MAE=np.abs(e).mean(),
                             RMSE=np.sqrt((e ** 2).mean()),
                             WMAPE=np.abs(e).sum() / max(y.sum(), 1e-9)))
    new = pd.DataFrame(rows)
    if save:
        if len(results):   # replace earlier runs of the same model/split
            results = results[~((results["model"] == model) & (results["split"] == split))]
        results = pd.concat([results, new], ignore_index=True)
        results.to_csv(RESULTS_PATH, index=False)
    return new

def show(split="test", subset="all"):
    r = results[(results["split"] == split) & (results["subset"] == subset)]
    t = r.pivot_table(index="model", columns="horizon", values=["MAE", "RMSE", "WMAPE"])
    return t.sort_values(("MAE", EVAL_H[0])).round(3)

# ---------- Sanity check: perfect predictions must score zero ----------
perfect = {h: counts[test_o + h] for h in EVAL_H}
chk = evaluate(perfect, test_o, "_perfect_check", save=False)
assert np.allclose(chk[["MAE", "RMSE", "WMAPE"]].to_numpy(), 0)
print("\nEvaluation function sanity check passed (perfect forecast → zero error).")

# ---------- Example model input shapes ----------
o = train_o[0]
x_demand = L_scaled[o - T_IN + 1: o + 1]              # [T_IN, N]
x_context = context.to_numpy()[o - T_IN + 1: o + 1]    # [T_IN, n_context]
y_target = L[o + 1: o + H + 1]                          # [H, N]
print(f"\nExample sample — demand input {x_demand.shape}, "
      f"context input {x_context.shape}, target {y_target.shape}")

# ---------- Visualize the split ----------
daily = final_demand_matrix.sum(axis=1).resample("D").sum()
ax = daily.plot(figsize=(14, 4), color="black", lw=1,
                title="Seeded Whole-Month Train / Validation / Test Split")
shade_month_splits(ax)
ax.set_ylabel("Pickups per day"); ax.legend(loc="lower left"); plt.show()

np.savez(SAVE + "prep.npz", train_o=train_o, val_o=val_o, test_o=test_o,
         mu=mu, sd=sd, active=active, full_o=full_o, split_seed=SPLIT_SEED,
         **{f"{name}_months": months for name, months in month_groups.items()})
print("\nSPLIT AND EVALUATION FRAMEWORK READY!")

STEP 20: DATA SPLIT, FORECASTING WINDOWS, AND EVALUATION FRAMEWORK


,month,split,origins,first_target,last_target
0,2025-01,train,553,2025-01-08,2025-01-31 23:00:00
1,2025-02,test,481,2025-02-08,2025-02-28 23:00:00
2,2025-03,train,553,2025-03-08,2025-03-31 23:00:00
3,2025-04,train,529,2025-04-08,2025-04-30 23:00:00
4,2025-05,val,553,2025-05-08,2025-05-31 23:00:00
5,2025-06,train,529,2025-06-08,2025-06-30 23:00:00
6,2025-07,train,553,2025-07-08,2025-07-31 23:00:00
7,2025-08,train,553,2025-08-08,2025-08-31 23:00:00
8,2025-09,test,529,2025-09-08,2025-09-30 23:00:00
9,2025-10,train,553,2025-10-08,2025-10-31 23:00:00



Active zones (training mean ≥ 1 pickup/hr): 156 of 263
Share of training demand from active zones: 99.18%
Test cells excluded from evaluation: 56 (0.015%)
Test cells flagged as events:        100
Share of zero-demand test cells:     40.1%

Evaluation function sanity check passed (perfect forecast → zero error).

Example sample — demand input (12, 263), context input (12, 14), target (24, 263)


<Figure size 1400x400 with 1 Axes>


SPLIT AND EVALUATION FRAMEWORK READY!


## DeepETA-inspired EDA and feature decisions

DeepETA motivates examining what an initial estimate misses, how errors vary across operating segments, and whether spatial context and discretized features are worth modeling. This section uses training months only. It compares persistence, daily and weekly lags and a seasonal profile at all five horizons, reports error quantiles and signed borough bias, and measures residual co-movement along distance, contiguity and OD-flow edges. It also summarizes clock-adjusted demand across training weather strata. These associations are descriptive. Model selection uses validation MAE.

The earlier full-year figures describe the source dataset. This section is the modeling EDA.


In [ ]:
# Model-design EDA uses only training-month observations.
how = np.asarray(idx.dayofweek * 24 + idx.hour)
train_origins = train_o
zone_ids_eda = final_demand_matrix.columns
boroughs = zone_info.reindex(zone_ids_eda)["Borough"].to_numpy()
valid_hourly = final_demand_matrix.where(~eval_exclude)
train_hourly = valid_hourly.iloc[train_mask]
profile = train_hourly.groupby(how[train_mask]).mean().fillna(0).to_numpy()
log_resid = L - np.log1p(profile[how])
train_resid = pd.DataFrame(log_resid[train_mask], columns=zone_ids_eda).mask(eval_exclude[train_mask])

# Does a seasonal estimate leave actionable error at each horizon?
eda_rows = []
for h in EVAL_H:
    target = train_origins + h
    estimates = {"persistence": counts[train_origins],
                 "daily lag": counts[target - 24],
                 "weekly lag": counts[target - 168],
                 "seasonal profile": profile[how[target]]}
    for borough in np.unique(boroughs):
        keep = (boroughs == borough)[None, :] & ~eval_exclude[target]
        for name, pred in estimates.items():
            err = (pred - counts[target])[keep]
            eda_rows.append({"borough": borough, "horizon": h, "baseline": name,
                             "MAE": np.abs(err).mean(), "signed_bias": err.mean(),
                             "p90_absolute_error": np.quantile(np.abs(err), .90)})
eda_horizons = pd.DataFrame(eda_rows)
display(eda_horizons.query("baseline == 'seasonal profile'")
        .pivot(index="borough", columns="horizon", values="signed_bias").round(2))
eda_horizons.to_csv(SAVE + "eda_training_borough_horizons.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(eda_horizons.groupby(["horizon", "baseline"])["MAE"].mean().unstack()
 .plot(ax=axes[0], marker="o", title="Training baseline MAE, equal weight per borough"))
axes[0].set(xlabel="Horizon (hours)", ylabel="MAE (pickups)", xticks=EVAL_H)
r = eda_horizons.query("baseline == 'seasonal profile'")
r.pivot(index="borough", columns="horizon", values="signed_bias").plot.bar(ax=axes[1])
axes[1].set(title="Seasonal forecast bias by borough", ylabel="Forecast minus actual (pickups)")
plt.tight_layout()
plt.show()

# Compare neighbor residual correlations with geographical non-neighbors.
graph_data = np.load(SAVE + "graphs.npz")
resid_corr = train_resid.corr(min_periods=168).to_numpy()
upper = np.triu(np.ones((N, N), dtype=bool), 1) & np.isfinite(resid_corr)
supports = ["A_adj", "A_dist", "A_od"]  # exclude correlation graph to avoid circular analysis
graph_rows = []
for label in supports:
    mask = graph_data[label] > 0
    linked = resid_corr[upper & mask]
    unlinked = resid_corr[upper & ~mask]
    graph_rows.append({"graph": label, "edge_pairs": len(linked),
                       "median_edge_correlation": np.median(linked),
                       "median_nonedge_correlation": np.median(unlinked)})
graph_eda = pd.DataFrame(graph_rows)
display(graph_eda.round(3))
graph_eda.to_csv(SAVE + "eda_training_graph_residuals.csv", index=False)

# Quantile strata are fitted on training weather. Control for clock using residual demand.
wx_report = pd.read_parquet(SAVE + "weather_2025.parquet").loc[idx[train_mask]]
city_resid = train_resid.mean(axis=1).to_numpy()
weather_rows = []
for column in ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m"]:
    values = wx_report[column]
    bins = pd.qcut(values, q=8, duplicates="drop")
    # Preserve zero-vs-positive strata when ties collapse quantile bins.
    if bins.isna().all() or bins.cat.categories.size < 2:
        bins = pd.Series(np.where(values > 0, "positive", "zero"), index=values.index)
    grouped = pd.DataFrame({"bin": bins, "residual": city_resid}).groupby("bin", observed=True)
    summary = grouped["residual"].agg(hours="count", median="median",
                                      q10=lambda x: x.quantile(.1), q90=lambda x: x.quantile(.9))
    weather_rows.append(summary.reset_index().assign(variable=column))
weather_eda = pd.concat(weather_rows, ignore_index=True)
display(weather_eda.round(3))
weather_eda.to_csv(SAVE + "eda_training_weather_residuals.csv", index=False)

print("At +24h the same-hour-yesterday lag is the forecast-origin count, so it equals persistence.")
print("Seasonal-profile errors above are in-sample diagnostics. Use the separate validation and test tables for accuracy claims.")


horizon,1,3,6,12,24
borough,,,,,
Bronx,0.00,0.00,0.00,0.00,-0.00
Brooklyn,0.01,0.00,0.00,0.00,-0.01
EWR,-0.00,-0.00,-0.00,0.00,0.00
Manhattan,-0.54,-0.53,-0.52,-0.47,-0.42
Queens,0.02,0.01,0.01,0.01,0.01
Staten Island,-0.00,-0.00,-0.00,-0.00,-0.00


<Figure size 1400x400 with 2 Axes>

,graph,edge_pairs,median_edge_correlation,median_nonedge_correlation
0,A_adj,665,0.186,0.091
1,A_dist,1228,0.164,0.090
2,A_od,2320,0.167,0.089


,bin,hours,median,q10,q90,variable
0,"(-16.701, -1.5]",744,-0.270,-0.489,0.121,temperature_2m
1,"(-1.5, 3.4]",743,-0.138,-0.449,0.092,temperature_2m
2,"(3.4, 8.3]",735,-0.119,-0.275,0.059,temperature_2m
3,"(8.3, 13.45]",729,-0.108,-0.199,0.071,temperature_2m
4,"(13.45, 18.6]",759,-0.087,-0.190,0.025,temperature_2m
5,"(18.6, 22.8]",724,-0.067,-0.174,0.068,temperature_2m
6,"(22.8, 26.8]",753,-0.047,-0.144,0.086,temperature_2m
7,"(26.8, 40.4]",716,-0.022,-0.113,0.137,temperature_2m
8,"(-0.001, 0.1]",5401,-0.087,-0.325,0.072,precipitation
9,"(0.1, 13.1]",502,-0.035,-0.238,0.164,precipitation


At +24h the same-hour-yesterday lag is the forecast-origin count, so it equals persistence.
Seasonal-profile errors above are in-sample diagnostics. Use the separate validation and test tables for accuracy claims.


In [ ]:
print("SIMPLE BASELINE MODELS")
print("=" * 55)

how = (idx.dayofweek * 24 + idx.hour).to_numpy()   # hour-of-week, 0..167
zone_list = final_demand_matrix.columns.tolist()

# Historical average table from TRAINING data only: [168 hour-of-week, N zones]
ha_table = (final_demand_matrix.where(~eval_exclude).iloc[train_mask].groupby(how[train_mask]).mean().fillna(0).to_numpy())

def within_month_weekly_average(orig, h):
    target = orig + h
    week_lags = target[:, None] - 168 * np.arange(1, 5)
    safe_lags = np.maximum(week_lags, 0)
    use = (week_lags >= 0) & (months_by_hour[safe_lags] == months_by_hour[orig, None])
    return (counts[safe_lags] * use[..., None]).sum(1) / use.sum(1, keepdims=True)

months_by_hour = idx.to_period("M").asi8

def baseline_predictions(orig):
    return {
        "Persistence (last hour)":
            {h: counts[orig] for h in EVAL_H},
        "Seasonal naive (same hour yesterday)":
            {h: counts[orig + h - 24] for h in EVAL_H},
        "Seasonal naive (same hour last week)":
            {h: counts[orig + h - 168] for h in EVAL_H},
        "Historical average (hour-of-week)":
            {h: ha_table[how[orig + h]] for h in EVAL_H},
        "Weekly moving average (up to 4 same-month weeks)":
            {h: within_month_weekly_average(orig, h)
             for h in EVAL_H},
    }

for split, orig in [("val", val_o), ("test", test_o)]:
    for name, preds in baseline_predictions(orig).items():
        evaluate(preds, orig, name, split=split)
print("Baselines evaluated on validation and test sets.")

# ---------- Results ----------
print("\nTEST SET — ALL ZONES")
display(show("test", "all"))
print("\nTEST SET — ACTIVE ZONES")
display(show("test", "active"))
print("\nTEST SET — EVENT CELLS")
display(show("test", "events"))
print("\nVALIDATION SET — ALL ZONES")
display(show("val", "all"))

# ---------- Demand level check (explains historical-average bias) ----------
city_train = counts[train_mask].sum(1).mean()
city_test = counts[test_mask].sum(1).mean()
print(f"\nMean citywide pickups/hour — train: {city_train:,.0f} | test: {city_test:,.0f} "
      f"({city_test / city_train - 1:+.1%})")

# ---------- Error growth with horizon ----------
r = results[(results["split"] == "test") & (results["subset"] == "all")]
r.pivot_table(index="horizon", columns="model", values="MAE").plot(
    marker="o", figsize=(10, 4), title="Baseline MAE by Forecast Horizon (Test, All Zones)")
plt.xlabel("Forecast horizon (hours)"); plt.ylabel("MAE (pickups)"); plt.xticks(EVAL_H)
plt.legend(fontsize=8); plt.show()

# ---------- Visual check: first test week, 1-hour-ahead forecasts ----------
preds_test = baseline_predictions(test_o)
week = test_o[:168]
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, z in zip(axes, [161, 132]):
    j = zone_list.index(z)
    ax.plot(idx[week + 1], counts[week + 1, j], color="black", lw=1.5, label="Actual")
    for name in ["Persistence (last hour)", "Historical average (hour-of-week)",
                 "Weekly moving average (up to 4 same-month weeks)"]:
        ax.plot(idx[week + 1], preds_test[name][1][:168, j], lw=1, alpha=0.8, label=name)
    ax.set_title(f"{zone_info.loc[z, 'Zone']} — 1-hour-ahead baseline forecasts (first test week)")
    ax.set_ylabel("Pickups")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

print("\nBASELINE EVALUATION COMPLETED!")

SIMPLE BASELINE MODELS
Baselines evaluated on validation and test sets.

TEST SET — ALL ZONES


MAE                  \
horizon                                              1       3       6    
model                                                                     
Historical average (hour-of-week)                 4.287   4.271   4.264   
Weekly moving average (up to 4 same-month weeks)  4.625   4.604   4.591   
Seasonal naive (same hour last week)              4.990   4.970   4.958   
Persistence (last hour)                           5.903  11.682  18.432   
Seasonal naive (same hour yesterday)              7.023   6.940   6.887   

                                                                   RMSE  \
horizon                                               12     24      1    
model                                                                     
Historical average (hour-of-week)                  4.266  4.270  13.173   
Weekly moving average (up to 4 same-month weeks)   4.555  4.462  14.697   
Seasonal naive (same hour last week)               4.941  4.907  15.715   
Persistence (last hour)                           23.425  6.822  17.709   
Seasonal naive (same hour yesterday)               6.857  6.822  25.148   

                                                                          \
horizon                                               3       6       12   
model                                                                      
Historical average (hour-of-week)                 13.115  13.096  13.102   
Weekly moving average (up to 4 same-month weeks)  14.658  14.641  14.471   
Seasonal naive (same hour last week)              15.678  15.662  15.544   
Persistence (last hour)                           36.710  55.267  67.311   
Seasonal naive (same hour yesterday)              24.671  24.442  24.367   

                                                          WMAPE                \
horizon                                               24     1      3      6    
model                                                                           
Historical average (hour-of-week)                 13.095  0.199  0.199  0.199   
Weekly moving average (up to 4 same-month weeks)  13.966  0.215  0.214  0.214   
Seasonal naive (same hour last week)              15.444  0.231  0.231  0.231   
Persistence (last hour)                           24.238  0.274  0.543  0.858   
Seasonal naive (same hour yesterday)              24.238  0.326  0.323  0.321   

                                                                
horizon                                              12     24  
model                                                           
Historical average (hour-of-week)                 0.198  0.197  
Weekly moving average (up to 4 same-month weeks)  0.211  0.206  
Seasonal naive (same hour last week)              0.229  0.227  
Persistence (last hour)                           1.087  0.315  
Seasonal naive (same hour yesterday)              0.318  0.315


TEST SET — ACTIVE ZONES


MAE                  \
horizon                                               1       3       6    
model                                                                      
Historical average (hour-of-week)                  6.943   6.917   6.905   
Weekly moving average (up to 4 same-month weeks)   7.499   7.465   7.443   
Seasonal naive (same hour last week)               8.095   8.063   8.042   
Persistence (last hour)                            9.631  19.342  30.704   
Seasonal naive (same hour yesterday)              11.513  11.373  11.284   

                                                                    RMSE  \
horizon                                               12      24      1    
model                                                                      
Historical average (hour-of-week)                  6.908   6.915  17.093   
Weekly moving average (up to 4 same-month weeks)   7.382   7.226  19.069   
Seasonal naive (same hour last week)               8.012   7.954  20.387   
Persistence (last hour)                           39.105  11.174  22.980   
Seasonal naive (same hour yesterday)              11.233  11.174  32.642   

                                                                          \
horizon                                               3       6       12   
model                                                                      
Historical average (hour-of-week)                 17.017  16.994  17.000   
Weekly moving average (up to 4 same-month weeks)  19.018  18.995  18.775   
Seasonal naive (same hour last week)              20.340  20.319  20.165   
Persistence (last hour)                           47.658  71.756  87.396   
Seasonal naive (same hour yesterday)              32.022  31.725  31.629   

                                                          WMAPE                \
horizon                                               24     1      3      6    
model                                                                           
Historical average (hour-of-week)                 16.991  0.192  0.192  0.192   
Weekly moving average (up to 4 same-month weeks)  18.118  0.208  0.207  0.207   
Seasonal naive (same hour last week)              20.035  0.224  0.224  0.224   
Persistence (last hour)                           31.460  0.267  0.538  0.854   
Seasonal naive (same hour yesterday)              31.460  0.319  0.316  0.314   

                                                                
horizon                                              12     24  
model                                                           
Historical average (hour-of-week)                 0.191  0.191  
Weekly moving average (up to 4 same-month weeks)  0.205  0.199  
Seasonal naive (same hour last week)              0.222  0.219  
Persistence (last hour)                           1.084  0.308  
Seasonal naive (same hour yesterday)              0.311  0.308


TEST SET — EVENT CELLS


MAE                    \
horizon                                                1        3        6    
model                                                                         
Persistence (last hour)                            33.676   45.595   37.405   
Seasonal naive (same hour yesterday)               57.865   57.865   57.865   
Historical average (hour-of-week)                 101.785  101.785  101.785   
Weekly moving average (up to 4 same-month weeks)  105.878  105.878  105.878   
Seasonal naive (same hour last week)              108.081  108.081  108.081   

                                                                       RMSE  \
horizon                                                12       24       1    
model                                                                         
Persistence (last hour)                            51.162   55.667   47.128   
Seasonal naive (same hour yesterday)               57.865   55.667   98.500   
Historical average (hour-of-week)                 101.785  102.080  171.474   
Weekly moving average (up to 4 same-month weeks)  105.878  107.042  182.025   
Seasonal naive (same hour last week)              108.081  109.306  190.498   

                                                                             \
horizon                                                3        6        12   
model                                                                         
Persistence (last hour)                            69.336   48.677   65.027   
Seasonal naive (same hour yesterday)               98.500   98.500   98.500   
Historical average (hour-of-week)                 171.474  171.474  171.474   
Weekly moving average (up to 4 same-month weeks)  182.025  182.025  182.025   
Seasonal naive (same hour last week)              190.498  190.498  190.498   

                                                           WMAPE         \
horizon                                                24     1      3    
model                                                                     
Persistence (last hour)                            97.213  0.593  0.803   
Seasonal naive (same hour yesterday)               97.213  1.020  1.020   
Historical average (hour-of-week)                 173.174  1.793  1.793   
Weekly moving average (up to 4 same-month weeks)  184.227  1.865  1.865   
Seasonal naive (same hour last week)              192.831  1.904  1.904   

                                                                       
horizon                                              6      12     24  
model                                                                  
Persistence (last hour)                           0.659  0.901  0.964  
Seasonal naive (same hour yesterday)              1.020  1.020  0.964  
Historical average (hour-of-week)                 1.793  1.793  1.768  
Weekly moving average (up to 4 same-month weeks)  1.865  1.865  1.854  
Seasonal naive (same hour last week)              1.904  1.904  1.894

... (2 outputs left out)


Mean citywide pickups/hour — train: 5,431 | test: 5,612 (+3.3%)


<Figure size 1000x400 with 1 Axes>

<Figure size 1400x700 with 2 Axes>


BASELINE EVALUATION COMPLETED!


In [ ]:
# LightGBM setup for Colab T4. PyTorch keeps the GPU even if LightGBM uses CPU.
import importlib.util
import os
import shutil
import subprocess
import sys

LIGHTGBM_TRY_CUDA = True
LIGHTGBM_HAS_CUDA_BUILD = False

def prepare_lightgbm():
    global LIGHTGBM_HAS_CUDA_BUILD
    if LIGHTGBM_HAS_CUDA_BUILD:
        return
    if LIGHTGBM_TRY_CUDA and shutil.which("nvcc") and torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability()
        subprocess.run([sys.executable, "-m", "pip", "install", "narwhals>=1.15"], check=True)
        cmd = [
            sys.executable, "-m", "pip", "install", "--force-reinstall", "--no-deps",
            "--no-binary=lightgbm",
            "--config-settings=cmake.define.USE_CUDA=ON",
            f"--config-settings=cmake.define.CMAKE_CUDA_ARCHITECTURES={major}{minor}",
            "lightgbm==4.7.0",
        ]
        print("Building LightGBM for the assigned GPU with two compiler jobs.", flush=True)
        build_env = dict(os.environ, CMAKE_BUILD_PARALLEL_LEVEL="2")
        built = subprocess.run(cmd, env=build_env, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True)
        LIGHTGBM_HAS_CUDA_BUILD = built.returncode == 0
        if not LIGHTGBM_HAS_CUDA_BUILD:
            print(built.stdout[-6000:])
            print("CUDA build failed. The LightGBM benchmark will use CPU.")
    if not LIGHTGBM_HAS_CUDA_BUILD and importlib.util.find_spec("lightgbm") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "lightgbm"], check=True)

prepare_lightgbm()


Building LightGBM for the assigned GPU with two compiler jobs.


In [ ]:
# LIGHTGBM BENCHMARK — ISOLATED CUDA TRAINING WITH CPU RECOVERY

import gc
import os
import json
import subprocess
from pathlib import Path
from tempfile import TemporaryDirectory
import sys
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- Robust Recovery of Parameters & Data ----------
SAVE = globals().get('SAVE', '/content/drive/MyDrive/cs668/random_months_five_horizons_seed42/')
os.makedirs(SAVE, exist_ok=True)

# Use the original preprocessing outputs and evaluation helpers in this session.
# After a runtime reset, rerun preparation rather than reconstructing a different split.
LIGHTGBM_HORIZONS = [1, 3, 6, 12, 24]
required = ("context", "counts", "L", "train_mask", "eval_exclude", "train_o",
            "val_o", "test_o", "full_o", "EVAL_H", "results", "idx", "N",
            "combine_predictions", "evaluate", "shade_month_splits", "split_names")
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError("Run data preparation first. Missing: " + ", ".join(missing))
if list(EVAL_H) != LIGHTGBM_HORIZONS:
    raise ValueError("Expected the prepared 1, 3, 6, 12, and 24-hour horizons.")
full_preds = globals().get("full_preds", {})

# ---------- Graph Setup & Helper functions ----------
def row_norm(A):
    totals = A.sum(axis=1, keepdims=True)
    return np.divide(A, totals, out=np.zeros_like(A, dtype=np.float32), where=totals > 0)

with np.load(SAVE + "graphs.npz") as graphs:
    W_adj = row_norm(graphs["A_adj"])
    W_od = row_norm(graphs["A_od"])

cal_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend",
            "is_holiday", "pre_holiday", "post_holiday", "dst_flag"]
wx_cols = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "is_raining"]
CAL = context[cal_cols].to_numpy(np.float32)
WX = context[wx_cols].to_numpy(np.float32)
zone_mean = L[train_mask].mean(axis=0)
zone_ids_arr = np.arange(N, dtype=np.int32)
cs = np.vstack([np.zeros((1, N), dtype=np.float32), np.cumsum(L, axis=0)])

def roll(origins_idx, hours):
    return (cs[origins_idx + 1] - cs[origins_idx + 1 - hours]) / hours

def make_features(origins_idx, horizon, spatial):
    target = origins_idx + horizon
    shape = (len(origins_idx), N)
    latest = L[origins_idx]
    recent_mean = roll(origins_idx, 3)
    weekly_lag = L[target - 168]

    features = {
        "lag_0": latest,
        "lag_1": L[origins_idx - 1],
        "lag_2": L[origins_idx - 2],
        "lag_3": L[origins_idx - 3],
        "lag_tgt_24": L[target - 24],
        "lag_tgt_168": weekly_lag,
        "roll_3": recent_mean,
        "roll_24": roll(origins_idx, 24),
        "roll_168": roll(origins_idx, 168),
        "zone_mean": np.broadcast_to(zone_mean, shape),
        "zone_id": np.broadcast_to(zone_ids_arr, shape),
    }
    for col_idx, name in enumerate(cal_cols):
        features[name] = np.broadcast_to(CAL[target, col_idx, None], shape)
    for col_idx, name in enumerate(wx_cols):
        features[name] = np.broadcast_to(WX[origins_idx, col_idx, None], shape)

    if spatial:
        for name, weights in [("adj", W_adj), ("od", W_od)]:
            features[f"nbr_{name}_lag_0"] = latest @ weights.T
            features[f"nbr_{name}_roll_3"] = recent_mean @ weights.T
            features[f"nbr_{name}_lag_tgt_168"] = weekly_lag @ weights.T

    X = pd.DataFrame({
        name: np.ascontiguousarray(values, dtype=np.int32 if name == "zone_id" else np.float32).ravel()
        for name, values in features.items()
    })
    return X, counts[target].ravel()

# ---------- Training hyperparameters shared by both variants ----------
params = dict(
    objective="poisson", metric="l1", learning_rate=0.1, max_bin=63,
    num_leaves=63, min_data_in_leaf=200, feature_fraction=0.8,
    bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
    seed=42, verbosity=-1, num_threads=min(4, os.cpu_count() or 1),
)
variants = {
    "LightGBM (no spatial features)": False,
    "LightGBM + spatial lags": True,
}

# Train and predict in a separate process. A native CUDA abort kills only the worker.
LIGHTGBM_WORKER = r"""
import json
import resource
import sys
from pathlib import Path
import lightgbm as lgb
import numpy as np

resource.setrlimit(resource.RLIMIT_CORE, (0, 0))
folder = Path(sys.argv[1])
config = json.loads((folder / "fit.json").read_text())
params = config["params"]
columns = config["columns"]
zone = columns.index("zone_id")
train = np.load(folder / "train.npy", mmap_mode="r")
val = np.load(folder / "val.npy", mmap_mode="r")
y_train = np.load(folder / "train_y.npy")
y_val = np.load(folder / "val_y.npy")
m_train = np.load(folder / "train_mask.npy")
m_val = np.load(folder / "val_mask.npy")
dtrain = lgb.Dataset(train[m_train], label=y_train[m_train],
                     feature_name=columns, categorical_feature=[zone])
dval = lgb.Dataset(val[m_val], label=y_val[m_val], reference=dtrain)
model = lgb.train(
    params, dtrain, num_boost_round=500, valid_sets=[dval], valid_names=["val"],
    callbacks=[lgb.early_stopping(25, verbose=False), lgb.log_evaluation(100)],
)
best = model.best_iteration or model.current_iteration()
gain = model.feature_importance("gain", iteration=best)
if not np.any(gain > 0):
    raise RuntimeError("Fit learned no splits. Check feature and target variation.")
model.save_model(str(folder / "model.txt"), num_iteration=best)
np.save(folder / "gain.npy", gain)
for split in ("train", "val", "test"):
    X = np.load(folder / (split + ".npy"), mmap_mode="r")
    pred = model.predict(X, num_iteration=best).astype(np.float32)
    if not np.isfinite(pred).all():
        raise RuntimeError("Model produced nonfinite predictions.")
    np.save(folder / (split + "_pred.npy"), pred)
(folder / "fit_result.json").write_text(json.dumps({
    "iteration": best, "val_mae": model.best_score["val"]["l1"],
}))
"""

def run_lightgbm(backend, staging):
    fit_params = dict(params, device_type=backend)
    if backend == "cuda":
        fit_params.update(gpu_device_id=0, is_enable_sparse=False)
    predictions, importance = {}, {}
    origins_by_split = {"train": train_o, "val": val_o, "test": test_o}
    worker = staging / "train_worker.py"
    worker.write_text(LIGHTGBM_WORKER, encoding="utf-8")

    for name, spatial in variants.items():
        tag = "spatial" if spatial else "base"
        predictions[name] = {split: {} for split in origins_by_split}
        for h in LIGHTGBM_HORIZONS:
            started = time.perf_counter()
            print(f"{backend.upper()} | {name} | +{h}h", flush=True)
            with TemporaryDirectory(dir=staging) as fit_path:
                folder = Path(fit_path)
                for split, orig in origins_by_split.items():
                    features, targets = make_features(orig, h, spatial)
                    columns = features.columns.tolist()
                    np.save(folder / f"{split}.npy", features.to_numpy(dtype=np.float32))
                    if split != "test":
                        valid = ~eval_exclude[orig + h].ravel()
                        if not valid.any():
                            raise ValueError(f"No valid {split} targets for +{h}h.")
                        np.save(folder / f"{split}_y.npy", targets)
                        np.save(folder / f"{split}_mask.npy", valid)
                    del features, targets
                gc.collect()

                (folder / "fit.json").write_text(json.dumps({
                    "params": fit_params, "columns": columns,
                }))
                proc = subprocess.run(
                    [sys.executable, str(worker), str(folder)],
                    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                )
                if proc.returncode:
                    raise RuntimeError(
                        f"{backend.upper()} worker failed for {name}, +{h}h "
                        f"(exit {proc.returncode}).\n{proc.stdout[-5000:]}"
                    )
                summary = json.loads((folder / "fit_result.json").read_text())
                (folder / "model.txt").replace(staging / f"lgbm_{tag}_h{h}.txt")
                importance[(name, h)] = pd.Series(np.load(folder / "gain.npy"), index=columns)
                for split, orig in origins_by_split.items():
                    pred = np.load(folder / f"{split}_pred.npy")
                    predictions[name][split][h] = pred.reshape(len(orig), N)

            print(f"  {summary['iteration']} trees | validation MAE {summary['val_mae']:.4f} | "
                  f"{time.perf_counter() - started:.1f}s", flush=True)
    return predictions, importance

preferred = "cuda" if globals().get("LIGHTGBM_HAS_CUDA_BUILD", False) else "cpu"
with TemporaryDirectory(prefix="urbanflow-lightgbm-") as temporary:
    root = Path(temporary)
    staging = root / preferred
    staging.mkdir()
    backend_used = preferred
    try:
        lgb_predictions, importances = run_lightgbm(preferred, staging)
    except RuntimeError as error:
        if preferred != "cuda":
            raise
        print(error, flush=True)
        print("Retrying both variants and all five horizons on CPU for a consistent benchmark.",
              flush=True)
        # End this except block before retrying so the failed traceback releases its arrays.
        backend_used = "cpu"
    if backend_used != preferred:
        staging = root / "cpu"
        staging.mkdir()
        lgb_predictions, importances = run_lightgbm("cpu", staging)

    # Publish only after every fit has succeeded. Keep existing downstream filenames.
    from shutil import copyfile
    for model_path in staging.glob("lgbm_*.txt"):
        copyfile(model_path, Path(SAVE) / model_path.name)

for name, preds in lgb_predictions.items():
    preds["full"] = combine_predictions(preds)
    for split, orig in (("train", train_o), ("val", val_o), ("test", test_o), ("full", full_o)):
        evaluate(preds[split], orig, name, split=split)
    full_preds[name] = preds["full"]
    tag = "spatial" if variants[name] else "base"
    np.savez(SAVE + f"preds_lgbm_{tag}_full.npz", **{f"h{h}": preds["full"][h] for h in EVAL_H})

del lgb_predictions
print(f"Completed LightGBM benchmark on {backend_used.upper()}.")
(Path(SAVE) / "lightgbm_backend.txt").write_text(backend_used + "\n", encoding="utf-8")

# ---------- Validation/Test Diagnostic Evaluation Summary ----------
split_order = ["train", "val", "test", "full"]
split_labels = {"train": f"Train ({split_names['train']}, in-sample)",
                "val": f"Validation ({split_names['val']})",
                "test": f"Test ({split_names['test']})", "full": "All eligible 2025 origins"}

def split_summary(models, subset="all", metric="MAE"):
    r = results[results["model"].isin(models) & (results["subset"] == subset)]
    t = r.pivot_table(index=["model", "split"], columns="horizon", values=metric)
    t = t.reindex(index=[(m, s) for m in models for s in split_order], columns=LIGHTGBM_HORIZONS)
    t = t.rename(index=split_labels, level=1)
    return t.round(3)

lgb_models = list(variants.keys())
for subset in ["all", "active", "events"]:
    print(f"\nMAE BY SPLIT — {subset.upper()} ZONES" if subset != "events" else "\nMAE BY SPLIT — EVENT CELLS")
    display(split_summary(lgb_models, subset, "MAE"))
print("\nWMAPE BY SPLIT — ALL ZONES")
display(split_summary(lgb_models, "all", "WMAPE"))

# Overfitting check
r = results[results["subset"] == "all"].set_index(["model", "split", "horizon"])["MAE"]
print("\nGeneralization gap (test MAE / train MAE):")
for m in lgb_models:
    print(f"  {m}: " + " | ".join(f"h={h}: {r[(m, 'test', h)] / r[(m, 'train', h)]:.2f}x" for h in EVAL_H))

# Spatial lag effect analysis
print("\nEffect of spatial lags on MAE (negative = improvement):")
for s in split_order:
    print(f"  {split_labels[s]}: " + " | ".join(
        f"h={h}: {(r[('LightGBM + spatial lags', s, h)] / r[('LightGBM (no spatial features)', s, h)] - 1) * 100:+.1f}%"
        for h in EVAL_H))

# ---------- Diagnostic Visualization Plots ----------
for h in LIGHTGBM_HORIZONS:
    target = full_o + h
    dates = idx[target]
    observed = counts[target]
    valid = ~eval_exclude[target]
    valid_zones = valid.sum(axis=1)
    monthly = {}

    fig, ax = plt.subplots(figsize=(14, 4))
    actual = pd.Series(observed.sum(axis=1), index=dates).resample("D").sum(min_count=24)
    ax.plot(actual, color="black", linewidth=1.5, label="Actual")
    for name in lgb_models:
        predicted = full_preds[name][h]
        daily = pd.Series(predicted.sum(axis=1), index=dates).resample("D").sum(min_count=24)
        ax.plot(daily, linewidth=1, alpha=0.8, label=name)

        error = np.abs(predicted - observed)
        hourly_mae = np.divide(
            np.where(valid, error, 0).sum(axis=1), valid_zones,
            out=np.full(len(target), np.nan), where=valid_zones > 0,
        )
        monthly[name] = pd.Series(hourly_mae, index=dates).resample("ME").mean()

    shade_month_splits(ax)
    ax.set(title=f"Daily Citywide Pickups +{h}h", ylabel="Pickups per day")
    ax.legend(fontsize=8, ncol=3)
    fig.tight_layout()
    plt.show()

    monthly_table = pd.DataFrame(monthly)
    ax = monthly_table.plot(kind="bar", figsize=(12, 4))
    ax.set_xticklabels(monthly_table.index.strftime("%b"), rotation=0)
    ax.set(title=f"Monthly MAE +{h}h", ylabel="MAE (pickups)")
    plt.tight_layout()
    plt.show()

    gain = importances[("LightGBM + spatial lags", h)]
    if gain.sum() > 0:
        top = (gain / gain.sum()).nlargest(15).sort_values()
        ax = top.plot(kind="barh", figsize=(9, 6))
        ax.set(title=f"Spatial LightGBM Feature Importance +{h}h", xlabel="Share of total gain")
        plt.tight_layout()
        plt.show()

print("\nLIGHTGBM EVALUATION COMPLETED FOR +1h, +3h, +6h, +12h, AND +24h")


CUDA | LightGBM (no spatial features) | +1h
CUDA worker failed for LightGBM (no spatial features), +1h (exit 1).
/tmp/urbanflow-lightgbm-5j85xfo1/cuda/train_worker.py:36: RuntimeWarning: overflow encountered in cast
  pred = model.predict(X, num_iteration=best).astype(np.float32)
Traceback (most recent call last):
  File "/tmp/urbanflow-lightgbm-5j85xfo1/cuda/train_worker.py", line 38, in <module>
    raise RuntimeError("Model produced nonfinite predictions.")
RuntimeError: Model produced nonfinite predictions.

Retrying both variants and all five horizons on CPU for a consistent benchmark.
CPU | LightGBM (no spatial features) | +1h
  500 trees | validation MAE 3.7246 | 38.8s
CPU | LightGBM (no spatial features) | +3h
  500 trees | validation MAE 4.2979 | 35.4s
CPU | LightGBM (no spatial features) | +6h
  500 trees | validation MAE 4.5263 | 34.5s
CPU | LightGBM (no spatial features) | +12h
  500 trees | validation MAE 4.5765 | 34.7s
CPU | LightGBM (no spatial features) | +24h
  500 tre

horizon                                                                               1   \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.156   
                               Validation (May, Nov)                               3.725   
                               Test (Feb, Sep)                                     3.492   
                               All eligible 2025 origins                           3.304   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.103   
                               Validation (May, Nov)                               3.625   
                               Test (Feb, Sep)                                     3.412   
                               All eligible 2025 origins                           3.239   

horizon                                                                               3   \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.402   
                               Validation (May, Nov)                               4.298   
                               Test (Feb, Sep)                                     3.851   
                               All eligible 2025 origins                           3.622   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.405   
                               Validation (May, Nov)                               4.295   
                               Test (Feb, Sep)                                     3.877   
                               All eligible 2025 origins                           3.628   

horizon                                                                               6   \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.463   
                               Validation (May, Nov)                               4.526   
                               Test (Feb, Sep)                                     3.992   
                               All eligible 2025 origins                           3.724   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.461   
                               Validation (May, Nov)                               4.513   
                               Test (Feb, Sep)                                     3.986   
                               All eligible 2025 origins                           3.719   

horizon                                                                               12  \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.498   
                               Validation (May, Nov)                               4.577   
                               Test (Feb, Sep)                                     4.029   
                               All eligible 2025 origins                           3.762   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.483   
                               Validation (May, Nov)                               4.573   
                               Test (Feb, Sep)                                     4.036   
                               All eligible 2025 origins                           3.752   

horizon                                                                               24  
model                          split                                                      
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.498  
                               Validation (


MAE BY SPLIT — ACTIVE ZONES


horizon                                                                               1   \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.038   
                               Validation (May, Nov)                               5.967   
                               Test (Feb, Sep)                                     5.611   
                               All eligible 2025 origins                           5.283   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  4.950   
                               Validation (May, Nov)                               5.800   
                               Test (Feb, Sep)                                     5.477   
                               All eligible 2025 origins                           5.174   

horizon                                                                               3   \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.450   
                               Validation (May, Nov)                               6.928   
                               Test (Feb, Sep)                                     6.213   
                               All eligible 2025 origins                           5.816   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.455   
                               Validation (May, Nov)                               6.924   
                               Test (Feb, Sep)                                     6.259   
                               All eligible 2025 origins                           5.826   

horizon                                                                               6   \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.552   
                               Validation (May, Nov)                               7.311   
                               Test (Feb, Sep)                                     6.451   
                               All eligible 2025 origins                           5.987   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.548   
                               Validation (May, Nov)                               7.288   
                               Test (Feb, Sep)                                     6.440   
                               All eligible 2025 origins                           5.978   

horizon                                                                               12  \
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.611   
                               Validation (May, Nov)                               7.396   
                               Test (Feb, Sep)                                     6.511   
                               All eligible 2025 origins                           6.050   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.584   
                               Validation (May, Nov)                               7.390   
                               Test (Feb, Sep)                                     6.524   
                               All eligible 2025 origins                           6.033   

horizon                                                                               24  
model                          split                                                      
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.608  
                               Validation (


MAE BY SPLIT — EVENT CELLS


horizon                                                                                1   \
model                          split                                                        
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  13.032   
                               Validation (May, Nov)                               16.443   
                               Test (Feb, Sep)                                     42.114   
                               All eligible 2025 origins                           15.323   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  12.417   
                               Validation (May, Nov)                               16.614   
                               Test (Feb, Sep)                                     41.979   
                               All eligible 2025 origins                           14.905   

horizon                                                                                3   \
model                          split                                                        
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  19.536   
                               Validation (May, Nov)                               33.062   
                               Test (Feb, Sep)                                     73.238   
                               All eligible 2025 origins                           25.339   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  19.595   
                               Validation (May, Nov)                               34.073   
                               Test (Feb, Sep)                                     68.551   
                               All eligible 2025 origins                           25.356   

horizon                                                                                6   \
model                          split                                                        
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  20.908   
                               Validation (May, Nov)                               44.895   
                               Test (Feb, Sep)                                     78.007   
                               All eligible 2025 origins                           29.203   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  20.507   
                               Validation (May, Nov)                               39.046   
                               Test (Feb, Sep)                                     78.365   
                               All eligible 2025 origins                           27.644   

horizon                                                                                12  \
model                          split                                                        
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  23.246   
                               Validation (May, Nov)                               55.367   
                               Test (Feb, Sep)                                     79.399   
                               All eligible 2025 origins                           33.139   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  22.272   
                               Validation (May, Nov)                               53.918   
                               Test (Feb, Sep)                                     83.114   
                               All eligible 2025 origins                           32.307   

horizon                                                                                24  
model                          split                                                       
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  23.937  


... (15 outputs left out)

<Figure size 1400x400 with 1 Axes>

<Figure size 1200x400 with 1 Axes>

<Figure size 900x600 with 1 Axes>


LIGHTGBM EVALUATION COMPLETED FOR +1h, +3h, +6h, +12h, AND +24h


In [ ]:
import re
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display

HORIZONS = (1, 3, 6, 12, 24)
if H < max(HORIZONS) or tuple(EVAL_H) != HORIZONS:
    raise ValueError(
        "Set H = 24 and EVAL_H = [1, 3, 6, 12, 24] in the preparation cell, "
        "then rerun preparation before running this cell."
    )

print("STEP 23: DEEP LEARNING BASELINE — GRU WITHOUT GRAPH")
print("=" * 55)

print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
torch.manual_seed(42); np.random.seed(42)

# ---------- Tensors on GPU ----------
cal_cols = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "is_weekend",
            "is_holiday", "pre_holiday", "post_holiday", "dst_flag"]
wx_cols = ["temperature_2m", "precipitation", "snowfall", "wind_speed_10m", "is_raining"]
ctx_cols = cal_cols + wx_cols

Ls_t = torch.tensor(L_scaled, dtype=torch.float32, device=device)            # [T, N]
C_t = torch.tensor(context[ctx_cols].to_numpy(np.float32), device=device)   # [T, F]
counts_t = torch.tensor(counts, device=device)
EX_t = torch.tensor(eval_exclude, device=device)                            # [T, N] bool
mu_t = torch.tensor(mu, dtype=torch.float32, device=device)
sd_t = torch.tensor(sd, dtype=torch.float32, device=device)
tgt_off = torch.arange(1, H + 1, device=device)
N_CTX = len(ctx_cols)
zone_loss_weight = torch.tensor(np.sqrt(1 + counts[train_mask].mean(0)),
                                dtype=torch.float32, device=device)
zone_loss_weight /= zone_loss_weight.mean()

full_preds = globals().get("full_preds", {})            # keeps LightGBM predictions if present

def window_offsets(model):
    history = getattr(model, "history_hours", T_IN)
    return torch.arange(1 - history, 1, device=device)

def get_batch(orig, offsets):
    """orig: LongTensor of forecast origins -> inputs, targets, valid-target mask."""
    win = orig[:, None] + offsets            # [B, T_IN]
    tgt = orig[:, None] + tgt_off            # [B, H]
    return Ls_t[win], C_t[win], Ls_t[tgt], ~EX_t[tgt]

# ---------- Shared training / prediction utilities  ----------
eval_steps_t = torch.tensor([h - 1 for h in EVAL_H], dtype=torch.long, device=device)

def neural_forward(model, demand, features):
    with torch.autocast(device_type=device.type, dtype=amp_dtype, enabled=amp_enabled):
        return model(demand, features)

@torch.inference_mode()
def eval_loss(model, orig_t, batch_size=64):
    model.eval()
    num = torch.zeros(len(EVAL_H), dtype=torch.float64, device=device)
    den = torch.zeros(len(EVAL_H), dtype=torch.int64, device=device)
    offsets = window_offsets(model)
    if len(offsets) > T_IN:
        batch_size = min(batch_size, 8)
    for origins in orig_t.split(batch_size):
        xd, xc, _, valid = get_batch(origins, offsets)
        selected = neural_forward(model, xd, xc).index_select(1, eval_steps_t).float()
        pred = torch.expm1(selected * sd_t + mu_t).clamp_min(0)
        actual = counts_t[origins[:, None] + tgt_off[eval_steps_t]]
        mask = valid.index_select(1, eval_steps_t)
        num += (pred - actual).abs().masked_fill(~mask, 0).sum(dim=(0, 2))
        den += mask.sum(dim=(0, 2))
    return (num / den).mean().item()

def train_model(model, name, epochs=60, batch_size=32, lr=1e-3, patience=6):
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4,
                            fused=cuda_training)
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled and amp_dtype == torch.float16)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    offsets = window_offsets(model)
    if len(offsets) > T_IN:
        batch_size = min(batch_size, 8)
    tr = torch.as_tensor(train_o, device=device)
    va = torch.as_tensor(val_o, device=device)
    best, best_state, wait, hist = np.inf, None, 0, []
    robust_loss = getattr(model, "robust_loss", False)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nTraining {name} ({n_params:,} parameters)")

    for ep in range(1, epochs + 1):
        model.train()
        started = time.perf_counter()
        loss_sum = torch.zeros((), dtype=torch.float64, device=device)
        used_batches = torch.zeros((), dtype=torch.int64, device=device)
        shuffled = tr[torch.randperm(len(tr), device=device)]
        for origins in shuffled.split(batch_size):
            xd, xc, y, valid = get_batch(origins, offsets)
            opt.zero_grad(set_to_none=True)
            pred = neural_forward(model, xd, xc).float()
            if robust_loss:
                error = nn.functional.smooth_l1_loss(
                    pred * sd_t, y * sd_t, beta=0.15, reduction="none"
                ) * zone_loss_weight
            else:
                error = (pred - y).abs()
            valid_count = valid.sum()
            loss = error.masked_fill(~valid, 0).sum() / valid_count.clamp_min(1)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            loss_sum += loss.detach()
            used_batches += valid_count > 0

        train_loss = (loss_sum / used_batches.clamp_min(1)).item()
        val_loss = eval_loss(model, va)
        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            raise FloatingPointError(
                f"{name}: nonfinite loss. Try USE_MIXED_PRECISION = False and rerun."
            )
        sched.step(val_loss)
        hist.append((ep, train_loss, val_loss))
        print(f"  epoch {ep:2d} | train {train_loss:.4f} | val {val_loss:.4f} | "
              f"lr {opt.param_groups[0]['lr']:.1e} | {time.perf_counter() - started:.1f}s")
        if val_loss < best - 1e-4:
            best, wait = val_loss, 0
            best_state = {key: tensor.detach().to("cpu", copy=True)
                          for key, tensor in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"  early stopping at epoch {ep} (best val {best:.4f})")
                break

    model.load_state_dict(best_state)
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    torch.save(best_state, SAVE + f"model_{slug}.pt")
    return model, pd.DataFrame(hist, columns=["epoch", "train_loss", "val_loss"])

@torch.inference_mode()
def predict_counts(model, orig, batch_size=64):
    model.eval()
    offsets = window_offsets(model)
    if len(offsets) > T_IN:
        batch_size = min(batch_size, 8)
    destinations = {h: np.empty((len(orig), N), dtype=np.float32) for h in EVAL_H}
    origins_t = torch.as_tensor(orig, device=device)
    for start in range(0, len(orig), batch_size):
        xd, xc, _, _ = get_batch(origins_t[start:start + batch_size], offsets)
        selected = neural_forward(model, xd, xc).index_select(1, eval_steps_t).float()
        predictions = torch.expm1(selected * sd_t + mu_t).clamp_min(0).cpu().numpy()
        for j, horizon in enumerate(EVAL_H):
            destinations[horizon][start:start + len(predictions)] = predictions[:, j]
    return destinations

def fit_and_evaluate(model, name, **kw):
    """Fit on training months and select the checkpoint on validation months."""
    model, hist = train_model(model, name, **kw)
    axes = hist.plot(x="epoch", y=["train_loss", "val_loss"], subplots=True,
                     marker="o", figsize=(8, 5), title=f"{name} — Training Curve")
    axes[0].set_ylabel("Training loss")
    axes[1].set_ylabel("Validation MAE (pickups)")
    plt.tight_layout()
    plt.show()

    preds = {s: predict_counts(model, o) for s, o in
             [("train", train_o), ("val", val_o), ("test", test_o)]}
    preds["full"] = combine_predictions(preds)
    for s, o in [("train", train_o), ("val", val_o), ("test", test_o), ("full", full_o)]:
        evaluate(preds[s], o, name, split=s)

    full_preds[name] = preds["full"]
    slug = re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")
    np.savez(SAVE + f"preds_{slug}_full.npz", **{f"h{h}": preds["full"][h] for h in EVAL_H})
    return model, hist

# ---------- Shared reporting utilities (reused in Step 24) ----------
split_order = ["train", "val", "test", "full"]
split_labels = {"train": f"Train ({split_names['train']}, in-sample)",
                "val": f"Validation ({split_names['val']})",
                "test": f"Test ({split_names['test']})", "full": "All eligible 2025 origins"}

def split_summary(models, subset="all", metric="MAE"):
    models = [m for m in models if m in set(results["model"])]
    r = results[results["model"].isin(models) & (results["subset"] == subset)]
    t = r.pivot_table(index=["model", "split"], columns="horizon", values=metric)
    t = t.reindex(index=[(m, s) for m in models for s in split_order], columns=HORIZONS)
    t = t.rename(index=split_labels, level=1)
    return t.round(3)

def generalization_gap(models):
    r = results[results["subset"] == "all"].set_index(["model", "split", "horizon"])["MAE"]
    print("\nGeneralization gap (test MAE / train MAE):")
    for m in models:
        available = [h for h in HORIZONS if (m, "train", h) in r.index and (m, "test", h) in r.index]
        if available:
            print(f"  {m}: " + " | ".join(
                f"h={h}: {r[(m, 'test', h)] / r[(m, 'train', h)]:.2f}x" for h in available))

def plot_full_year(models, h=1):
    models = [m for m in models if h in full_preds.get(m, {})]
    tgt = full_o + h
    actual = pd.Series(counts[tgt].sum(1), index=idx[tgt]).resample("D").sum(min_count=24)
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.plot(actual, color="black", lw=1.5, label="Actual")
    for m in models:
        ax.plot(pd.Series(full_preds[m][h].sum(1), index=idx[tgt]).resample("D").sum(min_count=24),
                lw=1, alpha=0.8, label=m)
    shade_month_splits(ax)
    ax.set_title(f"Daily Citywide Pickups: Actual vs {h}-Hour-Ahead Forecasts, Full Year 2025")
    ax.set_ylabel("Pickups per day"); ax.legend(fontsize=8, ncol=3); plt.show()

def plot_monthly_mae(models, h=1):
    models = [m for m in models if h in full_preds.get(m, {})]
    tgt = full_o + h
    valid = ~eval_exclude[tgt]
    monthly = {}
    valid_zones = valid.sum(axis=1)
    for m in models:
        err = np.abs(np.clip(full_preds[m][h], 0, None) - counts[tgt])
        hourly_mae = np.divide(
            np.where(valid, err, 0).sum(axis=1),
            valid_zones,
            out=np.full(len(tgt), np.nan),
            where=valid_zones > 0,
        )
        monthly[m] = pd.Series(hourly_mae, index=idx[tgt]).resample("ME").mean()
    mdf = pd.DataFrame(monthly)
    ax = mdf.plot(kind="bar", figsize=(12, 4),
                  title=f"Monthly MAE Across 2025 ({h}-Hour-Ahead, All Zones)")
    ax.set_xticks(range(len(mdf))); ax.set_xticklabels([d.strftime("%b") for d in mdf.index], rotation=0)
    ax.set_ylabel("MAE (pickups)"); ax.legend(fontsize=8); plt.show()
    return mdf.round(3)

# ---------- Model: one shared GRU applied to every zone independently ----------
class NodeGRU(nn.Module):
    def __init__(self, n_nodes, n_ctx, hidden=64, emb=16, layers=2, horizon=H, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, emb)
        self.gru = nn.GRU(1 + n_ctx + emb, hidden, layers, batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, horizon))

    def forward(self, x_dem, x_ctx):            # x_dem [B,T,N], x_ctx [B,T,F]
        B, T, N = x_dem.shape
        e = self.emb.weight[None, None].expand(B, T, N, -1)
        c = x_ctx[:, :, None, :].expand(B, T, N, -1)
        z = torch.cat([x_dem[..., None], c, e], dim=-1)          # [B,T,N,D]
        z = z.permute(0, 2, 1, 3).reshape(B * N, T, -1)          # each zone = one sequence
        out, _ = self.gru(z)
        y = self.head(out[:, -1])                                # [B*N, H]
        return y.view(B, N, -1).permute(0, 2, 1)                 # [B, H, N]

gru_name = "GRU (no graph)"
results = results.loc[~results["model"].eq(gru_name)].copy()
gru_model, gru_hist = fit_and_evaluate(NodeGRU(N, N_CTX), gru_name)

# ---------- Results across the whole dataset ----------
compare = ["GRU (no graph)", "LightGBM (no spatial features)", "LightGBM + spatial lags"]

print("\nMAE BY SPLIT — ALL ZONES")
display(split_summary(compare, "all", "MAE"))
print("\nMAE BY SPLIT — ACTIVE ZONES")
display(split_summary(compare, "active", "MAE"))
print("\nMAE BY SPLIT — EVENT CELLS")
display(split_summary(compare, "events", "MAE"))
print("\nWMAPE BY SPLIT — ALL ZONES")
display(split_summary(compare, "all", "WMAPE"))

generalization_gap(compare)

r = results[results["subset"] == "all"].set_index(["model", "split", "horizon"])["MAE"]
print("\nGRU vs LightGBM — MAE on every split (all zones):")
for s in split_order:
    print(f"  {split_labels[s]}:")
    for h in EVAL_H:
        vals = " | ".join(f"{m.split(' (')[0] if m.startswith('GRU') else m}: {r[(m, s, h)]:.3f}"
                          for m in compare if (m, s, h) in r.index)
        print(f"    h={h}: {vals}")

for h in HORIZONS:
    plot_full_year(compare, h=h)
    print(f"Monthly MAE (+{h}h, all zones)")
    display(plot_monthly_mae(compare, h=h))

print("\nGRU BASELINE COMPLETED FOR +1h, +3h, +6h, +12h, AND +24h")

STEP 23: DEEP LEARNING BASELINE — GRU WITHOUT GRAPH
Device: NVIDIA L4

Training GRU (no graph) (53,512 parameters)
  epoch  1 | train 0.6539 | val 7.7084 | lr 1.0e-03 | 2.3s
  epoch  2 | train 0.5716 | val 7.1817 | lr 1.0e-03 | 1.0s
  epoch  3 | train 0.5369 | val 6.4957 | lr 1.0e-03 | 1.0s
  epoch  4 | train 0.5157 | val 6.1842 | lr 1.0e-03 | 1.0s
  epoch  5 | train 0.5026 | val 5.9365 | lr 1.0e-03 | 1.0s
  epoch  6 | train 0.4948 | val 5.5021 | lr 1.0e-03 | 1.0s
  epoch  7 | train 0.4900 | val 5.4203 | lr 1.0e-03 | 1.0s
  epoch  8 | train 0.4860 | val 5.3964 | lr 1.0e-03 | 1.0s
  epoch  9 | train 0.4834 | val 5.2732 | lr 1.0e-03 | 1.0s
  epoch 10 | train 0.4806 | val 5.1712 | lr 1.0e-03 | 1.0s
  epoch 11 | train 0.4788 | val 5.0996 | lr 1.0e-03 | 1.0s
  epoch 12 | train 0.4771 | val 5.1275 | lr 1.0e-03 | 1.0s
  epoch 13 | train 0.4756 | val 5.1098 | lr 1.0e-03 | 1.0s
  epoch 14 | train 0.4747 | val 5.1029 | lr 5.0e-04 | 1.0s
  epoch 15 | train 0.4724 | val 5.0396 | lr 5.0e-04 | 1.0s


<Figure size 800x500 with 2 Axes>


MAE BY SPLIT — ALL ZONES


horizon                                                                               1   \
model                          split                                                       
GRU (no graph)                 Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  4.126   
                               Validation (May, Nov)                               4.512   
                               Test (Feb, Sep)                                     4.287   
                               All eligible 2025 origins                           4.216   
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.156   
                               Validation (May, Nov)                               3.725   
                               Test (Feb, Sep)                                     3.492   
                               All eligible 2025 origins                           3.304   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.103   
                               Validation (May, Nov)                               3.625   
                               Test (Feb, Sep)                                     3.412   
                               All eligible 2025 origins                           3.239   

horizon                                                                               3   \
model                          split                                                       
GRU (no graph)                 Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  4.272   
                               Validation (May, Nov)                               4.717   
                               Test (Feb, Sep)                                     4.432   
                               All eligible 2025 origins                           4.371   
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.402   
                               Validation (May, Nov)                               4.298   
                               Test (Feb, Sep)                                     3.851   
                               All eligible 2025 origins                           3.622   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.405   
                               Validation (May, Nov)                               4.295   
                               Test (Feb, Sep)                                     3.877   
                               All eligible 2025 origins                           3.628   

horizon                                                                               6   \
model                          split                                                       
GRU (no graph)                 Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  4.350   
                               Validation (May, Nov)                               4.901   
                               Test (Feb, Sep)                                     4.549   
                               All eligible 2025 origins                           4.473   
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.463   
                               Validation (May, Nov)                               4.526   
                               Test (Feb, Sep)                                     3.992   
                               All eligible 2025 origins                           3.724   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  3.461   
                               Validation (May, Nov)                               4.513   
                               Test (Feb, Sep)                                     3.986   
                               All eligible 2025 origins                           3.719   

horizon                                                                               12  \
model                          split     


MAE BY SPLIT — ACTIVE ZONES


horizon                                                                               1   \
model                          split                                                       
GRU (no graph)                 Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  6.704   
                               Validation (May, Nov)                               7.317   
                               Test (Feb, Sep)                                     6.985   
                               All eligible 2025 origins                           6.850   
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.038   
                               Validation (May, Nov)                               5.967   
                               Test (Feb, Sep)                                     5.611   
                               All eligible 2025 origins                           5.283   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  4.950   
                               Validation (May, Nov)                               5.800   
                               Test (Feb, Sep)                                     5.477   
                               All eligible 2025 origins                           5.174   

horizon                                                                               3   \
model                          split                                                       
GRU (no graph)                 Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  6.950   
                               Validation (May, Nov)                               7.662   
                               Test (Feb, Sep)                                     7.230   
                               All eligible 2025 origins                           7.113   
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.450   
                               Validation (May, Nov)                               6.928   
                               Test (Feb, Sep)                                     6.213   
                               All eligible 2025 origins                           5.816   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.455   
                               Validation (May, Nov)                               6.924   
                               Test (Feb, Sep)                                     6.259   
                               All eligible 2025 origins                           5.826   

horizon                                                                               6   \
model                          split                                                       
GRU (no graph)                 Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  7.081   
                               Validation (May, Nov)                               7.970   
                               Test (Feb, Sep)                                     7.427   
                               All eligible 2025 origins                           7.284   
LightGBM (no spatial features) Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.552   
                               Validation (May, Nov)                               7.311   
                               Test (Feb, Sep)                                     6.451   
                               All eligible 2025 origins                           5.987   
LightGBM + spatial lags        Train (Jan, Mar, Apr, Jun, Jul, Aug, Oct, Dec, ...  5.548   
                               Validation (May, Nov)                               7.288   
                               Test (Feb, Sep)                                     6.440   
                               All eligible 2025 origins                           5.978   

horizon                                                                               12  \
model                          split     

... (22 outputs left out)

Monthly MAE (+24h, all zones)


<Figure size 1200x400 with 1 Axes>

,GRU (no graph),LightGBM (no spatial features),LightGBM + spatial lags
2025-01-31,4.196,3.234,3.251
2025-02-28,4.530,4.065,4.062
2025-03-31,4.601,3.573,3.541
2025-04-30,4.147,3.502,3.519
2025-05-31,5.179,4.678,4.665
2025-06-30,4.709,3.782,3.804
2025-07-31,4.143,3.432,3.461
2025-08-31,4.072,3.213,3.209
2025-09-30,4.527,4.013,4.001
2025-10-31,4.407,3.616,3.611



GRU BASELINE COMPLETED FOR +1h, +3h, +6h, +12h, AND +24h


## How Uber’s DeepETA informs this experiment

Uber’s [DeepETA engineering article](https://www.uber.com/us/en/blog/deepeta-how-uber-predicts-arrival-times/) describes an ETA post-processor that corrects a routing estimate using features such as geography, time and traffic. Uber reports experiments with self-attention, quantile-bucket embeddings, segment bias and asymmetric Huber loss. Its linear Transformer and sparse embedding lookups address online serving latency.

| DeepETA idea | Adaptation to taxi pickup demand |
|---|---|
| Learn the error of an existing estimate | Start with a zone-by-hour-of-week demand profile fitted on the training months. Learn corrections from recent demand, target-aligned daily and weekly lags, and context. |
| Encode spatial and temporal information | Learn zone and forecast-hour embeddings. Pass demand signals across distance, OD flow and training-residual correlation graphs. |
| Use attention to combine feature groups | Apply attention across four tokens per zone and horizon covering graph-mixed demand, origin context, target calendar and the seasonal estimate with daily and weekly lag deviations. |
| Quantile-bucket continuous features | Fit weather cut points on training months and learn bucket embeddings alongside continuous context. Repeated cut points are allowed for zero-heavy weather variables. |
| Adjust for segment bias and handle outliers | Learn a zone-by-horizon bias. Use a training-demand-weighted, symmetric Huber loss on log-demand residuals. Checkpoint selection uses unweighted pickup-count MAE. |
| Measure performance across segments | Report overall and active-zone MAE, event errors, and a borough breakdown. The demo reports inference time. |

This adaptation forecasts hourly counts. Uber’s system forecasts trip-level ETAs. The model here uses standard attention over four tokens. It does not reproduce Uber’s linear Transformer, multiresolution spatial hashing or production latency results.

A matched no-graph residual control checks what graph message passing adds to the distance + OD configuration. Both residual graph configurations are compared with the existing graph networks, both LightGBM baselines, the graph-free GRU and seasonal baselines. The primary score is mean MAE across the 1-, 3-, 6-, 12- and 24-hour horizons. Validation MAE selects the graph candidate and checkpoints. The held-out leaderboard reports the observed order, including cases where a baseline wins. Because the test windows stay inside their assigned months, the first seven days serve as history and the last 24 hours can only be used as targets. Full-year plots and scores combine eligible origins from all splits and include in-sample forecasts.


In [ ]:
import gc

import torch.nn.functional as F

GNN_HORIZONS = (1, 3, 6, 12, 24)
if H < max(GNN_HORIZONS) or tuple(EVAL_H) != GNN_HORIZONS:
    raise ValueError(
        "Set H = 24 and EVAL_H = [1, 3, 6, 12, 24] in preparation, "
        "then rerun preparation and the GRU cell before this cell."
    )

print("GRAPH NEURAL NETWORK MODELS")
print("=" * 55)

G = np.load(SAVE + "graphs.npz")

def sym_norm(A):
    """GCN normalization with self-loops: D^-1/2 (A + I) D^-1/2."""
    A = A + np.eye(len(A))
    d = 1.0 / np.sqrt(A.sum(1))
    return (A * d[:, None]) * d[None, :]

def row_norm(A):
    d = A.sum(1, keepdims=True)
    return np.divide(A, d, out=np.zeros_like(A, dtype=float), where=d > 0)

# Random control graph: same degree distribution as contiguity, edges assigned to random zones
rng = np.random.default_rng(42)
perm = rng.permutation(N)
A_rand = G["A_adj"][perm][:, perm]

graph_set = {
    "contiguity": G["A_adj"],
    "distance": G["A_dist"],
    "OD flow": G["A_od"],
    "correlation": G["A_corr"],
    "random (control)": A_rand,
}

# ---------- Model 1: GCN-GRU ----------
class GraphConv(nn.Module):
    """K-hop graph convolution: concatenates [X, AX, A²X, ...] then projects."""
    def __init__(self, d_in, d_out, A_hat, K=2):
        super().__init__()
        self.register_buffer("A", torch.tensor(A_hat, dtype=torch.float32))
        self.K = K
        self.lin = nn.Linear(d_in * (K + 1), d_out)

    def forward(self, x):                          # x [..., N, D]
        out, h = [x], x
        for _ in range(self.K):
            h = torch.einsum("nm,...md->...nd", self.A, h)
            out.append(h)
        return self.lin(torch.cat(out, dim=-1))

class GCNGRU(nn.Module):
    def __init__(self, n_nodes, n_ctx, A_hat, hidden=64, emb=16, layers=2,
                 horizon=H, dropout=0.1, K=2):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, emb)
        self.inp = nn.Linear(1 + n_ctx + emb, hidden)
        self.gc_in = GraphConv(hidden, hidden, A_hat, K)       # spatial mixing per hour
        self.gru = nn.GRU(hidden, hidden, layers, batch_first=True, dropout=dropout)
        self.gc_out = GraphConv(hidden, hidden, A_hat, K)      # spatial mixing of temporal state
        self.norm = nn.LayerNorm(hidden)
        self.head = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, horizon))

    def forward(self, x_dem, x_ctx):               # x_dem [B,T,N], x_ctx [B,T,F]
        B, T, N = x_dem.shape
        e = self.emb.weight[None, None].expand(B, T, N, -1)
        c = x_ctx[:, :, None, :].expand(B, T, N, -1)
        h = torch.relu(self.inp(torch.cat([x_dem[..., None], c, e], dim=-1)))   # [B,T,N,D]
        h = h + torch.relu(self.gc_in(h))
        h = h.permute(0, 2, 1, 3).reshape(B * N, T, -1)
        out, _ = self.gru(h)
        s = out[:, -1].view(B, N, -1)
        s = self.norm(s + torch.relu(self.gc_out(s)))
        return self.head(s).permute(0, 2, 1)       # [B,H,N]

# ---------- Model 2: Graph WaveNet ----------
class DiffusionConv(nn.Module):
    def __init__(self, c_in, c_out, n_supports, order=2, dropout=0.3):
        super().__init__()
        self.order = order
        self.mlp = nn.Conv2d(c_in * (1 + order * n_supports), c_out, 1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, supports):                # x [B,C,N,T]
        out = [x]
        for A in supports:
            x1 = torch.einsum("bcvt,wv->bcwt", x, A)
            out.append(x1)
            for _ in range(self.order - 1):
                x1 = torch.einsum("bcvt,wv->bcwt", x1, A)
                out.append(x1)
        return self.dropout(self.mlp(torch.cat(out, dim=1)))

class GraphWaveNet(nn.Module):
    def __init__(self, n_nodes, n_ctx, supports, adaptive=True, emb=16, res_ch=32,
                 dil_ch=32, skip_ch=128, end_ch=256, horizon=H, blocks=4, layers=2,
                 dropout=0.3, apt_dim=10):
        super().__init__()
        self.emb = nn.Embedding(n_nodes, emb)
        self.n_fixed = len(supports)
        for i, s in enumerate(supports):
            self.register_buffer(f"S{i}", torch.tensor(s, dtype=torch.float32))
        self.adaptive = adaptive
        if adaptive:
            self.nodevec1 = nn.Parameter(torch.randn(n_nodes, apt_dim) * 0.1)
            self.nodevec2 = nn.Parameter(torch.randn(apt_dim, n_nodes) * 0.1)
        n_sup = self.n_fixed + int(adaptive)

        self.start = nn.Conv2d(1 + n_ctx + emb, res_ch, 1)
        self.filt, self.gate, self.skip, self.gconv, self.bn = (nn.ModuleList() for _ in range(5))
        rf = 1
        for _ in range(blocks):
            d = 1
            for _ in range(layers):
                self.filt.append(nn.Conv2d(res_ch, dil_ch, (1, 2), dilation=(1, d)))
                self.gate.append(nn.Conv2d(res_ch, dil_ch, (1, 2), dilation=(1, d)))
                self.skip.append(nn.Conv2d(dil_ch, skip_ch, 1))
                self.gconv.append(DiffusionConv(dil_ch, res_ch, n_sup, dropout=dropout))
                self.bn.append(nn.BatchNorm2d(res_ch))
                rf += d
                d *= 2
        self.receptive_field = rf
        self.end1 = nn.Conv2d(skip_ch, end_ch, 1)
        self.end2 = nn.Conv2d(end_ch, horizon, 1)

    def adaptive_adj(self):
        return torch.softmax(torch.relu(self.nodevec1 @ self.nodevec2), dim=1)

    def forward(self, x_dem, x_ctx):               # x_dem [B,T,N], x_ctx [B,T,F]
        B, T, N = x_dem.shape
        e = self.emb.weight[None, None].expand(B, T, N, -1)
        c = x_ctx[:, :, None, :].expand(B, T, N, -1)
        x = torch.cat([x_dem[..., None], c, e], dim=-1).permute(0, 3, 2, 1)   # [B,D,N,T]
        if T < self.receptive_field:
            x = F.pad(x, (self.receptive_field - T, 0))
        supports = [getattr(self, f"S{i}") for i in range(self.n_fixed)]
        if self.adaptive:
            supports = supports + [self.adaptive_adj()]

        x = self.start(x)
        skip = None
        for i in range(len(self.filt)):
            res = x
            h = torch.tanh(self.filt[i](x)) * torch.sigmoid(self.gate[i](x))
            s = self.skip[i](h)
            skip = s if skip is None else s + skip[..., -s.size(3):]
            h = self.gconv[i](h, supports)
            x = self.bn[i](h + res[..., -h.size(3):])
        out = self.end2(torch.relu(self.end1(torch.relu(skip))))   # [B,H,N,1]
        return out[..., -1]                                        # [B,H,N]

# ---------- DeepETA-inspired seasonal residual graph forecaster ----------
class ResidualGraph(nn.Module):
    """Learn a correction to the training hour-of-week profile for each zone."""
    robust_loss = True
    history_hours = MIN_HISTORY

    def __init__(self, supports, hidden=64):
        super().__init__()
        self.register_buffer("supports", torch.tensor(np.asarray(supports), dtype=torch.float32))
        seasonal = np.log1p(ha_table)
        self.register_buffer("seasonal", torch.tensor((seasonal - mu) / sd, dtype=torch.float32))
        self.register_buffer("offsets", torch.arange(1, H + 1))

        # Quantile cut points are fixed from training weather only.
        weather = context.loc[train_mask, wx_cols[:4]].to_numpy()
        edges = np.quantile(weather, np.linspace(0.125, 0.875, 7), axis=0).T
        self.register_buffer("weather_edges", torch.tensor(edges.copy(), dtype=torch.float32))
        self.register_buffer("weather_offsets", torch.arange(len(edges)) * 8)
        self.weather_emb = nn.Embedding(len(edges) * 8, hidden)
        self.zone_emb = nn.Embedding(N, hidden)
        self.clock_emb = nn.Embedding(168, hidden)
        self.lead_emb = nn.Embedding(H, hidden)
        self.history = nn.Linear(2 * T_IN, hidden)
        self.graph = nn.Linear(hidden * (1 + len(supports)), hidden)
        self.context = nn.Linear(N_CTX, hidden)
        self.baseline = nn.Linear(3, hidden)
        self.attention = nn.MultiheadAttention(hidden, 4, dropout=0.1, batch_first=True)
        self.norm = nn.LayerNorm(hidden)
        self.decoder = nn.Sequential(nn.Linear(hidden, hidden), nn.GELU(), nn.Dropout(0.1),
                                     nn.Linear(hidden, 1))
        self.zone_bias = nn.Parameter(torch.zeros(H, N))
        nn.init.normal_(self.decoder[-1].weight, std=0.01)
        nn.init.zeros_(self.decoder[-1].bias)

    @staticmethod
    def clock(sine, cosine, period):
        return torch.round(torch.atan2(sine, cosine) * (period / (2 * np.pi))).long() % period

    def forward(self, x_dem, x_ctx):
        hour = self.clock(x_ctx[..., 0], x_ctx[..., 1], 24)
        day = self.clock(x_ctx[..., 2], x_ctx[..., 3], 7)
        history_clock = day * 24 + hour
        target_clock = (history_clock[:, -1, None] + self.offsets) % 168
        baseline = self.seasonal[target_clock]

        recent = x_dem[:, -T_IN:]
        anomaly = recent - self.seasonal[history_clock[:, -T_IN:]]
        z = self.history(torch.cat((recent.transpose(1, 2), anomaly.transpose(1, 2)), dim=-1))
        neighbors = [support @ z for support in self.supports]
        z = z + F.gelu(self.graph(torch.cat([z, *neighbors], dim=-1)))
        z = z + self.zone_emb.weight

        latest = x_ctx[:, -1]
        wx = latest[:, len(cal_cols):len(cal_cols) + len(self.weather_edges)]
        buckets = (wx[..., None] > self.weather_edges).sum(-1)
        weather = self.weather_emb(buckets + self.weather_offsets).mean(1)
        current_context = self.context(latest) + weather
        future_clock = self.clock_emb(target_clock) + self.lead_emb.weight

        daily = x_dem[:, self.offsets - 25]
        weekly = x_dem[:, self.offsets - 169]
        anchors = torch.stack((baseline, daily - baseline, weekly - baseline), dim=-1)
        b, h, n = baseline.shape
        tokens = torch.stack((z[:, None].expand(b, h, n, -1),
                              current_context[:, None, None].expand(b, h, n, -1),
                              future_clock[:, :, None].expand(b, h, n, -1),
                              self.baseline(anchors)), dim=-2)
        tokens = tokens.flatten(0, 2)
        attended, _ = self.attention(tokens, tokens, tokens, need_weights=False)
        correction = self.decoder(self.norm(tokens + attended).mean(1)).reshape(b, h, n)
        return baseline + correction + self.zone_bias


residual_control = "Seasonal residual (no graph)"
residual_specs = {
    residual_control: ((), 48),
    "Residual Graph (distance + OD)": (("distance", "OD flow"), 48),
    "Residual Graph (OD + correlation)": (("OD flow", "correlation"), 64),
}

def make_residual_model(name):
    graphs, hidden = residual_specs[name]
    return ResidualGraph([row_norm(graph_set[g]) for g in graphs], hidden=hidden)


# ----------GCN-GRU on each graph + random control ----------
for gname, A in graph_set.items():
    torch.manual_seed(42); np.random.seed(42)
    m, _ = fit_and_evaluate(GCNGRU(N, N_CTX, sym_norm(A)), f"GCN-GRU ({gname} graph)")
    del m; gc.collect()

# ---------- Select the best REAL graph on VALIDATION MAE (no test leakage) ----------
rv = results[(results["split"] == "val") & (results["subset"] == "all") &
             results["model"].str.startswith("GCN-GRU") &
             ~results["model"].str.contains("random")]
val_rank = rv.groupby("model")["MAE"].mean().sort_values()
print("\nValidation MAE (mean over horizons) by graph:")
print(val_rank.round(4))
best_model_name = val_rank.index[0]
best_gname = best_model_name.split("(")[1].replace(" graph)", "")
print(f"\nBest graph on validation: {best_gname}")

# ---------- Graph WaveNet ----------
torch.manual_seed(42); np.random.seed(42)
gwn_adp, _ = fit_and_evaluate(
    GraphWaveNet(N, N_CTX, supports=[], adaptive=True), "Graph WaveNet (adaptive only)")

torch.manual_seed(42); np.random.seed(42)
gwn_name = f"Graph WaveNet ({best_gname} + adaptive)"
gwn_best, _ = fit_and_evaluate(
    GraphWaveNet(N, N_CTX, supports=[row_norm(graph_set[best_gname])], adaptive=True), gwn_name)

# Compare graph support and capacity without selecting on test error.
for residual_name in residual_specs:
    torch.manual_seed(42)
    fit_and_evaluate(make_residual_model(residual_name), residual_name,
                     batch_size=8, lr=8e-4, patience=8)

# ---------- Graph ablation: which graph helps, relative to the no-graph GRU ----------
abl_models = (["GRU (no graph)"] + [f"GCN-GRU ({g} graph)" for g in graph_set] +
              ["Graph WaveNet (adaptive only)", gwn_name] + list(residual_specs))
rt = (results[(results["split"] == "test") & (results["subset"] == "all") &
              results["model"].isin(abl_models)]
      .pivot_table(index="model", columns="horizon", values="MAE")
      .reindex(index=abl_models, columns=GNN_HORIZONS))
for h in GNN_HORIZONS:
    rt[f"vs GRU h={h} (%)"] = (rt[h] / rt.loc["GRU (no graph)", h] - 1) * 100
print("\nGRAPH ABLATION — TEST MAE (all zones); negative % = better than no-graph GRU")
display(rt.round(3))

# ---------- Full leaderboard ----------
print("\nLEADERBOARD — TEST SET, ALL ZONES")
display(show("test", "all"))
print("\nLEADERBOARD — TEST SET, ACTIVE ZONES")
display(show("test", "active"))
print("\nLEADERBOARD — TEST SET, EVENT CELLS")
display(show("test", "events"))

# ---------- Whole-dataset view of the key models ----------
key_models = ["GRU (no graph)", "LightGBM + spatial lags", best_model_name, gwn_name] + list(residual_specs)
print("\nMAE BY SPLIT — KEY MODELS (all zones)")
display(split_summary(key_models, "all", "MAE"))
print("\nWMAPE BY SPLIT — KEY MODELS (all zones)")
display(split_summary(key_models, "all", "WMAPE"))
generalization_gap(key_models)

for h in GNN_HORIZONS:
    available_models = [name for name in key_models if h in full_preds.get(name, {})]
    plot_full_year(available_models, h=h)
    print(f"Monthly MAE (+{h}h, all zones)")
    display(plot_monthly_mae(available_models, h=h))

# ---------- What did the adaptive graph learn? ----------
with torch.no_grad():
    adp = gwn_adp.adaptive_adj().cpu().numpy()
np.fill_diagonal(adp, 0)
top5 = np.argsort(-adp, axis=1)[:, :5]
r5 = np.repeat(np.arange(N), 5)
for gname in ["contiguity", "OD flow", "correlation"]:
    overlap = (graph_set[gname][r5, top5.ravel()] > 0).mean()
    print(f"Share of learned top-5 neighbours that are also {gname} neighbours: {overlap:.1%}")

print("\nStrongest learned connections:")
flat = np.argsort(-adp, axis=None)[:10]
for a, b in zip(*np.unravel_index(flat, adp.shape)):
    print(f"  {zone_info.iloc[a]['Zone']} → {zone_info.iloc[b]['Zone']}: "
          f"weight {adp[a, b]:.3f} ({D[a, b]:.1f} mi)")

print("\nGNN MODELLING COMPLETED!")


GRAPH NEURAL NETWORK MODELS

Training GCN-GRU (contiguity graph) (86,728 parameters)
  epoch  1 | train 0.6453 | val 7.6190 | lr 1.0e-03 | 1.4s
  epoch  2 | train 0.5525 | val 6.6540 | lr 1.0e-03 | 1.4s
  epoch  3 | train 0.5219 | val 6.1760 | lr 1.0e-03 | 1.4s
  epoch  4 | train 0.5057 | val 5.8250 | lr 1.0e-03 | 1.4s
  epoch  5 | train 0.4963 | val 5.8670 | lr 1.0e-03 | 1.4s
  epoch  6 | train 0.4907 | val 5.5805 | lr 1.0e-03 | 1.4s
  epoch  7 | train 0.4866 | val 5.3868 | lr 1.0e-03 | 1.4s
  epoch  8 | train 0.4836 | val 5.7243 | lr 1.0e-03 | 1.4s
  epoch  9 | train 0.4815 | val 5.2634 | lr 1.0e-03 | 1.4s
  epoch 10 | train 0.4787 | val 5.1809 | lr 1.0e-03 | 1.4s
  epoch 11 | train 0.4771 | val 5.2387 | lr 1.0e-03 | 1.4s
  epoch 12 | train 0.4757 | val 5.1634 | lr 1.0e-03 | 1.4s
  epoch 13 | train 0.4741 | val 5.0998 | lr 1.0e-03 | 1.4s
  epoch 14 | train 0.4729 | val 5.5649 | lr 1.0e-03 | 1.4s
  epoch 15 | train 0.4715 | val 5.0882 | lr 1.0e-03 | 1.4s
  epoch 16 | train 0.4706 | va

<Figure size 800x500 with 2 Axes>


Training GCN-GRU (distance graph) (86,728 parameters)
  epoch  1 | train 0.6457 | val 7.6579 | lr 1.0e-03 | 1.4s
  epoch  2 | train 0.5535 | val 6.7261 | lr 1.0e-03 | 1.4s
  epoch  3 | train 0.5236 | val 6.1709 | lr 1.0e-03 | 1.4s
  epoch  4 | train 0.5077 | val 5.8990 | lr 1.0e-03 | 1.4s
  epoch  5 | train 0.4979 | val 5.8464 | lr 1.0e-03 | 1.4s
  epoch  6 | train 0.4915 | val 5.7236 | lr 1.0e-03 | 1.4s
  epoch  7 | train 0.4876 | val 5.4881 | lr 1.0e-03 | 1.4s
  epoch  8 | train 0.4842 | val 5.6748 | lr 1.0e-03 | 1.4s
  epoch  9 | train 0.4821 | val 5.3193 | lr 1.0e-03 | 1.4s
  epoch 10 | train 0.4790 | val 5.2765 | lr 1.0e-03 | 1.4s
  epoch 11 | train 0.4775 | val 5.1770 | lr 1.0e-03 | 1.4s
  epoch 12 | train 0.4757 | val 5.1621 | lr 1.0e-03 | 1.4s
  epoch 13 | train 0.4744 | val 5.1196 | lr 1.0e-03 | 1.4s
  epoch 14 | train 0.4730 | val 5.4978 | lr 1.0e-03 | 1.4s
  epoch 15 | train 0.4717 | val 5.0919 | lr 1.0e-03 | 1.4s
  epoch 16 | train 0.4709 | val 5.2984 | lr 1.0e-03 | 1.4s
 

<Figure size 800x500 with 2 Axes>


Training GCN-GRU (OD flow graph) (86,728 parameters)
  epoch  1 | train 0.6431 | val 7.5873 | lr 1.0e-03 | 1.4s
  epoch  2 | train 0.5508 | val 6.7302 | lr 1.0e-03 | 1.4s
  epoch  3 | train 0.5202 | val 6.2949 | lr 1.0e-03 | 1.4s
  epoch  4 | train 0.5041 | val 5.9906 | lr 1.0e-03 | 1.4s
  epoch  5 | train 0.4955 | val 5.9353 | lr 1.0e-03 | 1.4s
  epoch  6 | train 0.4895 | val 5.6169 | lr 1.0e-03 | 1.4s
  epoch  7 | train 0.4861 | val 5.5411 | lr 1.0e-03 | 1.4s
  epoch  8 | train 0.4827 | val 5.6718 | lr 1.0e-03 | 1.4s
  epoch  9 | train 0.4801 | val 5.3658 | lr 1.0e-03 | 1.4s
  epoch 10 | train 0.4779 | val 5.2403 | lr 1.0e-03 | 1.4s
  epoch 11 | train 0.4761 | val 5.4148 | lr 1.0e-03 | 1.4s
  epoch 12 | train 0.4748 | val 5.1286 | lr 1.0e-03 | 1.4s
  epoch 13 | train 0.4730 | val 5.0796 | lr 1.0e-03 | 1.4s
  epoch 14 | train 0.4719 | val 5.3875 | lr 1.0e-03 | 1.4s
  epoch 15 | train 0.4706 | val 5.0620 | lr 1.0e-03 | 1.4s
  epoch 16 | train 0.4700 | val 5.1380 | lr 1.0e-03 | 1.4s
  

<Figure size 800x500 with 2 Axes>

... (44 outputs left out)

Monthly MAE (+24h, all zones)


<Figure size 1200x400 with 1 Axes>

,GRU (no graph),LightGBM + spatial lags,GCN-GRU (distance graph),Graph WaveNet (distance + adaptive),Seasonal residual (no graph),Residual Graph (distance + OD),Residual Graph (OD + correlation)
2025-01-31,4.196,3.251,4.015,3.845,3.269,3.035,3.073
2025-02-28,4.530,4.062,4.508,4.260,3.651,3.568,3.559
2025-03-31,4.601,3.541,4.434,4.346,3.503,3.463,3.429
2025-04-30,4.147,3.519,4.100,3.915,3.372,3.298,3.260
2025-05-31,5.179,4.665,5.060,4.801,4.336,4.348,4.339
2025-06-30,4.709,3.804,4.570,4.306,3.702,3.573,3.548
2025-07-31,4.143,3.461,3.972,3.900,3.289,3.222,3.219
2025-08-31,4.072,3.209,3.761,3.926,3.221,3.100,3.087
2025-09-30,4.527,4.001,4.383,4.127,3.576,3.582,3.618
2025-10-31,4.407,3.611,4.320,4.056,3.415,3.372,3.358


Share of learned top-5 neighbours that are also contiguity neighbours: 1.7%
Share of learned top-5 neighbours that are also OD flow neighbours: 8.6%
Share of learned top-5 neighbours that are also correlation neighbours: 16.3%

Strongest learned connections:
  Battery Park → Dyker Heights: weight 0.233 (5.8 mi)
  Homecrest → Jamaica: weight 0.195 (11.5 mi)
  East Elmhurst → Dyker Heights: weight 0.175 (12.5 mi)
  SoHo → Stuyvesant Heights: weight 0.156 (4.4 mi)
  Bushwick North → Astoria: weight 0.155 (4.2 mi)
  Saint Albans → Jamaica: weight 0.150 (1.9 mi)
  Alphabet City → Astoria: weight 0.149 (4.0 mi)
  Far Rockaway → Jamaica: weight 0.147 (7.3 mi)
  Hollis → Washington Heights South: weight 0.146 (13.1 mi)
  Pelham Parkway → Washington Heights South: weight 0.143 (4.6 mi)

GNN MODELLING COMPLETED!


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
import branca.colormap as bcm
import scipy.stats as stats

# =====================================================================
# RESTORE SETUP VARIABLES AND CONTEXT FOR A DEFAULT HORIZON (e.g., +1h)
# =====================================================================
H_A = 1  # Analysis forecast horizon (choose from 1, 3, 6, 12, 24)
tgt = test_o + H_A
Y = counts[tgt].astype(np.float64)
zone_list = final_demand_matrix.columns.tolist()

# Map models to their long labels in the results dataset
models = {
    "Best GNN": "Residual Graph (OD + correlation)",
    "GRU (no graph)": "GRU (no graph)",
    "LightGBM + spatial": "LightGBM + spatial lags"
}
best_gnn = models["Best GNN"]

# Load predictions
def test_pred(model_name, h):
    if model_name in full_preds:
        return np.clip(full_preds[model_name][h][test_positions], 0, None).astype(np.float64)
    # Fallback to load from saved npz if not in memory
    import re
    slug = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    z = np.load(SAVE + f"preds_{slug}_full.npz")
    return np.clip(z[f"h{h}"][test_positions], 0, None).astype(np.float64)

# Re-evaluate absolute errors (E) on the held-out test split
E = {}
for label, full_name in models.items():
    P = test_pred(full_name, H_A)
    E_raw = np.abs(P - Y)
    E_raw[eval_exclude[tgt]] = np.nan
    E[label] = E_raw

# Re-initialize the borough table (bor_tab)
borough = zone_info["Borough"].to_numpy()
bor_tab = pd.DataFrame({
    lab: {b: np.nanmean(E[lab][:, borough == b]) for b in np.unique(borough)}
    for lab in E
})

# Initialize dictionary tables for saving the results
borough_tables = {}
demand_tables = {}
condition_tables = {}
dm_table = pd.read_csv(SAVE + "diebold_mariano_tests.csv") if os.path.exists(SAVE + "diebold_mariano_tests.csv") else pd.DataFrame()

# =====================================================================
# RUN ANALYSIS CODE
# =====================================================================
bor_tab["Best GNN vs GRU (%)"] = (bor_tab["Best GNN"] / bor_tab["GRU (no graph)"] - 1) * 100
print(f"\n3a. MAE BY BOROUGH (+{H_A}h)")
display(bor_tab.round(3))

# By zone demand level (WMAPE, so busy and quiet zones are comparable)
lvl = pd.cut(counts[train_mask].mean(0), [-0.01, 1, 10, 50, np.inf],
             labels=["<1/hr", "1–10/hr", "10–50/hr", "50+/hr"])
lvl_tab = pd.DataFrame({lab: {str(c): np.nansum(E[lab][:, lvl == c]) / np.nansum(Y[:, lvl == c])
                              for c in lvl.categories} for lab in E})
lvl_tab.insert(0, "zones", [int((lvl == c).sum()) for c in lvl.categories])
print("\n3b. WMAPE BY ZONE DEMAND LEVEL (training mean pickups/hour)")
display(lvl_tab.round(3))

# By condition
wd = idx[tgt].dayofweek < 5
hol = context["is_holiday"].to_numpy()[tgt] == 1
ev = event_flags[tgt]
cond_tab = pd.DataFrame({lab: {
    "Weekday hours": np.nanmean(E[lab][wd]), "Weekend hours": np.nanmean(E[lab][~wd]),
    "Holiday hours": np.nanmean(E[lab][hol]), "Non-holiday hours": np.nanmean(E[lab][~hol]),
    "Event cells": np.nanmean(E[lab][ev]), "Normal cells": np.nanmean(E[lab][~ev]),
} for lab in E})
print(f"\n3c. MAE BY CONDITION (+{H_A}h)")
display(cond_tab.round(3))

# By hour of day and by day across the test period
hrs = idx[tgt].hour.to_numpy()
hour_tab = pd.DataFrame({lab: pd.Series(np.nanmean(E[lab], 1)).groupby(hrs).mean() for lab in E})
daily_tab = pd.DataFrame({lab: pd.Series(np.nanmean(E[lab], 1), index=idx[tgt]).resample("D").mean()
                          for lab in E})
fig, ax = plt.subplots(2, 1, figsize=(14, 8))
hour_tab.plot(ax=ax[0], marker="o", title=f"MAE by Hour of Day (Test, +{H_A}h)")
ax[0].set_xlabel("Hour of day"); ax[0].set_ylabel("MAE (pickups)"); ax[0].set_xticks(range(24))
daily_tab.plot(ax=ax[1], title=f"Daily MAE Across the Held-Out Months · +{H_A}h")
for d in pd.DatetimeIndex(idx[tgt][hol]).normalize().unique():
    ax[1].axvline(d, ls="--", c="gray", alpha=0.5)
ax[1].set_ylabel("MAE (pickups)")
for a in ax: a.legend(fontsize=8)
plt.tight_layout(); plt.show()

# =====================================================================
# WHERE DOES THE GRAPH HELP? (per-zone improvement map)
# =====================================================================
act = counts[train_mask].mean(0) >= 1
zone_gru = np.nanmean(E["GRU (no graph)"], 0)
zone_gnn = np.nanmean(E["Best GNN"], 0)
zone_pct = np.where(act, (zone_gnn / zone_gru - 1) * 100, np.nan)

print("\n4. PER-ZONE IMPROVEMENT (Best GNN vs GRU, active zones)")
print(f"Zones improved: {(zone_pct < 0).sum()} of {act.sum()} active zones")
print(f"Median change: {np.nanmedian(zone_pct):+.1f}%")
deg = (G["A_adj"] > 0).sum(1)
for lab, x in [("training mean demand", counts[train_mask].mean(0)), ("contiguity degree", deg)]:
    rho, p = stats.spearmanr(x[act], zone_pct[act])
    print(f"Spearman correlation of improvement with {lab}: {rho:+.3f} (p = {p:.3f})")

imp = gdf[["geometry"]].reset_index(drop=True).copy()
imp["Zone"], imp["Borough"] = zone_info["Zone"].values, borough
imp["GRU_MAE"], imp["GNN_MAE"] = zone_gru.round(2), zone_gnn.round(2)
imp["change_pct"] = np.round(zone_pct, 1)

# Static map for the paper
ax = imp.plot(column=imp["change_pct"].clip(-30, 30), cmap="RdBu_r", vmin=-30, vmax=30,
              legend=True, figsize=(10, 10), missing_kwds={"color": "lightgray"},
              legend_kwds={"label": "MAE change vs GRU (%) — blue = GNN better", "shrink": 0.6})
ax.set_title(f"Where the Graph Helps: {best_gnn} vs GRU (Test, +{H_A}h)"); ax.axis("off")
plt.show()

# Interactive map
imp_ll = imp.to_crs(4326)
imp_ll["change_pct"] = imp_ll["change_pct"].astype(float)
cmap = bcm.LinearColormap(["#2166ac", "#f7f7f7", "#b2182b"], vmin=-30, vmax=30,
                          caption="MAE change vs GRU (%) — blue = GNN better")
m = folium.Map(location=[40.73, -73.94], zoom_start=11, tiles="cartodbpositron")
folium.GeoJson(
    imp_ll,
    style_function=lambda f: {
        "fillColor": "#d9d9d9" if f["properties"]["change_pct"] is None or
                     np.isnan(f["properties"]["change_pct"])
                     else cmap(np.clip(f["properties"]["change_pct"], -30, 30)),
        "color": "#555", "weight": 0.4, "fillOpacity": 0.7},
    highlight_function=lambda f: {"weight": 2, "color": "black"},
    tooltip=folium.GeoJsonTooltip(
        fields=["Zone", "Borough", "GRU_MAE", "GNN_MAE", "change_pct"],
        aliases=["Zone", "Borough", "GRU MAE", "Best GNN MAE", "Change (%)"]),
).add_to(m)
cmap.add_to(m)
m.save(SAVE + f"gnn_improvement_map_h{H_A}.html")
display(m)

# =====================================================================
# EXAMPLE FORECASTS — FIRST HELD-OUT WEEK
# =====================================================================
wk = np.arange(len(tgt)) < 168
example_predictions = {
    label: test_pred(models[label], H_A)
    for label in ("Best GNN", "GRU (no graph)", "LightGBM + spatial")
}
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for ax, z in zip(axes, [161, 132, 79]):
    j = zone_list.index(z)
    ax.plot(idx[tgt][wk], counts[tgt][wk, j], color="black", lw=1.8, label="Actual")
    for lab in ["Best GNN", "GRU (no graph)", "LightGBM + spatial"]:
        ax.plot(idx[tgt][wk], example_predictions[lab][wk, j], lw=1, alpha=0.85, label=lab)
    ax.set_title(f"{zone_info.loc[z, 'Zone']} — +{H_A}h Forecasts, First Held-Out Week")
    ax.set_ylabel("Pickups")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

borough_tables[H_A] = bor_tab
demand_tables[H_A] = lvl_tab
condition_tables[H_A] = cond_tab

# =====================================================================
# EXPORT FINAL TABLES
# =====================================================================
final_table = (results[(results["split"] == "test") & results["subset"].isin(["all", "active"])]
               .pivot_table(index=["subset", "model"], columns="horizon",
                            values=["MAE", "RMSE", "WMAPE"]).round(3))
final_table.to_csv(SAVE + "final_test_results.csv")
if not dm_table.empty:
    dm_table.to_csv(SAVE + "diebold_mariano_tests.csv", index=False)
pd.concat(borough_tables, names=["horizon", "borough"]).to_csv(SAVE + "mae_by_borough.csv")
pd.concat(demand_tables, names=["horizon", "demand_level"]).to_csv(SAVE + "wmape_by_demand_level.csv")
pd.concat(condition_tables, names=["horizon", "condition"]).to_csv(SAVE + "mae_by_condition.csv")
print("\nTables saved to Drive: final_test_results, diebold_mariano_tests, mae_by_borough, "
      "wmape_by_demand_level, mae_by_condition")

print("\nFINAL ANALYSIS COMPLETED FOR +1h FORECASTS")


3a. MAE BY BOROUGH (+1h)


,Best GNN,GRU (no graph),LightGBM + spatial,Best GNN vs GRU (%)
Bronx,0.647,0.660,0.687,-2.094
Brooklyn,1.340,1.440,1.372,-6.904
EWR,0.546,0.568,0.633,-3.871
Manhattan,9.162,12.453,9.475,-26.426
Queens,1.816,2.187,1.864,-16.950
Staten Island,0.034,0.029,0.051,18.088



3b. WMAPE BY ZONE DEMAND LEVEL (training mean pickups/hour)


,zones,Best GNN,GRU (no graph),LightGBM + spatial
<1/hr,107,0.917,0.919,1.040
1–10/hr,92,0.460,0.491,0.468
10–50/hr,26,0.210,0.248,0.214
50+/hr,38,0.121,0.169,0.125



3c. MAE BY CONDITION (+1h)


,Best GNN,GRU (no graph),LightGBM + spatial
Weekday hours,3.129,4.038,3.287
Weekend hours,3.733,4.910,3.724
Holiday hours,3.455,3.331,2.840
Non-holiday hours,3.298,4.310,3.425
Event cells,73.563,58.325,41.979
Normal cells,3.291,4.279,3.406


... (2 outputs left out)

<Figure size 1000x1000 with 2 Axes>

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


<Figure size 1400x1000 with 3 Axes>


Tables saved to Drive: final_test_results, diebold_mariano_tests, mae_by_borough, wmape_by_demand_level, mae_by_condition

FINAL ANALYSIS COMPLETED FOR +1h FORECASTS


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import folium, branca.colormap as bcm

DEMO_HORIZONS = (1, 3, 6, 12, 24)
if H < max(DEMO_HORIZONS) or tuple(EVAL_H) != DEMO_HORIZONS:
    raise ValueError("Run preparation and train the selected checkpoint for 1, 3, 6, 12, and 24 hours first.")

print("FORECASTING SYSTEM DEMO")
print("=" * 55)

slug = lambda n: re.sub(r"[^a-z0-9]+", "_", n.lower()).strip("_")
zone_list = final_demand_matrix.columns.tolist()
zone_info = zones.set_index("LocationID").loc[zone_list, ["Borough", "Zone"]]
how = (idx.dayofweek * 24 + idx.hour).to_numpy()
if "ha_table" not in globals():
    ha_table = (final_demand_matrix.where(~eval_exclude).iloc[train_mask].groupby(how[train_mask]).mean().fillna(0).to_numpy())

# ---------- Load the trained best GNN ----------
system_model = build_model(best_gnn).to(device)
system_model.load_state_dict(torch.load(SAVE + f"model_{slug(best_gnn)}.pt", map_location=device))
system_model.eval()
print("Forecasting model:", best_gnn)

base_geo = gdf.reindex(zone_list)[["geometry"]].reset_index(drop=True).to_crs(4326)

# ---------- Core forecasting function ----------
@torch.no_grad()
def forecast_at(ts):
    """Forecast pickups for all zones for the next H hours, using the history required by the selected model."""
    ts = pd.Timestamp(ts).floor("h")
    o = idx.get_loc(ts)
    history = getattr(system_model, "history_hours", T_IN)
    if o < history - 1:
        raise ValueError(f"Need {history} observed hours through the forecast origin.")
    t0 = time.time()
    win = o + window_offsets(system_model)
    logp = system_model(Ls_t[win][None], C_t[win][None])[0] * sd_t + mu_t     # [H, N]
    pred = torch.expm1(logp).clamp_min(0).cpu().numpy()
    latency_ms = (time.time() - t0) * 1000

    df = pd.DataFrame({"LocationID": zone_list,
                       "Zone": zone_info["Zone"].values,
                       "Borough": zone_info["Borough"].values,
                       "now": counts[o]})
    for h in range(1, H + 1):
        df[f"+{h}h"] = pred[h - 1].round(1)
        df[f"actual_+{h}h"] = counts[o + h] if o + h < T else np.nan
        df[f"typical_+{h}h"] = ha_table[how[o + h]].round(1) if o + h < T else np.nan
    return ts, o, df, latency_ms

# ---------- Map builder ----------
def forecast_map(ts, df, horizons=DEMO_HORIZONS):
    m = folium.Map(location=[40.73, -73.94], zoom_start=11, tiles=None)
    folium.TileLayer("cartodbpositron", control=False).add_to(m)
    vmax = np.log1p(df[[f"+{h}h" for h in horizons]].to_numpy().max())
    cmap = bcm.linear.YlOrRd_09.scale(0, max(float(vmax), 1e-6))
    cmap.caption = "log(1 + forecast pickups)"
    for k, h in enumerate(horizons):
        g = base_geo.copy()
        g["Zone"], g["Borough"] = df["Zone"].values, df["Borough"].values
        g["now"] = df["now"].astype(int).values
        g["forecast"] = df[f"+{h}h"].values
        g["actual"] = [f"{v:.0f}" if np.isfinite(v) else "n/a" for v in df[f"actual_+{h}h"]]
        g["typical"] = [f"{v:.1f}" if np.isfinite(v) else "n/a" for v in df[f"typical_+{h}h"]]
        when = (ts + pd.Timedelta(hours=h)).strftime("%a %b %d, %H:%M")
        fg = folium.FeatureGroup(name=f"Forecast +{h}h ({when})", overlay=False, show=(k == 0))
        folium.GeoJson(
            g,
            style_function=lambda f: {"fillColor": cmap(np.log1p(f["properties"]["forecast"])),
                                      "color": "#555", "weight": 0.3, "fillOpacity": 0.75},
            highlight_function=lambda f: {"weight": 2, "color": "black"},
            tooltip=folium.GeoJsonTooltip(
                fields=["Zone", "Borough", "now", "forecast", "actual", "typical"],
                aliases=["Zone", "Borough", "Pickups now", f"Forecast (+{h}h)",
                         "Actual", "Typical for this hour"]),
        ).add_to(fg)
        fg.add_to(m)
    cmap.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

# ---------- Full report for one forecast time ----------
def forecast_report(ts, zone_id=161, show_map=True):
    ts, o, df, latency = forecast_at(ts)
    split = month_to_split[ts.month].upper()
    print(f"Forecast issued at {ts:%A %Y-%m-%d %H:%M} | period: {split} | "
          f"inference time: {latency:.1f} ms for {N} zones × {H} hours")

    # Citywide forecasts for the selected reporting horizons.
    city = pd.DataFrame({
        "target time": [ts + pd.Timedelta(hours=h) for h in DEMO_HORIZONS],
        "forecast": [df[f"+{h}h"].sum() for h in DEMO_HORIZONS],
        "actual": [df[f"actual_+{h}h"].sum(min_count=len(df)) for h in DEMO_HORIZONS],
    }, index=[f"+{h}h" for h in DEMO_HORIZONS])
    city["error_%"] = (city["forecast"] / city["actual"].replace(0, np.nan) - 1) * 100
    print(f"\nCitywide pickups now: {int(df['now'].sum()):,}")
    display(city.round(1))

    # Rank each zone by forecast demand and by increase from the origin.
    for h in DEMO_HORIZONS:
        forecast_col, actual_col = f"+{h}h", f"actual_+{h}h"
        columns = ["Zone", "Borough", "now", forecast_col, actual_col]
        print(f"\nTop 10 zones by forecast demand · +{h}h")
        display(df.nlargest(10, forecast_col)[columns])

        rising = df.assign(change=df[forecast_col] - df["now"])
        rising = rising.loc[(rising[forecast_col] >= 20) & (rising["change"] > 0)]
        print(f"Zones with the largest forecast increase · +{h}h (forecast ≥ 20)")
        display(rising.nlargest(10, "change")[columns + ["change"]])

    j = zone_list.index(zone_id)
    past = np.arange(max(0, o - 23), o + 1)
    target_times = [ts + pd.Timedelta(hours=h) for h in DEMO_HORIZONS]
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(idx[past], counts[past, j], color="black", marker=".", label="Recent observed pickups")
    ax.plot([ts, *target_times],
            [counts[o, j], *[df.loc[j, f"+{h}h"] for h in DEMO_HORIZONS]],
            color="tab:red", marker="o", label=f"Forecast ({best_gnn})")
    ax.plot(target_times, [df.loc[j, f"actual_+{h}h"] for h in DEMO_HORIZONS],
            color="black", ls="--", marker="x", label="Actual after origin where available")
    ax.plot(target_times, [df.loc[j, f"typical_+{h}h"] for h in DEMO_HORIZONS],
            color="gray", ls=":", marker=".", label="Typical for the target hour")
    ax.axvline(ts, color="gray", lw=0.8)
    ax.set_title(f"{zone_info.loc[zone_id, 'Zone']} ({zone_info.loc[zone_id, 'Borough']}) — "
                 f"forecast issued {ts:%a %b %d, %H:%M}")
    ax.set_ylabel("Pickups per hour")
    ax.legend(fontsize=8)
    fig.autofmt_xdate()
    plt.show()

    df.to_csv(SAVE + f"forecast_{ts:%Y%m%d_%H%M}.csv", index=False)
    if show_map:
        m = forecast_map(ts, df)
        m.save(SAVE + f"forecast_map_{ts:%Y%m%d_%H%M}.html")
        display(m)
    return df

# ---------- Demo on a held-out forecast origin ----------
demo_ts = idx[test_o[len(test_o) // 4]]
demo_df = forecast_report(demo_ts, zone_id=161)

# ---------- Interactive control panel ----------
date_w = widgets.DatePicker(value=demo_ts.date(), description="Date")
hour_w = widgets.IntSlider(value=demo_ts.hour, min=0, max=23, description="Hour")
zone_w = widgets.Dropdown(
    options=sorted([(f"{zone_info.loc[z, 'Zone']} ({zone_info.loc[z, 'Borough']})", z)
                    for z in zone_list]),
    value=161, description="Zone", layout=widgets.Layout(width="450px"))
map_w = widgets.Checkbox(value=True, description="Show map")
run_w = widgets.Button(description="Run forecast", button_style="primary")
out_w = widgets.Output()

def on_run(_):
    with out_w:
        clear_output(wait=True)
        if date_w.value is None:
            print("Choose a date."); return
        ts = pd.Timestamp(date_w.value) + pd.Timedelta(hours=hour_w.value)
        history = getattr(system_model, "history_hours", T_IN)
        if ts < idx[history - 1] or ts > idx[-1]:
            print(f"Choose a time between {idx[history - 1]} and {idx[-1]}.")
            return
        forecast_report(ts, zone_id=zone_w.value, show_map=map_w.value)

run_w.on_click(on_run)
print("\nINTERACTIVE FORECASTING PANEL — choose a time and zone, then click 'Run forecast'")
display(widgets.VBox([widgets.HBox([date_w, hour_w]), widgets.HBox([zone_w, map_w]), run_w, out_w]))


FORECASTING SYSTEM DEMO
Forecasting model: Residual Graph (OD + correlation)
Forecast issued at Tuesday 2025-02-18 11:00 | period: TEST | inference time: 27.9 ms for 263 zones × 24 hours

Citywide pickups now: 5,848


,target time,forecast,actual,error_%
+1h,2025-02-18 12:00:00,6067.200195,6168.0,-1.6
+3h,2025-02-18 14:00:00,6754.500000,6722.0,0.5
+6h,2025-02-18 17:00:00,8977.900391,8234.0,9.0
+12h,2025-02-18 23:00:00,3221.800049,2871.0,12.2
+24h,2025-02-19 11:00:00,5780.100098,6208.0,-6.9



Top 10 zones by forecast demand · +1h


,Zone,Borough,now,+1h,actual_+1h
236,Upper East Side South,Manhattan,382.0,413.200012,399.0
235,Upper East Side North,Manhattan,362.0,399.299988,414.0
160,Midtown Center,Manhattan,266.0,329.500000,316.0
161,Midtown East,Manhattan,181.0,222.000000,230.0
185,Penn Station/Madison Sq West,Manhattan,279.0,214.899994,169.0
238,Upper West Side South,Manhattan,203.0,203.000000,220.0
169,Murray Hill,Manhattan,195.0,202.600006,228.0
139,Lenox Hill East,Manhattan,199.0,196.100006,218.0
162,Midtown North,Manhattan,175.0,192.600006,179.0
137,LaGuardia Airport,Queens,211.0,189.100006,163.0


Zones with the largest forecast increase · +1h (forecast ≥ 20)


,Zone,Borough,now,+1h,actual_+1h,change
160,Midtown Center,Manhattan,266.0,329.500000,316.0,63.500000
161,Midtown East,Manhattan,181.0,222.000000,230.0,41.000000
235,Upper East Side North,Manhattan,362.0,399.299988,414.0,37.299988
42,Central Park,Manhattan,94.0,127.000000,123.0,33.000000
140,Lenox Hill West,Manhattan,135.0,167.000000,166.0,32.000000
236,Upper East Side South,Manhattan,382.0,413.200012,399.0,31.200012
163,Midtown South,Manhattan,109.0,126.800003,130.0,17.800003
162,Midtown North,Manhattan,175.0,192.600006,179.0,17.600006
232,UN/Turtle Bay South,Manhattan,60.0,77.099998,67.0,17.099998
228,Sutton Place/Turtle Bay North,Manhattan,101.0,116.599998,124.0,15.599998


... (15 outputs left out)

,Zone,Borough,now,+24h,actual_+24h,change
161,Midtown East,Manhattan,181.0,219.500000,225.0,38.500000
160,Midtown Center,Manhattan,266.0,298.399994,300.0,32.399994
140,Lenox Hill West,Manhattan,135.0,159.899994,178.0,24.899994
232,UN/Turtle Bay South,Manhattan,60.0,72.800003,82.0,12.800003
261,Yorkville East,Manhattan,110.0,122.300003,111.0,12.300003
163,Midtown South,Manhattan,109.0,120.300003,149.0,11.300003
228,Sutton Place/Turtle Bay North,Manhattan,101.0,111.800003,114.0,10.800003
237,Upper West Side North,Manhattan,149.0,159.000000,153.0,10.000000
12,Battery Park City,Manhattan,24.0,33.799999,35.0,9.799999
235,Upper East Side North,Manhattan,362.0,370.200012,384.0,8.200012


<Figure size 1200x400 with 1 Axes>

/usr/local/lib/python3.13/dist-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore



INTERACTIVE FORECASTING PANEL — choose a time and zone, then click 'Run forecast'


In [ ]:
import sys, platform, os
import subprocess

PAPER_HORIZONS = [1, 3, 6, 12, 24]
if tuple(EVAL_H) != tuple(PAPER_HORIZONS):
    raise ValueError("Run preparation and evaluation for 1, 3, 6, 12, and 24 hours first.")

print("PROJECT SUMMARY, REPRODUCIBILITY, AND PAPER-READY OUTPUTS")
print("=" * 65)

# =====================================================================
# REPRODUCIBILITY RECORD
# =====================================================================
import sklearn, scipy, statsmodels, lightgbm, geopandas, folium, holidays, pyarrow
versions = {
    "Python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
    "torch": torch.__version__, "lightgbm": lightgbm.__version__, "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__, "geopandas": geopandas.__version__,
    "folium": folium.__version__, "holidays": holidays.__version__, "pyarrow": pyarrow.__version__,
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none",
}
print("1. ENVIRONMENT")
for k, v in versions.items():
    print(f"  {k:12s} {v}")
packages = set("pandas numpy torch lightgbm geopandas folium scipy statsmodels holidays matplotlib pyarrow ipywidgets branca mapclassify".split())
installed = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
with open(SAVE + "requirements.txt", "w") as requirements:
    requirements.writelines(
        line + "\n" for line in installed.splitlines()
        if "==" in line and line.split("==", 1)[0].lower() in packages
    )
print(f"  requirements.txt saved to Drive")

# =====================================================================
# ARTIFACT INVENTORY
# =====================================================================
def category(f):
    if f.startswith("demand_") or f in ("dst_flag.parquet", "weather_2025.parquet",
                                         "context_features.parquet", "prep.npz"):
        return "Data"
    if f.startswith(("graphs", "A_adj", "D_miles")): return "Graphs"
    if f.startswith(("model_", "lgbm_")) and not f.endswith(".csv"): return "Trained models"
    if f.startswith("preds_"): return "Predictions"
    if f.endswith(".html"): return "Interactive maps"
    if f.endswith((".csv", ".tex", ".md")): return "Results tables"
    return "Other"

files = [f for f in sorted(os.listdir(SAVE)) if os.path.isfile(SAVE + f)]
inv = pd.DataFrame({"file": files,
                    "size_MB": [os.path.getsize(SAVE + f) / 1e6 for f in files]})
inv["category"] = inv["file"].map(category)
print("\n2. SAVED ARTIFACTS")
display(inv.groupby("category").agg(files=("file", "count"), total_MB=("size_MB", "sum")).round(1))

# =====================================================================
# KEY FINDINGS (auto-generated from saved results)
# =====================================================================
results = pd.read_csv(SAVE + "model_results.csv")
main = results[results["horizon"].isin(PAPER_HORIZONS) & ~results["model"].str.contains(r"\[", regex=True)]      # exclude seed/ablation runs
test_all = main[(main["split"] == "test") & (main["subset"] == "all")]
val_all = main[(main["split"] == "val") & (main["subset"] == "all")].groupby("model")["MAE"].mean()

baseline_names = [m for m in val_all.index if m.startswith(("Persistence", "Seasonal", "Historical", "Weekly"))]
gnn_names = [m for m in val_all.index if (m.startswith("GCN-GRU") and "random" not in m) or m.startswith(("Graph WaveNet", "Residual Graph"))]
best_baseline = val_all[baseline_names].idxmin()
best_gnn = val_all[gnn_names].idxmin()
mae = test_all.set_index(["model", "horizon"])["MAE"]

def pct(a, b, h):
    return (mae[(a, h)] / mae[(b, h)] - 1) * 100

print("\n3. KEY FINDINGS")
print(f"• Data: {int(counts.sum()):,} cleaned pickups, {T:,} hours × {N} zones (2025).")
print(f"• Best GNN (selected on validation): {best_gnn}")
print(f"• Best baseline (selected on validation): {best_baseline}")
for h in PAPER_HORIZONS:
    print(f"• {h}-hour-ahead test MAE: Best GNN {mae[(best_gnn, h)]:.3f} | "
          f"vs best baseline {pct(best_gnn, best_baseline, h):+.1f}% | "
          f"vs LightGBM + spatial {pct(best_gnn, 'LightGBM + spatial lags', h):+.1f}% | "
          f"vs GRU (no graph) {pct(best_gnn, 'GRU (no graph)', h):+.1f}% | "
          f"vs random graph {pct(best_gnn, 'GCN-GRU (random (control) graph)', h):+.1f}%")

graph_models = [m for m in test_all["model"].unique() if m.startswith("GCN-GRU")]
best_graph_test = min(graph_models, key=lambda m: test_all[test_all["model"] == m]["MAE"].mean())
print(f"• Graph with lowest average test MAE (GCN-GRU family): {best_graph_test}")

if os.path.exists(SAVE + "diebold_mariano_tests.csv"):
    dm = pd.read_csv(SAVE + "diebold_mariano_tests.csv")
    sig = dm[(dm["significant_5%"]) & (dm["DM_statistic"] < 0)]
    print(f"• Diebold–Mariano: Best GNN significantly better in {len(sig)} of {len(dm)} comparisons (5% level).")
    nonsig = dm[~((dm["significant_5%"]) & (dm["DM_statistic"] < 0))]
    for _, row in nonsig.iterrows():
        print(f"    not significant / not better: {row['comparison']} (h={row['horizon']}, p={row['p_value']:.3f})")

if os.path.exists(SAVE + "seed_robustness.csv"):
    sr = pd.read_csv(SAVE + "seed_robustness.csv")
    for h in PAPER_HORIZONS:
        g = sr[(sr["model"] == best_gnn) & (sr["horizon"] == h)].iloc[0]
        r_ = sr[(sr["model"] == "GRU (no graph)") & (sr["horizon"] == h)].iloc[0]
        print(f"• Seeds (h={h}): GNN {g['MAE_mean']:.3f} ± {g['MAE_std']:.3f} vs "
              f"GRU {r_['MAE_mean']:.3f} ± {r_['MAE_std']:.3f}")

if os.path.exists(SAVE + "context_ablation.csv"):
    ab = pd.read_csv(SAVE + "context_ablation.csv", index_col=0)
    for row in ab.index.drop("full model"):
        print(f"• Removing {row}: test MAE change " +
              ", ".join(f"h={h}: {ab.loc[row, f'change h={h} (%)']:+.1f}%" for h in PAPER_HORIZONS))

# =====================================================================
# PAPER-READY RESULTS TABLE
# =====================================================================
paper_models = ([best_baseline, "LightGBM (no spatial features)", "LightGBM + spatial lags",
                 "GRU (no graph)", residual_control] + sorted(graph_models) +
                [m for m in test_all["model"].unique() if m.startswith(("Graph WaveNet", "Residual Graph"))])
paper_table = (test_all[test_all["model"].isin(paper_models)]
               .pivot_table(index="model", columns="horizon", values=["MAE", "RMSE", "WMAPE"])
               .reindex(paper_models))
paper_table.columns = [f"{m} (h={h})" for m, h in paper_table.columns]
paper_table = paper_table[[f"{m} (h={h})" for h in PAPER_HORIZONS for m in ["MAE", "RMSE", "WMAPE"]]]
print("\n4. MAIN RESULTS TABLE (test set, all zones)")
display(paper_table.round(3))

paper_table.round(3).to_csv(SAVE + "paper_main_table.csv")
paper_table.to_latex(SAVE + "paper_main_table.tex", float_format="%.3f",
                     caption=f"Test-set forecasting accuracy ({split_names['test']} 2025), all 263 zones.",
                     label="tab:main_results")
with open(SAVE + "paper_main_table.md", "w") as fh:
    fh.write(paper_table.round(3).to_markdown())
print("Saved: paper_main_table.csv / .tex / .md")

# =====================================================================
#PUBLICATION-QUALITY FIGURES (300 dpi PNG + PDF)
# =====================================================================
FIG = SAVE + "figures/"
os.makedirs(FIG, exist_ok=True)
plt.rcParams.update({"savefig.dpi": 300, "font.size": 11})

def save_fig(fig, name):
    fig.savefig(FIG + name + ".png", bbox_inches="tight")
    fig.savefig(FIG + name + ".pdf", bbox_inches="tight")
    plt.show()

short = {best_baseline: "Best baseline", "LightGBM (no spatial features)": "LightGBM",
         "LightGBM + spatial lags": "LightGBM + spatial", "GRU (no graph)": "GRU (no graph)"}
short.update({m: m.replace("GCN-GRU (", "GCN-GRU\n(").replace(" graph)", ")") for m in graph_models})
short.update({m: m.replace("Graph WaveNet (", "GWN\n(") for m in paper_models if m.startswith(("Graph WaveNet", "Residual Graph"))})

# demand over 2025 with data split
daily = final_demand_matrix.sum(axis=1).resample("D").sum()
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(daily, color="black", lw=1)
shade_month_splits(ax)
ax.set_ylabel("Pickups per day"); ax.set_title("NYC Yellow Taxi Daily Pickups, 2025")
ax.legend(loc="lower left"); save_fig(fig, "fig1_daily_demand_split")

# test MAE by model and horizon
mt = test_all[test_all["model"].isin(paper_models)].pivot_table(
    index="model", columns="horizon", values="MAE").reindex(paper_models)
fig, ax = plt.subplots(figsize=(12, 4.5))
mt.rename(index=short).plot(kind="bar", ax=ax, width=0.8)
ax.set_ylabel("Test MAE (pickups per zone-hour)"); ax.set_xlabel("")
ax.set_title("Forecast Error by Model and Horizon (Test Set, All Zones)")
ax.legend(title="Horizon (h)"); plt.xticks(rotation=0, fontsize=8)
save_fig(fig, "fig2_mae_by_model")

# graph ablation relative to the no-graph GRU
abl_models = graph_models + [m for m in paper_models if m.startswith(("Graph WaveNet", "Residual Graph"))]
rel = pd.DataFrame({h: [pct(m, "GRU (no graph)", h) for m in abl_models] for h in PAPER_HORIZONS},
                   index=[short[m] for m in abl_models])
fig, ax = plt.subplots(figsize=(11, 4))
rel.plot(kind="bar", ax=ax, width=0.8)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("MAE change vs GRU (%)"); ax.set_title("Effect of Graph Structure (negative = better than no graph)")
ax.legend(title="Horizon (h)"); plt.xticks(rotation=0, fontsize=8)
save_fig(fig, "fig3_graph_ablation")

# error growth with horizon
fig, ax = plt.subplots(figsize=(8, 4))
for m in [best_baseline, "LightGBM + spatial lags", "GRU (no graph)", best_gnn]:
    ax.plot(PAPER_HORIZONS, [mae[(m, h)] for h in PAPER_HORIZONS], marker="o", label=short.get(m, m).replace("\n", " "))
ax.set_xticks(PAPER_HORIZONS); ax.set_xlabel("Forecast horizon (hours)"); ax.set_ylabel("Test MAE")
ax.set_title("Error Growth with Forecast Horizon"); ax.legend(fontsize=9)
save_fig(fig, "fig4_mae_vs_horizon")

# seed robustness
if os.path.exists(SAVE + "seed_robustness.csv"):
    fig, ax = plt.subplots(figsize=(8, 4))
    for k, m in enumerate(["GRU (no graph)", best_gnn]):
        s = sr[sr["model"] == m].set_index("horizon").reindex(PAPER_HORIZONS)
        ax.errorbar(np.array(PAPER_HORIZONS) + (k - 0.5) * 0.15, s["MAE_mean"], yerr=s["MAE_std"],
                    marker="o", capsize=4, label=short.get(m, m).replace("\n", " "))
    ax.set_xticks(PAPER_HORIZONS); ax.set_xlabel("Forecast horizon (hours)"); ax.set_ylabel("Test MAE")
    ax.set_title("Test MAE Mean ± Std Over 3 Random Seeds"); ax.legend()
    save_fig(fig, "fig5_seed_robustness")

# Improvement maps calculated separately for each held-out horizon.
if "gdf" in globals() and "abs_err" in globals():
    geometry = gdf.reindex(final_demand_matrix.columns)[["geometry"]].copy()
    active_zones = counts[train_mask].mean(axis=0) >= 1
    for h in PAPER_HORIZONS:
        gru_mae = np.nanmean(abs_err("GRU (no graph)", h), axis=0)
        gnn_mae = np.nanmean(abs_err(best_gnn, h), axis=0)
        ratio = np.divide(
            gnn_mae, gru_mae, out=np.full(gru_mae.shape, np.nan),
            where=active_zones & (gru_mae > 0),
        )
        change = pd.Series(100 * (ratio - 1), index=geometry.index)
        fig, ax = plt.subplots(figsize=(8, 8))
        geometry.plot(
            column=change.clip(-30, 30).to_numpy(), cmap="RdBu_r", vmin=-30, vmax=30,
            ax=ax, legend=True, missing_kwds={"color": "lightgray"},
            legend_kwds={"label": "MAE change vs GRU (%)", "shrink": 0.6},
        )
        ax.set_title(f"Best GNN vs GRU · +{h}h (blue = lower GNN MAE)")
        ax.axis("off")
        save_fig(fig, f"fig6_improvement_map_h{h}")
else:
    print("Run Final Model Comparison and Error Analysis before exporting the improvement maps.")

print("\nFigures saved to Drive:", sorted(os.listdir(FIG)))
print("\nPROJECT PIPELINE COMPLETE!")


PROJECT SUMMARY, REPRODUCIBILITY, AND PAPER-READY OUTPUTS
1. ENVIRONMENT
  Python       3.13.16
  pandas       2.2.3
  numpy        2.4.6
  torch        2.14.1+cu130
  lightgbm     4.7.0
  scipy        1.16.3
  statsmodels  0.15.0
  geopandas    1.2.0
  folium       0.20.0
  holidays     0.108
  pyarrow      23.0.1
  GPU          NVIDIA L4
  requirements.txt saved to Drive

2. SAVED ARTIFACTS


,files,total_MB
category,,
Data,8,6.9
Graphs,3,3.9
Interactive maps,3,34.9
Other,5,0.6
Predictions,20,680.5
Results tables,12,0.2
Trained models,28,54.0



3. KEY FINDINGS
• Data: 48,629,564 cleaned pickups, 8,760 hours × 263 zones (2025).
• Best GNN (selected on validation): Residual Graph (OD + correlation)
• Best baseline (selected on validation): Seasonal residual (no graph)
• 1-hour-ahead test MAE: Best GNN 3.301 | vs best baseline -2.9% | vs LightGBM + spatial -3.2% | vs GRU (no graph) -23.0% | vs random graph -19.8%
• 3-hour-ahead test MAE: Best GNN 3.466 | vs best baseline -1.4% | vs LightGBM + spatial -10.6% | vs GRU (no graph) -21.8% | vs random graph -20.2%
• 6-hour-ahead test MAE: Best GNN 3.518 | vs best baseline -1.8% | vs LightGBM + spatial -11.7% | vs GRU (no graph) -22.7% | vs random graph -19.2%
• 12-hour-ahead test MAE: Best GNN 3.593 | vs best baseline -1.8% | vs LightGBM + spatial -11.0% | vs GRU (no graph) -21.6% | vs random graph -17.8%
• 24-hour-ahead test MAE: Best GNN 3.590 | vs best baseline -0.6% | vs LightGBM + spatial -10.9% | vs GRU (no graph) -20.7% | vs random graph -18.3%
• Graph with lowest average test

,MAE (h=1),RMSE (h=1),WMAPE (h=1),MAE (h=3),RMSE (h=3),WMAPE (h=3),MAE (h=6),RMSE (h=6),WMAPE (h=6),MAE (h=12),RMSE (h=12),WMAPE (h=12),MAE (h=24),RMSE (h=24),WMAPE (h=24)
model,,,,,,,,,,,,,,,
Seasonal residual (no graph),3.400,10.182,0.158,3.515,10.633,0.164,3.584,10.862,0.167,3.658,11.278,0.170,3.612,11.080,0.167
LightGBM (no spatial features),3.492,10.330,0.162,3.851,11.644,0.179,3.992,12.329,0.186,4.029,12.322,0.187,4.038,12.414,0.186
LightGBM + spatial lags,3.412,9.976,0.158,3.877,11.875,0.180,3.986,12.259,0.186,4.036,12.363,0.187,4.030,12.392,0.186
GRU (no graph),4.287,13.803,0.199,4.432,14.129,0.206,4.549,14.614,0.212,4.581,15.080,0.213,4.529,14.912,0.209
Seasonal residual (no graph),3.400,10.182,0.158,3.515,10.633,0.164,3.584,10.862,0.167,3.658,11.278,0.170,3.612,11.080,0.167
GCN-GRU (OD flow graph),4.286,13.940,0.199,4.428,14.172,0.206,4.528,14.435,0.211,4.487,14.441,0.208,4.508,14.900,0.208
GCN-GRU (contiguity graph),4.161,13.165,0.193,4.279,13.590,0.199,4.375,13.994,0.204,4.433,14.380,0.206,4.391,14.471,0.203
GCN-GRU (correlation graph),4.240,13.595,0.197,4.412,14.093,0.205,4.461,14.174,0.208,4.501,14.657,0.209,4.562,15.089,0.211
GCN-GRU (distance graph),4.125,13.316,0.191,4.276,13.764,0.199,4.353,13.925,0.203,4.333,13.797,0.201,4.443,14.707,0.205


Saved: paper_main_table.csv / .tex / .md


<Figure size 1200x350 with 1 Axes>

... (5 outputs left out)

<Figure size 800x800 with 2 Axes>

<Figure size 800x800 with 2 Axes>

<Figure size 800x800 with 2 Axes>


Figures saved to Drive: ['fig1_daily_demand_split.pdf', 'fig1_daily_demand_split.png', 'fig2_mae_by_model.pdf', 'fig2_mae_by_model.png', 'fig3_graph_ablation.pdf', 'fig3_graph_ablation.png', 'fig4_mae_vs_horizon.pdf', 'fig4_mae_vs_horizon.png', 'fig6_improvement_map_h1.pdf', 'fig6_improvement_map_h1.png', 'fig6_improvement_map_h12.pdf', 'fig6_improvement_map_h12.png', 'fig6_improvement_map_h24.pdf', 'fig6_improvement_map_h24.png', 'fig6_improvement_map_h3.pdf', 'fig6_improvement_map_h3.png', 'fig6_improvement_map_h6.pdf', 'fig6_improvement_map_h6.png']

PROJECT PIPELINE COMPLETE!


## Export the trained model to Vercel

The export packages the validation-selected graph model, an ONNX-powered /api/forecast endpoint, an input schema and a dashboard. It checks ONNX parity with PyTorch on real held-out TLC observations. The dashboard shows a historical example and accepts a JSON upload of recent observed counts and corresponding context. Each horizon forecasts pickups during its target hour.

Run this section after training. Unzip nyc-taxi-vercel.zip into the Git repository linked to Vercel and deploy its root using the Other framework preset. The public input schema lists the model-specific history length, zone order, context order and scaling statistics. A live service additionally needs a current observed pickup feed. The public monthly TLC data provide the historical example.


In [ ]:
%pip -q install onnx onnxruntime


In [ ]:
import json
import shutil
from pathlib import Path
import onnxruntime as ort

class ForecastExport(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.history = getattr(model, "history_hours", T_IN)
        self.register_buffer("mean", torch.as_tensor(mu))
        self.register_buffer("scale", torch.as_tensor(sd))
        self.register_buffer("steps", torch.tensor([h - 1 for h in EVAL_H]))

    def forward(self, demand, context_features):
        scaled = (torch.log1p(demand) - self.mean) / self.scale
        log_count = self.model(scaled, context_features)[:, self.steps]
        return torch.clamp_min(torch.exp(log_count * self.scale + self.mean) - 1, 0)

graph_model = build_model(best_gnn).cpu()
graph_model.load_state_dict(torch.load(SAVE + f"model_{slug(best_gnn)}.pt", map_location="cpu"))
export_model = ForecastExport(graph_model.eval()).eval()
project = Path(SAVE) / "nyc-taxi-vercel"
for folder in ("api", "public", "model"):
    (project / folder).mkdir(parents=True, exist_ok=True)

history = export_model.history
validation_examples = [test_o[0], test_o[len(test_o) // 2], test_o[-1]]

def export_inputs(o):
    w = slice(o - history + 1, o + 1)
    return (torch.tensor(counts[w][None]),
            torch.tensor(context[ctx_cols].iloc[w].to_numpy(np.float32)[None]))

model_file = project / "model" / "forecast.onnx"
previous_fastpath = torch.backends.mha.get_fastpath_enabled()
try:
    torch.backends.mha.set_fastpath_enabled(False)
    with torch.no_grad():
        torch.onnx.export(export_model, export_inputs(validation_examples[0]), str(model_file),
                          input_names=["demand", "context"], output_names=["pickups"],
                          opset_version=17, dynamo=False)
finally:
    torch.backends.mha.set_fastpath_enabled(previous_fastpath)

options = ort.SessionOptions()
options.intra_op_num_threads = 2
session = ort.InferenceSession(str(model_file), sess_options=options, providers=["CPUExecutionProvider"])
for o in validation_examples:
    xd, xc = export_inputs(o)
    with torch.inference_mode():
        expected = export_model(xd, xc).numpy()
    [forecast] = session.run(["pickups"], {"demand": xd.numpy(), "context": xc.numpy()})
    np.testing.assert_allclose(forecast, expected, rtol=1e-3, atol=0.02)
print("ONNX parity verified on three held-out TLC forecast origins.")

API_SOURCE = "import { readFile } from 'node:fs/promises';\nimport path from 'node:path';\nimport * as ort from 'onnxruntime-node';\n\nconst files = path.join(process.cwd(), 'model');\nlet resources;\nfunction loadModel() {\n  if (!resources) {\n    resources = Promise.all([\n      readFile(path.join(files, 'metadata.json'), 'utf8').then(JSON.parse),\n      ort.InferenceSession.create(path.join(files, 'forecast.onnx')),\n    ]).catch(error => {\n      resources = undefined;\n      throw error;\n    });\n  }\n  return resources;\n}\n\nfunction matrix(value, rows, cols, nonnegative = false) {\n  return Array.isArray(value) && value.length === rows &&\n    value.every(row => Array.isArray(row) && row.length === cols &&\n      row.every(x => Number.isFinite(x) && Math.abs(x) <= 1e6 && (!nonnegative || x >= 0)));\n}\n\nfunction json(body, status = 200) {\n  return Response.json(body, { status, headers: { 'Cache-Control': 'no-store' } });\n}\n\nasync function forecast(request) {\n\n  try {\n    const [meta, session] = await loadModel();\n    let body;\n    try {\n      body = await request.json();\n    } catch {\n      return json({ error: 'Invalid JSON.' }, 400);\n    }\n    const { demand, context } = body ?? {};\n    if (!matrix(demand, meta.history_hours, meta.zones.length, true) ||\n        !matrix(context, meta.history_hours, meta.context_columns.length)) {\n      return json({ error: 'Expected demand and context matrices matching the published schema.' }, 400);\n    }\n    const { pickups } = await session.run({\n      demand: new ort.Tensor('float32', Float32Array.from(demand.flat()),\n                            [1, meta.history_hours, meta.zones.length]),\n      context: new ort.Tensor('float32', Float32Array.from(context.flat()),\n                             [1, meta.history_hours, meta.context_columns.length]),\n    });\n    const values = Array.from(pickups.data);\n    if (values.some(x => !Number.isFinite(x))) {\n      return json({ error: 'Forecast overflowed. Check input values and scaling.' }, 422);\n    }\n    const zones = meta.zones.map((zone, i) => ({\n      ...zone,\n      forecasts: Object.fromEntries(meta.horizons.map((h, j) => [h, values[j * meta.zones.length + i]])),\n    }));\n    return json({ model: meta.model, horizons: meta.horizons, zones });\n  } catch (error) {\n    console.error(error);\n    return json({ error: 'Forecast service is unavailable.' }, 500);\n  }\n}\n\nexport default {\n  async fetch(request) {\n    if (request.method === 'POST') return forecast(request);\n    return Response.json({ error: 'Use POST.' }, {\n      status: 405, headers: { Allow: 'POST', 'Cache-Control': 'no-store' },\n    });\n  },\n};\n"
DASHBOARD_SOURCE = '<!doctype html>\n<html lang="en">\n<meta charset="utf-8">\n<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>NYC Taxi Pickup Forecast</title>\n<style>\n  * { box-sizing: border-box; }\n  body { margin: 0; background: #f4f6fa; color: #172138; font: 16px system-ui, sans-serif; }\n  main { max-width: 860px; margin: 5vh auto; padding: 24px; }\n  section { padding: 24px; background: white; border: 1px solid #dce2ed; border-radius: 14px; }\n  label { display: block; margin: 16px 0 4px; }\n  select, button, input { font: inherit; max-width: 100%; }\n  button { margin-top: 16px; padding: 9px 15px; background: #173e79; color: white; border: 0; border-radius: 6px; cursor: pointer; }\n  table { width: 100%; border-collapse: collapse; margin-top: 14px; font-variant-numeric: tabular-nums; }\n  th, td { text-align: left; padding: 10px; border-bottom: 1px solid #dce2ed; }\n  p { line-height: 1.45; }\n</style>\n<main>\n  <h1>NYC taxi pickup forecast</h1>\n  <p id="origin">Loading the held-out 2025 example…</p>\n  <section>\n    <label for="zone">Taxi zone</label>\n    <select id="zone" disabled></select>\n    <table><thead><tr><th scope="col">Hours ahead</th><th scope="col">Predicted pickups in target hour</th></tr></thead>\n    <tbody id="results"></tbody></table>\n    <label for="observations">Upload observed history JSON</label>\n    <input type="file" id="observations" accept=".json,application/json">\n    <br><button type="button" id="run">Run forecast</button>\n    <p id="status" role="status"></p>\n    <p><a href="/example-request.json">Example request</a> · <a href="/schema.json">Input schema</a></p>\n  </section>\n</main>\n<script>\n  const select = document.querySelector(\'#zone\');\n  const rows = document.querySelector(\'#results\');\n  const status = document.querySelector(\'#status\');\n  const origin = document.querySelector(\'#origin\');\n  const run = document.querySelector(\'#run\');\n  let output;\n  function render() {\n    const zone = output.zones[Number(select.value)];\n    rows.replaceChildren(...output.horizons.map(h => {\n      const tr = document.createElement(\'tr\');\n      const horizon = tr.insertCell();\n      horizon.textContent = `+${h} h`;\n      tr.insertCell().textContent = zone.forecasts[h].toFixed(1);\n      return tr;\n    }));\n  }\n  async function forecast(input, example) {\n    const response = await fetch(\'/api/forecast\', {\n      method: \'POST\',\n      headers: { \'Content-Type\': \'application/json\' },\n      body: JSON.stringify(input),\n    });\n    const data = await response.json();\n    if (!response.ok) throw new Error(data.error ?? \'Forecast failed.\');\n    output = data;\n    const previous = select.value;\n    select.replaceChildren(...data.zones.map((z, i) => new Option(`${z.name} · ${z.borough}`, i)));\n    select.value = previous || \'0\';\n    select.disabled = false;\n    origin.textContent = example ? `Held-out historical example · ${input.origin} New York time`\n      : `Forecast from uploaded observations${input.origin ? \' · \' + input.origin : \'\'}`;\n    render();\n  }\n  select.addEventListener(\'change\', render);\n  run.addEventListener(\'click\', async () => {\n    const file = document.querySelector(\'#observations\').files[0];\n    if (!file) { status.textContent = \'Choose a history JSON file first.\'; return; }\n    run.disabled = true;\n    status.textContent = \'Running forecast…\';\n    try { await forecast(JSON.parse(await file.text()), false); status.textContent = \'Forecast complete.\'; }\n    catch (error) { status.textContent = error.message; }\n    finally { run.disabled = false; }\n  });\n  (async () => {\n    try {\n      const response = await fetch(\'/example-request.json\');\n      if (!response.ok) throw new Error(\'Example unavailable.\');\n      await forecast(await response.json(), true);\n    } catch (error) { status.textContent = error.message; }\n  })();\n</script>\n</html>\n'

def write_json(path, payload):
    path.write_text(json.dumps(payload, ensure_ascii=False, allow_nan=False), encoding="utf-8")

meta = {
    "model": best_gnn, "horizons": EVAL_H, "history_hours": history,
    "zones": [{"id": int(z), "name": str(zone_info.loc[z, "Zone"]),
               "borough": str(zone_info.loc[z, "Borough"])} for z in final_demand_matrix.columns],
    "context_columns": ctx_cols,
    "weather_scaling": pd.read_csv(SAVE + "weather_scaling.csv", index_col=0).to_dict(),
    "input": {
        "demand": "Raw nonnegative pickup counts in zone order. Oldest hour first.",
        "context": "Aligned numeric inputs in context_columns order. Calendar flags are 0/1; continuous weather uses (value - mean) / std. is_raining is 1 when precipitation exceeds 0.1 mm.",
        "calendar": "Use New York local time. Hour encodings use 2*pi*hour/24. Weekday encodings use 2*pi*weekday/7 with Monday=0. Use federal holiday flags and the transition-hour DST flag from the notebook.",
        "origin": "Optional New York local timestamp for the last observed row.",
    },
}
write_json(project / "model" / "metadata.json", meta)
write_json(project / "public" / "schema.json", meta)
o = validation_examples[0]
xd, xc = export_inputs(o)
write_json(project / "public" / "example-request.json",
           {"origin": str(idx[o]), "demand": xd[0].tolist(), "context": xc[0].tolist()})
write_json(project / "package.json", {
    "private": True, "type": "module", "engines": {"node": "22.x"},
    "dependencies": {"onnxruntime-node": "1.22.0"},
})
write_json(project / "vercel.json", {
    "framework": None, "outputDirectory": "public",
    "functions": {"api/forecast.mjs": {"includeFiles": "model/**", "maxDuration": 30}},
})
(project / "api" / "forecast.mjs").write_text(API_SOURCE, encoding="utf-8")
(project / "public" / "index.html").write_text(DASHBOARD_SOURCE, encoding="utf-8")
archive = shutil.make_archive(str(project), "zip", root_dir=project)
print("Vercel project:", archive)


/tmp/ipykernel_733/3498601658.py:40: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(export_model, export_inputs(validation_examples[0]), str(model_file),
/tmp/ipykernel_733/1351432419.py:204: TracerWarning: Iterating over a tensor might cause the trace to be incorrect. Passing a tensor of different shape won't change the number of iterations executed (and might lead to errors or silently give incorrect results).
  neighbors = [support @ z for support in self.supports]
/tmp/ipykernel_733/1351432419.py:209: TracerWarning: Using len to get tensor shape might cause the trace to be incorrect. Recommended usage would be tensor.shape[0]. Passing 

ONNX parity verified on three held-out TLC forecast origins.
Vercel project: /content/drive/MyDrive/cs668/random_months_five_horizons_seed42/nyc-taxi-vercel.zip


In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/cs668/random_months_five_horizons_seed42/"
    "nyc-taxi-vercel.zip"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>